# RSNA Knee — RGB 2.5D voisins DICOM, 320

**Shard 0/12** (CPU, internet off). **1778 series**, ~10.7 Go.

UID **figés** depuis `data/crop130_rgb25d/manifest_rgb25d_proto_part*.csv` (6 slots v20b, contraste exact). Même recette que le proto 224, seul le resize passe à **320** :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 %
2. bande profondeur **20–80 %**
3. linspace **24** centres dans la bande
4. pour chaque centre `k`, **`k-1, k, k+1` dans le volume DICOM**
5. resize **320×320**, uint8, percentiles 1–99

Shape `(24, 3, 320, 320)`, canaux R=k-1, G=k, B=k+1.

Fichiers dans `/kaggle/working/` :
- `volumes_crop130_rgb25d_320_part0.zip`
- `manifest_rgb25d_320_part0.csv`
- `RECIPE.json`

Download : `python -X utf8 kaggle_DICOM_exctract/download_rgb25d_320.py` → `data/crop130_rgb25d_320/`.

In [ ]:
SHARD_ID = 0
N_SHARDS = 12

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260","series":"1.2.826.0.1.3680043.8.498.13821229744997220641575291927426543265","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231","series":"1.2.826.0.1.3680043.8.498.30827030879314217305902637111255984309","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10013663742400736029415768216599146542","series":"1.2.826.0.1.3680043.8.498.17496901872433262274477070632489305322","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10021179701366615563817491713738306606","series":"1.2.826.0.1.3680043.8.498.29394504594736036797747420525794983796","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10025765742726180988537955593315007559","series":"1.2.826.0.1.3680043.8.498.12296387404690065979747592907647620416","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806","series":"1.2.826.0.1.3680043.8.498.40067587795096624398858253836281924077","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10048842414095994814596043262132965018","series":"1.2.826.0.1.3680043.8.498.60519317366971570170934879228359117304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10061120822929703004201627784850458401","series":"1.2.826.0.1.3680043.8.498.67214754783910771084803504086410905872","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922","series":"1.2.826.0.1.3680043.8.498.11587327685249364808271419937082956182","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018","series":"1.2.826.0.1.3680043.8.498.11984659184847340623187455521474478239","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10088120558600684332737135714636269365","series":"1.2.826.0.1.3680043.8.498.26880088452946170819426054931654314730","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10090136633727522725960302994815319097","series":"1.2.826.0.1.3680043.8.498.66703952733591295003466156599462820913","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863","series":"1.2.826.0.1.3680043.8.498.68794986082746925829057693643199433591","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10096919689272424388113819845481617364","series":"1.2.826.0.1.3680043.8.498.30785636453554791359320199635948074779","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636","series":"1.2.826.0.1.3680043.8.498.35820233711714570510950714573636799689","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10121558854084271154265851933273260339","series":"1.2.826.0.1.3680043.8.498.83414511284588606467536827083958768831","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10124063607403942330156450768015985834","series":"1.2.826.0.1.3680043.8.498.57145142959774700470225001112647817901","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166","series":"1.2.826.0.1.3680043.8.498.12283767751881998837899173131991452052","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471","series":"1.2.826.0.1.3680043.8.498.96789827286986302126324549422969988462","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10136486417446492428821313562961937182","series":"1.2.826.0.1.3680043.8.498.69425699921184484485239858679622033597","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10147180484860577398303524379319212586","series":"1.2.826.0.1.3680043.8.498.64706108996911707457204149525016516009","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10148037169326559689992758180208780221","series":"1.2.826.0.1.3680043.8.498.70944668429307975193901939863950885929","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205","series":"1.2.826.0.1.3680043.8.498.28941751228712272115476522972829518582","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10157549662917062273945150928668828930","series":"1.2.826.0.1.3680043.8.498.67915755116255755725663948554272613542","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10161011217774350674534600439166319161","series":"1.2.826.0.1.3680043.8.498.10027425647703671514725436833633629899","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10165669527316708372190215034240922433","series":"1.2.826.0.1.3680043.8.498.57051414885653176199571522374478663812","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973","series":"1.2.826.0.1.3680043.8.498.11063373555867596933273578209763066062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10182565688010898600749222617373997197","series":"1.2.826.0.1.3680043.8.498.12265612008049925563053411153883345222","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10190506315646895064803936408630113507","series":"1.2.826.0.1.3680043.8.498.10176379852205420311211715017017353604","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10197565004992937722841731691775578390","series":"1.2.826.0.1.3680043.8.498.12654827639333114479747396372948670861","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10201660732108909756259359440298787775","series":"1.2.826.0.1.3680043.8.498.24813367384656548655969236042819182290","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10209387047693448282446870911381660585","series":"1.2.826.0.1.3680043.8.498.31164916096341609329568398610993475558","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10211815027850731240126403602239595062","series":"1.2.826.0.1.3680043.8.498.74679775959848493896703528446535776627","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10218458664774126578027318307405130821","series":"1.2.826.0.1.3680043.8.498.43752233978238519514683268098957735692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10240166440044087554182120832488098189","series":"1.2.826.0.1.3680043.8.498.61710011321979902777595326712949498452","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414","series":"1.2.826.0.1.3680043.8.498.25973814213635179857730551720976417311","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10252121617380434672930525780360219531","series":"1.2.826.0.1.3680043.8.498.80710954301817876622775434101355413065","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10254828365033984708960944586564394286","series":"1.2.826.0.1.3680043.8.498.49484085833686491610764462854617013287","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10256441455107860474283716992657597873","series":"1.2.826.0.1.3680043.8.498.12279222319660760130623733767347406207","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10262784969531507133353458615303007839","series":"1.2.826.0.1.3680043.8.498.12263993498434717277241917699046305672","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10273861326753707436672556008991334542","series":"1.2.826.0.1.3680043.8.498.12831627238089871245025491720802928097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10275265831651276229168539541982478639","series":"1.2.826.0.1.3680043.8.498.12457219846821348991346076820517184942","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10287488031206457680758379452936914669","series":"1.2.826.0.1.3680043.8.498.17001619315704702130658986157286177485","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10299618611626190927173051117228647808","series":"1.2.826.0.1.3680043.8.498.88865317478556511777435126448188139442","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052","series":"1.2.826.0.1.3680043.8.498.19770471468778499025432715468366899017","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10310831394568337664174691281092566272","series":"1.2.826.0.1.3680043.8.498.54525275123662378342416116691077298407","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10318366890716843554246682186865048846","series":"1.2.826.0.1.3680043.8.498.10812668322490503397843861888978525173","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10321556388278642027434955207617064548","series":"1.2.826.0.1.3680043.8.498.47423712242731636401261637350602861900","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10322853084122920408377349136830993432","series":"1.2.826.0.1.3680043.8.498.48851037476961747621757338862952330123","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10327608571317884761792196888985963332","series":"1.2.826.0.1.3680043.8.498.39506465627529543765264712561750329245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10336300544286006569730381727202519821","series":"1.2.826.0.1.3680043.8.498.28772180387049739042543086372880643166","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10348611327224519953673858098392135936","series":"1.2.826.0.1.3680043.8.498.29230858168705832280815309056644742215","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10351460911387106879135924409514835385","series":"1.2.826.0.1.3680043.8.498.16772713777458804807641316499533830968","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10352044313309892076985642485759368197","series":"1.2.826.0.1.3680043.8.498.28396584200596757793551569591388681058","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10361030765327974510262894062047216433","series":"1.2.826.0.1.3680043.8.498.17148003617765316760015327235903013723","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10374791734880310643587680407133671218","series":"1.2.826.0.1.3680043.8.498.73032168183352196382573783084975617443","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10386894500329189457070808193345170500","series":"1.2.826.0.1.3680043.8.498.72270986454939961511955133947802204246","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10387383212921524066516723428665512341","series":"1.2.826.0.1.3680043.8.498.67073654411933211197837776054082019978","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10395211177230760565674811878437948933","series":"1.2.826.0.1.3680043.8.498.11593920344175678734701512504883629457","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10402431429660337753694827208437698007","series":"1.2.826.0.1.3680043.8.498.33060889247587023780117147063392571355","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10409870090434047408497289167124673434","series":"1.2.826.0.1.3680043.8.498.56056886813519906330808385434902626888","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10417042894702157309437002434676100667","series":"1.2.826.0.1.3680043.8.498.36648636720808780775218960791522759678","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10419993051271996183722192755371692029","series":"1.2.826.0.1.3680043.8.498.31323764221064410275601436786326987637","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10422536395266111565753575369125069689","series":"1.2.826.0.1.3680043.8.498.12633680824451258558843983373054563300","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10427893579790102216163565692202919399","series":"1.2.826.0.1.3680043.8.498.16389372010297677026859439703485230388","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10434135978927658622039675788413759069","series":"1.2.826.0.1.3680043.8.498.11798306767994349595961889415922717610","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10442828926117258624043497279351962738","series":"1.2.826.0.1.3680043.8.498.19335685552269112843468091484312101491","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10447920208680921688438738178539475509","series":"1.2.826.0.1.3680043.8.498.86726828908210715470371549783664000011","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10459044164648238749799867924228983674","series":"1.2.826.0.1.3680043.8.498.84579710592211040026972271294979758499","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10470282959537049979906878778884086884","series":"1.2.826.0.1.3680043.8.498.57755319057531808264138262291734850753","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10485026284343436524865922231113748546","series":"1.2.826.0.1.3680043.8.498.94419337704787591441036573749688416478","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10495281520660483987888839682017074584","series":"1.2.826.0.1.3680043.8.498.44689783839514242692597037745725725473","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10499265498716562276605625682538326716","series":"1.2.826.0.1.3680043.8.498.11172454621372449008115410482767753959","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10502440129091559437492859441158882078","series":"1.2.826.0.1.3680043.8.498.29363151875487127579455791733454225829","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10512466806929129734785318932315918522","series":"1.2.826.0.1.3680043.8.498.10025486141631245287651296460734161877","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10519566075312736546346469231439794627","series":"1.2.826.0.1.3680043.8.498.73833957956120486166129566557088880576","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10524164196949381411165626176732559196","series":"1.2.826.0.1.3680043.8.498.29823086899663286443705287746025601451","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10533568124272062696642461303295694739","series":"1.2.826.0.1.3680043.8.498.37115091642012541532816450895595633546","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10536557774352716497356905292652597953","series":"1.2.826.0.1.3680043.8.498.12634660311448442004066559013821348483","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828","series":"1.2.826.0.1.3680043.8.498.11488188191637132309135956282182919822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10541029960718591409318399557079055927","series":"1.2.826.0.1.3680043.8.498.10373800917990226709177233833212984363","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10541227283184832783436242773084724216","series":"1.2.826.0.1.3680043.8.498.80701544799369946819373392154488738274","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10549360192675575195491716398555148512","series":"1.2.826.0.1.3680043.8.498.47879153099452954825082930602586585487","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10563016328082326759266280856730347260","series":"1.2.826.0.1.3680043.8.498.23410431414728261692500154720536915875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10580260109117947709069163110100632085","series":"1.2.826.0.1.3680043.8.498.26927080523335661175077300030739013437","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10589889055723532336864750992142411145","series":"1.2.826.0.1.3680043.8.498.31742856310510345677187008291941459085","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10590169032156570386146081157598789893","series":"1.2.826.0.1.3680043.8.498.31340420050152715354523257675584186004","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10599128554628698518723223547497007137","series":"1.2.826.0.1.3680043.8.498.20839811402117187709507496282216813260","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10601062881015669567737602031338578146","series":"1.2.826.0.1.3680043.8.498.48148686253842768266321588486210012215","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196","series":"1.2.826.0.1.3680043.8.498.98178618723700778476070397442130840867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10614508096225944703643482200835890514","series":"1.2.826.0.1.3680043.8.498.90632270480377325950680478185169525482","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10619967804774235096151367674093177430","series":"1.2.826.0.1.3680043.8.498.31681030816712782042069143399948427809","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10626071553432761206594400919097854593","series":"1.2.826.0.1.3680043.8.498.10984567513970444469127446194149946682","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10627795990821506536342369227971125477","series":"1.2.826.0.1.3680043.8.498.30865918226846159205955375296138159412","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10632177122947017620482277062226288392","series":"1.2.826.0.1.3680043.8.498.17062966254631181293923780826401005375","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10648346793256473462304051626410900340","series":"1.2.826.0.1.3680043.8.498.12592854543953428015999239918349110334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10652764425372125015319160977894047658","series":"1.2.826.0.1.3680043.8.498.10671689104654808607888410814053702999","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10656388982978239967297620214873636996","series":"1.2.826.0.1.3680043.8.498.26327054269041464520808101285144095617","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10657584461509809047185519742966035356","series":"1.2.826.0.1.3680043.8.498.41459726027665864200670417725083238935","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10673300644364559354643106316993112731","series":"1.2.826.0.1.3680043.8.498.11181744005436250820622348491269950456","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10675139840690240606562530926349987331","series":"1.2.826.0.1.3680043.8.498.22839091492823307551266668255383689304","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10684020894033412667820478260168097071","series":"1.2.826.0.1.3680043.8.498.98044115978451765084918582310326643011","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10699348333280906956391277892869999716","series":"1.2.826.0.1.3680043.8.498.22096014193744946396168906878241497885","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080","series":"1.2.826.0.1.3680043.8.498.12876673442620591022773197355692357418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10715956446507906169485719013515169244","series":"1.2.826.0.1.3680043.8.498.85022733774732591351862068627041611421","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10728448245470003618625208276233778895","series":"1.2.826.0.1.3680043.8.498.44479948102529623422333650030793666739","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10736346409444614168571281812134757732","series":"1.2.826.0.1.3680043.8.498.43798714947386439041052705185976937904","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802","series":"1.2.826.0.1.3680043.8.498.10902849749213465346403392331533932201","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10748806810865802591313438958975301375","series":"1.2.826.0.1.3680043.8.498.13305876833864723307690636770798652908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10751624265599812767108610284996090628","series":"1.2.826.0.1.3680043.8.498.50249993008293946455817179766866623860","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826","series":"1.2.826.0.1.3680043.8.498.10198801949545620764705686958715624232","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10762741411077456296656623422020928762","series":"1.2.826.0.1.3680043.8.498.36139811491548941456829201644194900991","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10768305343887790230989914438483009428","series":"1.2.826.0.1.3680043.8.498.11083518817199897486042464802591730382","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10774147135673365265606645781662088745","series":"1.2.826.0.1.3680043.8.498.23437060161156744004884432885970158164","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334","series":"1.2.826.0.1.3680043.8.498.10719231877551676984182272268301940778","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10802653832050736637820225444196833450","series":"1.2.826.0.1.3680043.8.498.81397759507359771941004232135024063933","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10808663384878038371710260386807468588","series":"1.2.826.0.1.3680043.8.498.18822579387269360700711574326558520541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10824420630965162625242888916805538597","series":"1.2.826.0.1.3680043.8.498.11539352714274162147339761152394249711","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10830288389793651695251406408666084306","series":"1.2.826.0.1.3680043.8.498.64579185994209703521242152258827632432","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10842906919708856642000130208791281263","series":"1.2.826.0.1.3680043.8.498.31570563991867471779420276274624051671","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10843917293035279597622029625809500972","series":"1.2.826.0.1.3680043.8.498.97962715867797440969864610648575009199","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10851562687641986545442408470139368318","series":"1.2.826.0.1.3680043.8.498.94211624159239996631227457774338506550","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10854223557359336882116168394037581766","series":"1.2.826.0.1.3680043.8.498.51623380996526781282586532876592817381","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10857485602186206735293765836065270655","series":"1.2.826.0.1.3680043.8.498.53213148828170026137041940312722718485","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10867622488467375444427908425709367385","series":"1.2.826.0.1.3680043.8.498.11618089689196197454113263566166206566","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10880553082640412137979450100928382021","series":"1.2.826.0.1.3680043.8.498.39335444187778338105353026469698903905","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10886583574130697346788286647281863932","series":"1.2.826.0.1.3680043.8.498.10199464076352134980970281334377107311","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10896245499713818848442117831482755673","series":"1.2.826.0.1.3680043.8.498.10901894472460864458307552465787674694","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10917389623891555306584069980815338466","series":"1.2.826.0.1.3680043.8.498.43687113574833540554614995607624067537","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10919665962088003807153039302898262235","series":"1.2.826.0.1.3680043.8.498.11178952638002016908964608883280593637","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10924685588962763768106617368647678902","series":"1.2.826.0.1.3680043.8.498.25973422841054771930247859605243672053","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10927716750119165925963563081261605359","series":"1.2.826.0.1.3680043.8.498.10715532741981405984762674590482771651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10932818400202426453525918723737011737","series":"1.2.826.0.1.3680043.8.498.79091240465651593290202400499708104660","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10937504474673840994058935502377258768","series":"1.2.826.0.1.3680043.8.498.58122080552492159564806979770087873007","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10938719716535758543863717421521822626","series":"1.2.826.0.1.3680043.8.498.71176993937045984887485178192374216812","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10951328287594724738476663200798896670","series":"1.2.826.0.1.3680043.8.498.23874901446318550372688392717199915987","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10964287649604862436796341368993301245","series":"1.2.826.0.1.3680043.8.498.12480998855954565019212550074647697960","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10975347305662629858093628672419901528","series":"1.2.826.0.1.3680043.8.498.98165206172243153272609068119882718014","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10981062990502278795793220223076241424","series":"1.2.826.0.1.3680043.8.498.52585316187335441610377926514859853325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994554560803816407362754025763036100","series":"1.2.826.0.1.3680043.8.498.12932296025754915518103135380099375285","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10994854354532517072359864903910225174","series":"1.2.826.0.1.3680043.8.498.20686165019522422015264636675507379443","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10998144118962608870602180672090998054","series":"1.2.826.0.1.3680043.8.498.67015647886174644970881565263702416545","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11006241238894862275805535376070111510","series":"1.2.826.0.1.3680043.8.498.45411429802308303868823325943673731137","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11011061033863592500429481719734134286","series":"1.2.826.0.1.3680043.8.498.12835516304973761808560494457875386970","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11040296449332764126530525726037541310","series":"1.2.826.0.1.3680043.8.498.82671420112591847879015328189811158037","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11043543420859531726426521315399258528","series":"1.2.826.0.1.3680043.8.498.10210152493008841342409288632473466215","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11045459770905584149950259201414074545","series":"1.2.826.0.1.3680043.8.498.33583383619311044797133141231587812638","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11052656375391331522834364601050945860","series":"1.2.826.0.1.3680043.8.498.47422136973362087034105397935760881505","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11060834081463491262807709963877480026","series":"1.2.826.0.1.3680043.8.498.72025816512329373615242238293629830151","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11070900122260301212131296598983372089","series":"1.2.826.0.1.3680043.8.498.11494042071256178897174372219199581179","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11079556243110299049578981210837702995","series":"1.2.826.0.1.3680043.8.498.80173846444587696730843254547598369684","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11090884548990390279582832205294943635","series":"1.2.826.0.1.3680043.8.498.59256359637084936433429688016464034231","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11100224438877567316062152508871079233","series":"1.2.826.0.1.3680043.8.498.43809302830742185205868748102792734318","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11110575656494405149601490027444484170","series":"1.2.826.0.1.3680043.8.498.37322338774910038320991575953964460336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11119324966854127911591363740049886324","series":"1.2.826.0.1.3680043.8.498.20602156944907176249059419011452391880","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11123446342116837882811834332727990911","series":"1.2.826.0.1.3680043.8.498.51030196291457733484662016944497791969","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11126383719852491587081347886000714259","series":"1.2.826.0.1.3680043.8.498.51050160313385167183212336737677950491","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11131210140372132238166068091728877219","series":"1.2.826.0.1.3680043.8.498.86757961781039536283327799543090515146","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11136723251983227230447257874027263286","series":"1.2.826.0.1.3680043.8.498.28279973769786623823757306247809336663","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11148939654362681630811787929668902046","series":"1.2.826.0.1.3680043.8.498.99848626726594457093705367272619998372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11150597567791052354257165699079348456","series":"1.2.826.0.1.3680043.8.498.13383531435987091188910819212430274701","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11152585968424897353533125085307104033","series":"1.2.826.0.1.3680043.8.498.72104031803816292138270396918463036278","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11159546221032123495135103648570690656","series":"1.2.826.0.1.3680043.8.498.30064328905820577518654138081069948753","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11180457573346204459956051956213528997","series":"1.2.826.0.1.3680043.8.498.11989326054320406353518256418434798899","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11184968672176462852561645279750399747","series":"1.2.826.0.1.3680043.8.498.49022340225674703108931531297813342349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11192287547210248515805146368662883131","series":"1.2.826.0.1.3680043.8.498.88071392585564699552927555868048627622","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11201741528059502111642265469347871878","series":"1.2.826.0.1.3680043.8.498.11356929378751632155182522890654533839","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258","series":"1.2.826.0.1.3680043.8.498.11736508912380007255276556356590136998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11211598263863419448236436640277320409","series":"1.2.826.0.1.3680043.8.498.38448383544287635761202280162733639824","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11218351123386367961865331646414911627","series":"1.2.826.0.1.3680043.8.498.62575172238225809377457224518812625205","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11219813299300915347039206446596901205","series":"1.2.826.0.1.3680043.8.498.11844014608624775794519273449658118385","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11226582925516370364253861721606580867","series":"1.2.826.0.1.3680043.8.498.30435607167945954162597372229264356150","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11231822445970527276203191047278744341","series":"1.2.826.0.1.3680043.8.498.12067040695713170063501378722167820306","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11244115315611630362929576340631670750","series":"1.2.826.0.1.3680043.8.498.92937587744402434978243669825197731118","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11257168136762043265165109052826442974","series":"1.2.826.0.1.3680043.8.498.29523901881212580589386657150318809783","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11260248089108026217929330361529065683","series":"1.2.826.0.1.3680043.8.498.13350918879216090241304662816359514795","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11268635423021790546063903457498005204","series":"1.2.826.0.1.3680043.8.498.37831913874695318685496932943610540609","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11276674006815021828943434205349220551","series":"1.2.826.0.1.3680043.8.498.91781346525787208639280854696168028406","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11277078989123345865112116288626381353","series":"1.2.826.0.1.3680043.8.498.96590490968494270027858281764567587477","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11282706861378379965012380644125662529","series":"1.2.826.0.1.3680043.8.498.15339647100153814711637589468888200814","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580","series":"1.2.826.0.1.3680043.8.498.10819304863029245505396116222076917240","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11295899132697862632905938037090529644","series":"1.2.826.0.1.3680043.8.498.45922994568755907770823262272109933054","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11298099571581989006135838942599921393","series":"1.2.826.0.1.3680043.8.498.73526701787583247747457865686082021879","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11307870660160563329996058267938398411","series":"1.2.826.0.1.3680043.8.498.19576891203894197765482352927593102332","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11314687816975146963598226343179928616","series":"1.2.826.0.1.3680043.8.498.91190252972033305108200535355820995939","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11331824020096159502562313322049966712","series":"1.2.826.0.1.3680043.8.498.56149526209069826779845952006048171738","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11334776960319816844935217444817103072","series":"1.2.826.0.1.3680043.8.498.12809305093626463675437700560972968334","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11340429352365972769264064439788186260","series":"1.2.826.0.1.3680043.8.498.48470120565256947524371241732822117804","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11352722814008170661344291337061231555","series":"1.2.826.0.1.3680043.8.498.12023222157868539661480575742657674707","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11362547519125634779422991321337283241","series":"1.2.826.0.1.3680043.8.498.87467997433225444694259687594424997587","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238","series":"1.2.826.0.1.3680043.8.498.54935794411829066097355160457970604724","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11385845157446557781575804962818981994","series":"1.2.826.0.1.3680043.8.498.40559610047244305516999488080342141387","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11394861774586578112386386342670500749","series":"1.2.826.0.1.3680043.8.498.70939259419327711076655656059758143151","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11402325536521359105285323485008939602","series":"1.2.826.0.1.3680043.8.498.31109467828008213388688276106613953655","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11406381777983128446583783158882827226","series":"1.2.826.0.1.3680043.8.498.82658517802894400230895366622562091255","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11420670844338686032030860787688858057","series":"1.2.826.0.1.3680043.8.498.34129355722891294103716386332487083831","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11425683153901568541658945026861110110","series":"1.2.826.0.1.3680043.8.498.10427581481017218690762910333730454263","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11440214477653629899108986507154583794","series":"1.2.826.0.1.3680043.8.498.12608303877117770443552731059811238511","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11441489276425334980205891153179893797","series":"1.2.826.0.1.3680043.8.498.88886954862051633150097677980318458276","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11448326350892044215419380818679679579","series":"1.2.826.0.1.3680043.8.498.16677935992453446443072656572484976779","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11451981256762378203455481812714370193","series":"1.2.826.0.1.3680043.8.498.97223908955447329462852548848040505643","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11458488293461896930750018852127694875","series":"1.2.826.0.1.3680043.8.498.77413543786187024242220774935652913136","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11467955500991327547315970867313333592","series":"1.2.826.0.1.3680043.8.498.11353120419353900946409397270083068359","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11471148332557464549579838595592736047","series":"1.2.826.0.1.3680043.8.498.11644000660829808236023823890685327075","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11477583622065195478886525904745372063","series":"1.2.826.0.1.3680043.8.498.31431215432480727421024524823593522992","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11487799220518102363643849548556099719","series":"1.2.826.0.1.3680043.8.498.39214302777137833295623618076483815000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11497761063752461132736045914751707046","series":"1.2.826.0.1.3680043.8.498.79756136838487723458642007776689569645","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11498505068787084440256890724559761815","series":"1.2.826.0.1.3680043.8.498.13238832999913055946900175477510377279","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11501070108039994411346060488666341726","series":"1.2.826.0.1.3680043.8.498.10933906207790583697973185195348136622","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11506856887791286395737465774061395929","series":"1.2.826.0.1.3680043.8.498.80617489028173479833435411244776908361","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11515987024418648685676369132061127664","series":"1.2.826.0.1.3680043.8.498.11317799625901047811079114810673086657","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11516842786271998149681118182571061813","series":"1.2.826.0.1.3680043.8.498.76091201572891198793136559565129502226","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11529405479134457001825038259936796388","series":"1.2.826.0.1.3680043.8.498.67592398936915389119012536014081840765","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11533971444023236216745801026152089075","series":"1.2.826.0.1.3680043.8.498.29305521510702581817842563523627836910","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11543622448213658856759993374276266341","series":"1.2.826.0.1.3680043.8.498.90336413333229917755550597150069071368","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11547192037231064935759879673500408149","series":"1.2.826.0.1.3680043.8.498.11587691742845661390626024540868180807","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11548839502674800051973767797346682106","series":"1.2.826.0.1.3680043.8.498.36372604547672946956177617144321969338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840","series":"1.2.826.0.1.3680043.8.498.76053899657527315952853964202368500798","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11569601172991867969672947467467392425","series":"1.2.826.0.1.3680043.8.498.49660726841764354498670979714397992559","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11574480940878399452466699512074769686","series":"1.2.826.0.1.3680043.8.498.64198346668888987597635541739933324476","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11577630811395858629711985689258647804","series":"1.2.826.0.1.3680043.8.498.69959645603678624601124357435116117986","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11583098598455610459898849451117846552","series":"1.2.826.0.1.3680043.8.498.12731052053608982650396044637572604459","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11590096889491409910425456896187293515","series":"1.2.826.0.1.3680043.8.498.78916979580138721193676290333988755064","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11592838054859712596515534341207971997","series":"1.2.826.0.1.3680043.8.498.85512327611212076805056232775947318553","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11596866241403598097171345671773992202","series":"1.2.826.0.1.3680043.8.498.35412825101691261871109131497467068735","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11599387556469808488160280438305325860","series":"1.2.826.0.1.3680043.8.498.13194413063577467094566669903740210086","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11605266731872772432220415835172557961","series":"1.2.826.0.1.3680043.8.498.12834695737720578062715212135955912885","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11610585947047502123824835405421129552","series":"1.2.826.0.1.3680043.8.498.73803295676196580072488751011250276462","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11625598825018422177242262569319579378","series":"1.2.826.0.1.3680043.8.498.11027592332493009045556745444634265269","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11641274367861998748475337224297076397","series":"1.2.826.0.1.3680043.8.498.22761904023226593518646746145132829238","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11642678020345468499556230871411308014","series":"1.2.826.0.1.3680043.8.498.65607218604910062806801393615533617005","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11647595587705638372290862475518195181","series":"1.2.826.0.1.3680043.8.498.12852986430516983976848173955910710828","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11654064547593365398103107850228629066","series":"1.2.826.0.1.3680043.8.498.13270292373891023223652448580625607579","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11659861228342609979638095779959188093","series":"1.2.826.0.1.3680043.8.498.13270709852735850013894611059961663138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11674925658184069242322832524274428408","series":"1.2.826.0.1.3680043.8.498.74016039032458507413301146322642240124","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11679981444785137570894759919380932611","series":"1.2.826.0.1.3680043.8.498.26181108828052359382832672371870731428","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11685904538570360040472064238371379681","series":"1.2.826.0.1.3680043.8.498.46401983179415953510877934290216116335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11695097367686121784104505410611821860","series":"1.2.826.0.1.3680043.8.498.10102515465689053406757705070124186805","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11706378290185110677863653788588307917","series":"1.2.826.0.1.3680043.8.498.35447686011959372517019887398530945592","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11716730437469973434747875401501334429","series":"1.2.826.0.1.3680043.8.498.10177404205691317901248647542885976184","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11720535027022778971695130536590302635","series":"1.2.826.0.1.3680043.8.498.35976323934752293141311648322076736464","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11723286647024326363349453996818420374","series":"1.2.826.0.1.3680043.8.498.16659012354170576571074928438068267669","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11725843001854130477287484865097535511","series":"1.2.826.0.1.3680043.8.498.23726671064863031579749843907233415147","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11751697254880722008245447197446027607","series":"1.2.826.0.1.3680043.8.498.51645184291587405132208506506346915679","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11762352496600661264881556836248251400","series":"1.2.826.0.1.3680043.8.498.63432633461009865673047649637514835382","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756","series":"1.2.826.0.1.3680043.8.498.84820111452997120776541070690928490202","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11776647434143676427722616249587614803","series":"1.2.826.0.1.3680043.8.498.12575657174563256498865151338868052391","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11785138333495400389510795613252286069","series":"1.2.826.0.1.3680043.8.498.65622602041017225961605104307973742785","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11786582806577871618290342375465595269","series":"1.2.826.0.1.3680043.8.498.47020386999226105716312355687540241246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11794753699425693678056056340551238033","series":"1.2.826.0.1.3680043.8.498.12422406265799442746523460670177679152","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11798183157109435753679723596499976762","series":"1.2.826.0.1.3680043.8.498.60360368622335833658611107211398279087","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11803762390999881908101130422708589537","series":"1.2.826.0.1.3680043.8.498.95880688320323997496484959392641735778","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11807494462283790976935891042637502434","series":"1.2.826.0.1.3680043.8.498.72369234236665784150989419166146217235","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11814923363609140180293604689814785969","series":"1.2.826.0.1.3680043.8.498.17299933291401347611614443696214030820","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11830915612184936576804778995164042200","series":"1.2.826.0.1.3680043.8.498.36898043451662084138153405348162227501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11851285389689341495961233129719325417","series":"1.2.826.0.1.3680043.8.498.24910749809324793620183688210826430408","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11863890747360341056211511874563792862","series":"1.2.826.0.1.3680043.8.498.47373118734975749210007531150253156751","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11868154141292577776796381772987549697","series":"1.2.826.0.1.3680043.8.498.20156978796379531151588778181371045944","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11878847808072119549790690649256090471","series":"1.2.826.0.1.3680043.8.498.57816138119660072695941068819075471088","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11881735311125899170335963782017197184","series":"1.2.826.0.1.3680043.8.498.11471869244106786957263841200571617460","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11897083656099489353496670273111097794","series":"1.2.826.0.1.3680043.8.498.12370728879814998165634780468119567791","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11901474252960403779405194385221103673","series":"1.2.826.0.1.3680043.8.498.32441810509454610438277596406552081200","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11905397531490569218559534495408776672","series":"1.2.826.0.1.3680043.8.498.87753470509510140680705588542790701457","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11911688823908554151914734938172012708","series":"1.2.826.0.1.3680043.8.498.11674995492707382870604541483021007333","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11912618147674655465071922267060032851","series":"1.2.826.0.1.3680043.8.498.38326044129499460316395947476668023636","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11919420571588879483419578078755803351","series":"1.2.826.0.1.3680043.8.498.66953182222614774138948843690473347820","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11923218013055841952315996726408812182","series":"1.2.826.0.1.3680043.8.498.53769374453405795780673013586949768606","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11926412946593048423086380882538180028","series":"1.2.826.0.1.3680043.8.498.10153480706935193043374488942641888630","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11935116791150327546384719051023540185","series":"1.2.826.0.1.3680043.8.498.23752797444834730620906268793760654811","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11938864543152001817866013174907274014","series":"1.2.826.0.1.3680043.8.498.10252306463132736006385100230606340656","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11954711639198972958656966617027374769","series":"1.2.826.0.1.3680043.8.498.49182089348956939281436374177186604080","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11958117172819754862618612170999909480","series":"1.2.826.0.1.3680043.8.498.51713818650005470298482697915344613192","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11964919539297578836040550782883323317","series":"1.2.826.0.1.3680043.8.498.28528335861761799676754241353758693075","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11979906027101464432999416052202616662","series":"1.2.826.0.1.3680043.8.498.12557947072633824595850495018215986120","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11981170645605720774509861722988500673","series":"1.2.826.0.1.3680043.8.498.57564434446504861354801180053246421347","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11991512352474459753052051071717622771","series":"1.2.826.0.1.3680043.8.498.37057785839515408450478774328741460909","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12005327370242618323737652892230673473","series":"1.2.826.0.1.3680043.8.498.31402573432321625735253810458733801739","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12007335335641704192494936946829650792","series":"1.2.826.0.1.3680043.8.498.42233642495854255297235007563014231904","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12010150865087936209475730225377000084","series":"1.2.826.0.1.3680043.8.498.95872619895796427149919187015465293317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12026457191304650188547548517205757122","series":"1.2.826.0.1.3680043.8.498.35262821927638246436099588661520044668","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12034579945815402722591687609951730205","series":"1.2.826.0.1.3680043.8.498.55370825133859823863237853894919026086","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12041491173612341571223481616114183729","series":"1.2.826.0.1.3680043.8.498.18583936173644733857301345453323463261","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12050415099056648944934039580585859179","series":"1.2.826.0.1.3680043.8.498.12871315164135273309053079155357544070","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12063742398421783626420335095128808816","series":"1.2.826.0.1.3680043.8.498.11649442246379355811980089014439180970","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12067625141402105065337444349201280606","series":"1.2.826.0.1.3680043.8.498.11006547267363796074800573684973645898","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12071998470084955703027522152040042364","series":"1.2.826.0.1.3680043.8.498.47383175405691734351481093045064377382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12095779329738382018467257299597656491","series":"1.2.826.0.1.3680043.8.498.12818418424254965906550698990317440378","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12104799278874963688461235330096856415","series":"1.2.826.0.1.3680043.8.498.92888546700095706998698548407511315565","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12107680313133383587522762626929218051","series":"1.2.826.0.1.3680043.8.498.15254975328042763762379471314126753079","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12130605331118953242364684736005454581","series":"1.2.826.0.1.3680043.8.498.53358501167253753552096452507295312624","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12132814192991105841939278718608095213","series":"1.2.826.0.1.3680043.8.498.43120378721186681309442464816897187893","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12137739162059836759686062882514824614","series":"1.2.826.0.1.3680043.8.498.12420828072849503611383902941929334326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12150502202634860887314001818431257267","series":"1.2.826.0.1.3680043.8.498.66214009867810440930296992227457469145","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12158630368692962312115489365152319457","series":"1.2.826.0.1.3680043.8.498.66699515004245949765095445109492968741","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12165893434332227221101266719291133169","series":"1.2.826.0.1.3680043.8.498.18788959749471564877284535338723879936","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12166235678892893545054546864889743251","series":"1.2.826.0.1.3680043.8.498.22021519347988056551757909695530014604","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12171469604213841534511019872648327851","series":"1.2.826.0.1.3680043.8.498.93157109844013014444779429138579940548","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12176466551281285838907773521868954581","series":"1.2.826.0.1.3680043.8.498.10907206058012604003586316207568117358","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12181591358572249491885248089245858515","series":"1.2.826.0.1.3680043.8.498.12033703837096607495226239532985097096","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12185615973620890770158935333033805789","series":"1.2.826.0.1.3680043.8.498.58529057244495687117340143524313089455","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12192455583120681639582915365754780698","series":"1.2.826.0.1.3680043.8.498.29238191394047385723760920679322492984","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12195090472992304712555822541659656360","series":"1.2.826.0.1.3680043.8.498.66512546141864710953111961338752837226","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12202865491511254130359509254373393681","series":"1.2.826.0.1.3680043.8.498.56627722411465281832944381661909120075","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12208986088767216690584621773389997968","series":"1.2.826.0.1.3680043.8.498.75505038835088719854070125792753644754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12221892384948220776502422531414659289","series":"1.2.826.0.1.3680043.8.498.10380021732342969848543928014150497712","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12229628534316014336864328244217875465","series":"1.2.826.0.1.3680043.8.498.77664171814406878377789127426017865310","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12238753439953831519247789198174756615","series":"1.2.826.0.1.3680043.8.498.31791541341122039812964918816329369071","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12242172267695588655664456078003635837","series":"1.2.826.0.1.3680043.8.498.69896442556910516142264335645309498716","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12245760386130805526409111049874664817","series":"1.2.826.0.1.3680043.8.498.83463348211076190806043967722458862850","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12248983667596865960950671947508678392","series":"1.2.826.0.1.3680043.8.498.11803270268589733625953139722953206099","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12257484608743859456600273083454447341","series":"1.2.826.0.1.3680043.8.498.10437966082009605497527537047854919952","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12260661730837055872432103268905182876","series":"1.2.826.0.1.3680043.8.498.78916935744063016475138246978607789802","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12266054872802015665014176014323258615","series":"1.2.826.0.1.3680043.8.498.19524039576494581682328888352761780660","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12266644932420958618394322959047940542","series":"1.2.826.0.1.3680043.8.498.17557822948704586176709830886669173973","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12273356591019341319494035708698721367","series":"1.2.826.0.1.3680043.8.498.11930387877137044779889933968468592879","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12274801503555668613400469130773239683","series":"1.2.826.0.1.3680043.8.498.66293017598692586752048509553805691875","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12280407492991430179266369843997653205","series":"1.2.826.0.1.3680043.8.498.98547990995738129600352200890305696730","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12285177271228289867211553575739205132","series":"1.2.826.0.1.3680043.8.498.99356230865461580460772302188109219062","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12294073960417956629256115458135748872","series":"1.2.826.0.1.3680043.8.498.67936209592658378987494961155615863441","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12295443776523723913480216844110882937","series":"1.2.826.0.1.3680043.8.498.34537706025394011156937672163916579013","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12300749089664733313051543117302244508","series":"1.2.826.0.1.3680043.8.498.84691447791210210484752442807091720935","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12305281328019552390691110900015627668","series":"1.2.826.0.1.3680043.8.498.34720091538102298317823881302066740145","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12307356432736970350794033171174403632","series":"1.2.826.0.1.3680043.8.498.11281601290439367917286029508098881881","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12322760603170246518010233325038530097","series":"1.2.826.0.1.3680043.8.498.79946154697912006820532098330939309502","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12331841096593105748526634055968654914","series":"1.2.826.0.1.3680043.8.498.74482759482190920043713166714068068143","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12333996498811456012744466468462396081","series":"1.2.826.0.1.3680043.8.498.32376822456909015666885360106431481179","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12347007036329783363431172840583325937","series":"1.2.826.0.1.3680043.8.498.31024608989465975915561858402246475222","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12348875283993064999589090039392140219","series":"1.2.826.0.1.3680043.8.498.76800284715655706422140497918740198949","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12352132666379687393012255589355771387","series":"1.2.826.0.1.3680043.8.498.64466083433041592504533442646488641824","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12356611369549236159671896247094661767","series":"1.2.826.0.1.3680043.8.498.10162713952401157887561822751533618425","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12365033663152692616611548717152384269","series":"1.2.826.0.1.3680043.8.498.55833206205216391576180063358353640638","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12374183329974116022606010915169117790","series":"1.2.826.0.1.3680043.8.498.42869598345266012170425849757978768901","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12379570725733005682132316461218878636","series":"1.2.826.0.1.3680043.8.498.11145426990282812968098207379267215783","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12383716855639375235119728470042467411","series":"1.2.826.0.1.3680043.8.498.48451639325615810918610975769069395867","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12387075993162655781372091860026675548","series":"1.2.826.0.1.3680043.8.498.11110456150719901343942669182136964030","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12387667155732116394489547491673280062","series":"1.2.826.0.1.3680043.8.498.34986231717546741388297551179536931775","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12406043058701411628555010402350032739","series":"1.2.826.0.1.3680043.8.498.11548825741028162339664114903967566108","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12408314059030096510457440879771212020","series":"1.2.826.0.1.3680043.8.498.50623541619120568480821618308602541860","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408945963789246646229147558387793270","series":"1.2.826.0.1.3680043.8.498.11294879991391997344789265876107393644","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12414869188691502885668778427120766321","series":"1.2.826.0.1.3680043.8.498.85074586939520270917933476837961932763","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12432974885645948103184031984290883090","series":"1.2.826.0.1.3680043.8.498.92754172843712702617030140056119623923","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12438369033802887819851319347622787876","series":"1.2.826.0.1.3680043.8.498.10507207731390464732656502927518790454","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945","series":"1.2.826.0.1.3680043.8.498.94077776728386958447335448372021135873","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12462703906850759490077163612851243080","series":"1.2.826.0.1.3680043.8.498.98123726880340849667978228446462227561","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12473533314363654795650919604551685896","series":"1.2.826.0.1.3680043.8.498.87409194397255907005016597542809744999","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12482482491088588051076210158062455084","series":"1.2.826.0.1.3680043.8.498.18283480528640137818987238003153468143","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12487362549907644519947672428024977201","series":"1.2.826.0.1.3680043.8.498.67200698072235436922632704614714193203","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12490034912643658601552092574630438642","series":"1.2.826.0.1.3680043.8.498.61373526416117052335400883879354375201","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491747080684524905643593142141672951","series":"1.2.826.0.1.3680043.8.498.54071575479717319208470920965187965717","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12492431885298164225672926785397151525","series":"1.2.826.0.1.3680043.8.498.98088158106162517036349306062117146158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12501925425699863373513819814520995919","series":"1.2.826.0.1.3680043.8.498.38363149061180205507895315315870403508","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12507957850103916926719601413279798696","series":"1.2.826.0.1.3680043.8.498.42566523560301055196824632597350060280","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12510856755988712246013743789183840748","series":"1.2.826.0.1.3680043.8.498.11111563359713957345291605996950466942","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12518297594498173981937681848554641851","series":"1.2.826.0.1.3680043.8.498.12904950994982653601250530917424295286","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12522620957277632939731546099134770342","series":"1.2.826.0.1.3680043.8.498.64496758779659273553525337994601403663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12526590714362529606046044104426807158","series":"1.2.826.0.1.3680043.8.498.73264645815923458712822788190563775886","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12531442175319757740745664049001806494","series":"1.2.826.0.1.3680043.8.498.13359400753168086566842065761827049246","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12534628800787651008876070514080610643","series":"1.2.826.0.1.3680043.8.498.73502385699539153722344326572888007159","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12552701911592685137269544109687472988","series":"1.2.826.0.1.3680043.8.498.44942026977771961616121333096205530079","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12553455350653672209409408595948190982","series":"1.2.826.0.1.3680043.8.498.11555518967023812255872897926338992641","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12563940497360871791097093821926086616","series":"1.2.826.0.1.3680043.8.498.97647282285103274150932303772454790582","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12570576337098834311171883034999327454","series":"1.2.826.0.1.3680043.8.498.38305346324232023802999437079781462803","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12574270269124650675934073761741661292","series":"1.2.826.0.1.3680043.8.498.60724099929823565733600851009914469279","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12577828789206467631166875572423539298","series":"1.2.826.0.1.3680043.8.498.11533938700541278622422694860439869461","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12583872770399266463538638801797827468","series":"1.2.826.0.1.3680043.8.498.11493460537397554101966692201369949642","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12606136253498337506682547306933254691","series":"1.2.826.0.1.3680043.8.498.67478273558923484459672806371210877150","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12607203615180268117341271176820357242","series":"1.2.826.0.1.3680043.8.498.13084875690073690187107322574241334371","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12614097867016983076979065551985251584","series":"1.2.826.0.1.3680043.8.498.76698423091696603308977632822987281874","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12620718517255799151283445360332799702","series":"1.2.826.0.1.3680043.8.498.13128300210635954498987952873705124947","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12626383124142146001532197239671686442","series":"1.2.826.0.1.3680043.8.498.20154565558142877706182991249784492184","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172","series":"1.2.826.0.1.3680043.8.498.17759260281473746379249610110375053985","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12637591861595747322913829625646935491","series":"1.2.826.0.1.3680043.8.498.11154885276488973751714409289440343361","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12640439197386154384526090653248054582","series":"1.2.826.0.1.3680043.8.498.15283923119043135120488347192671967735","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12647791319443598318497312837147189300","series":"1.2.826.0.1.3680043.8.498.91865609289513682666304111466239736230","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12655324218817524683138594937984186560","series":"1.2.826.0.1.3680043.8.498.10265726252128223255109996657958690764","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12668708646283106269005674386248135220","series":"1.2.826.0.1.3680043.8.498.12577056978142041715865719154252285931","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12674069717230126880288353271062347095","series":"1.2.826.0.1.3680043.8.498.81443531263342136772832402936706514240","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12688954493547892047021455671738310186","series":"1.2.826.0.1.3680043.8.498.53799506902966850671165724475209743218","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12690803821768205116702487219908820630","series":"1.2.826.0.1.3680043.8.498.56119237171833508599824815910177396508","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12697859103448296208701081683487338111","series":"1.2.826.0.1.3680043.8.498.12577282470137467121775228908835576686","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12700591955254168977239658917260628326","series":"1.2.826.0.1.3680043.8.498.42068331116015651482318423716122554633","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12704063145595847725421427571860589471","series":"1.2.826.0.1.3680043.8.498.85007776385504109998299696021269426288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12715130507546263968763238702574191950","series":"1.2.826.0.1.3680043.8.498.11108003381611882194332132192906362168","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12717728703860353055063551162640833849","series":"1.2.826.0.1.3680043.8.498.12372304330687294354508640690740909628","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12719455674338091089495020475033082405","series":"1.2.826.0.1.3680043.8.498.33793061379823894628715291834986464417","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12725832336445568031849006413563892122","series":"1.2.826.0.1.3680043.8.498.29606991094896697594063964861520908555","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12731497178056855493473947157407264492","series":"1.2.826.0.1.3680043.8.498.12673957503938916067549531725760993224","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12754381146006839341912662615844567067","series":"1.2.826.0.1.3680043.8.498.16112135157393511305118403387719043388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12762076104929080571464647825161185643","series":"1.2.826.0.1.3680043.8.498.45638787042595377116717604912230218575","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12769221625850663248477857087347890908","series":"1.2.826.0.1.3680043.8.498.70686380236543391383384218835671043298","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12771448321084229688831202344007889535","series":"1.2.826.0.1.3680043.8.498.41605981345416410168256582939412035666","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12775769262869064592411499834441472333","series":"1.2.826.0.1.3680043.8.498.12084985666572907412626785904322063615","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12785377697188564760194569072434828180","series":"1.2.826.0.1.3680043.8.498.10541795141721533572218927909914464227","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12790542912560396272981117250277568532","series":"1.2.826.0.1.3680043.8.498.51423943662739744722650148996119340041","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12798419553759200632295966180228523212","series":"1.2.826.0.1.3680043.8.498.10378138244790122644353837146412526083","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492","series":"1.2.826.0.1.3680043.8.498.21310459920289088021941024557702629822","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12806792356922865929896103306611295151","series":"1.2.826.0.1.3680043.8.498.38270505896611099447974653217345516377","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12814598731776085957663058326979908652","series":"1.2.826.0.1.3680043.8.498.11144291148465692321143657130081142384","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12818010040786942613541254450332754046","series":"1.2.826.0.1.3680043.8.498.27963413412379765180472770218392990688","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12820051821104182459960036809430047569","series":"1.2.826.0.1.3680043.8.498.11412863545281292301039139501274452935","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12827120140904847104302376671076396695","series":"1.2.826.0.1.3680043.8.498.75272269527897080793204100663163004892","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12837030643153472711780894512565629507","series":"1.2.826.0.1.3680043.8.498.23012934236719596913926962386365848109","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12862782592303338109631225504988756108","series":"1.2.826.0.1.3680043.8.498.31586891647666321280994253136213123025","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12887544771002132349592859410979448655","series":"1.2.826.0.1.3680043.8.498.10470317684135727887390937086219475696","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12904190317996738774518258536997285832","series":"1.2.826.0.1.3680043.8.498.34547756625906208199357392570301137754","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12912436233176216254115594260643600155","series":"1.2.826.0.1.3680043.8.498.69100743239185966913838311084324597459","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664","series":"1.2.826.0.1.3680043.8.498.68608662210414660456623573930613845952","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12929915324982999692857962099866616265","series":"1.2.826.0.1.3680043.8.498.65832663854502617647033347682762036419","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12942677240800343332600884989364651588","series":"1.2.826.0.1.3680043.8.498.12933474169135149970829225688247673735","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12946789387115203718210904958275579544","series":"1.2.826.0.1.3680043.8.498.10598257710211068187541864564536636760","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12954163390367261226110152166469167935","series":"1.2.826.0.1.3680043.8.498.12268372180941314818115419114169323803","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12957752106697316015996777476239762111","series":"1.2.826.0.1.3680043.8.498.89278747104166212041613309707502098766","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12965787822216495014894819598440431626","series":"1.2.826.0.1.3680043.8.498.35011098425426619235390552784542833869","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12978172997053500029791786614906570285","series":"1.2.826.0.1.3680043.8.498.11413188394281485468403125532519900970","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12980448935316870673341494036873313299","series":"1.2.826.0.1.3680043.8.498.99590572457400723532407983100789010900","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12982826264059547618124186257414818172","series":"1.2.826.0.1.3680043.8.498.70756384117522607813842731571254640433","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12986028147682493711912242081052787346","series":"1.2.826.0.1.3680043.8.498.76889086689139179678902315420468394203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12992013816638129896738209015383985569","series":"1.2.826.0.1.3680043.8.498.11366455487861371988211512062216625899","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13004048880335015585285100185910148808","series":"1.2.826.0.1.3680043.8.498.11554810793861594003937899459491320096","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13016684730504329799460698013153308088","series":"1.2.826.0.1.3680043.8.498.13323661425454081787064110444545986953","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13022266764623819477228425331705640702","series":"1.2.826.0.1.3680043.8.498.11989572844548733116298385962081630171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13024625496981667037628275235696732200","series":"1.2.826.0.1.3680043.8.498.69864894815277448188805667387649207034","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13026463185033527579856819607371375594","series":"1.2.826.0.1.3680043.8.498.27525150252089027674801825010523475001","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13028226019192925363616760332442456953","series":"1.2.826.0.1.3680043.8.498.15817451223222107659079691911720402999","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13034169434311535671044673510644711229","series":"1.2.826.0.1.3680043.8.498.37039095455359038550335603255858891998","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13054766954092919774634499780241210835","series":"1.2.826.0.1.3680043.8.498.76859485033983737867395400072637068009","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13062978180647549151946243290102173508","series":"1.2.826.0.1.3680043.8.498.93844345228007789744992793312267142779","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13069081594164738319108129224400751974","series":"1.2.826.0.1.3680043.8.498.68900241887686256293534234556194881207","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13077357855769903335728068420884303993","series":"1.2.826.0.1.3680043.8.498.85450383283460038586396328853220019227","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13078551660783737720660607567713996703","series":"1.2.826.0.1.3680043.8.498.12564232225002558123563186421416280527","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13079170182386765602129976624023222742","series":"1.2.826.0.1.3680043.8.498.56773979231367170012822012681539944439","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13088550794087945662322609097092458487","series":"1.2.826.0.1.3680043.8.498.36610290109767339514377828682615437691","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13093528938282977263747792690249036456","series":"1.2.826.0.1.3680043.8.498.93342507336982033482135646808052527318","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13096288495834275184930201615129174987","series":"1.2.826.0.1.3680043.8.498.36655388650034640633519547569019758547","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13103513835731355705707295729643672786","series":"1.2.826.0.1.3680043.8.498.19832777451191374082500490678053052260","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13114228294359876639109410575915394712","series":"1.2.826.0.1.3680043.8.498.10385363153646704385982745442688167245","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13116941594523500503401720636978356127","series":"1.2.826.0.1.3680043.8.498.12585434086335603762934219031878074111","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13118532243906060645114391749986421707","series":"1.2.826.0.1.3680043.8.498.12245541631528026565989521201533552722","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13126380916005729420163981055214890048","series":"1.2.826.0.1.3680043.8.498.74968090002821121538087962601039550198","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13128506994126383651503283021834372142","series":"1.2.826.0.1.3680043.8.498.20732254537007009182273505336981606129","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13131420451380633781538717117200087258","series":"1.2.826.0.1.3680043.8.498.32934735104910993298054438193067078481","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13135567399913958681613600650440761032","series":"1.2.826.0.1.3680043.8.498.44177755196535822847248610561942355841","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13136392756016978765204420377500607545","series":"1.2.826.0.1.3680043.8.498.47608884290764926094574969729183226999","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13137549927896953291645459480590055925","series":"1.2.826.0.1.3680043.8.498.30734066169601690621701587898843268690","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13146466304138609448483938844487385775","series":"1.2.826.0.1.3680043.8.498.58055715923653783601694116546544786831","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13148805278538458401210722493048566510","series":"1.2.826.0.1.3680043.8.498.12812321094984832283363174266775326253","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13151845892175420505816340845155354956","series":"1.2.826.0.1.3680043.8.498.42706799043684135223358556795006140289","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13155572156594271049520969168607160312","series":"1.2.826.0.1.3680043.8.498.89971332957134926992141313219223560114","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13158227087287104386790063637592969032","series":"1.2.826.0.1.3680043.8.498.37866985410064639471312717437014615339","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13162479706670045075896993021883562680","series":"1.2.826.0.1.3680043.8.498.11652655235771413156216714074870902575","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13165770394160595162984722917452661205","series":"1.2.826.0.1.3680043.8.498.74859576396672934422831358770525181708","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13173050135651736408720687361834045815","series":"1.2.826.0.1.3680043.8.498.11661439834218659249352231638482437228","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13173926977743814814616127342102020710","series":"1.2.826.0.1.3680043.8.498.11319595832817212604878811769434721742","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13179829911816163852815103052496326166","series":"1.2.826.0.1.3680043.8.498.11778893789334947611632734647585838922","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13200225436827306542001129042739578441","series":"1.2.826.0.1.3680043.8.498.10928593084221235901735083037900500089","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13207489724395276840990928372610055495","series":"1.2.826.0.1.3680043.8.498.93783367406068500312295117660295164426","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13211838518836535713919214396985811434","series":"1.2.826.0.1.3680043.8.498.10201180786107092525596230046660080602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13222515831888962408632292783773537312","series":"1.2.826.0.1.3680043.8.498.30131984417750880029115479879583693391","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13226961394403558608712498872213808226","series":"1.2.826.0.1.3680043.8.498.52474735331658674744248336571243571370","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13234278816788324761159326624155958655","series":"1.2.826.0.1.3680043.8.498.57535261257404767995270696614350435156","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13242536312535334908597493894251783308","series":"1.2.826.0.1.3680043.8.498.72011431883477457970706483326473766089","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13256173302993031630968402868348974108","series":"1.2.826.0.1.3680043.8.498.48866862626339829260588560219632616180","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13262053361139017457941697384315610025","series":"1.2.826.0.1.3680043.8.498.22390359576977503831642842056695775037","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13263128568157735797391439005687112733","series":"1.2.826.0.1.3680043.8.498.10197401285699259379595830312484324938","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891","series":"1.2.826.0.1.3680043.8.498.11601405247092213838767293303300560912","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13280017707177438711374726939683941320","series":"1.2.826.0.1.3680043.8.498.53397213298160162895176541265187994389","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13284123714012951608998434802430211634","series":"1.2.826.0.1.3680043.8.498.84128890877451813539688079549197577597","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13290698873574296175460151724606672982","series":"1.2.826.0.1.3680043.8.498.39384613199720798544344535907067408742","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13295231176137292542631413614945667314","series":"1.2.826.0.1.3680043.8.498.30092602730022089766861149300903773661","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13297973188397662905314777604089528518","series":"1.2.826.0.1.3680043.8.498.13331094176343254570245797216437999966","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13314276151860609710431717811242659651","series":"1.2.826.0.1.3680043.8.498.11548814171535408943975528500420084996","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13321746459276191365724686301267421531","series":"1.2.826.0.1.3680043.8.498.78016941133550245818994967941761157315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13330694332190836651711643932953274573","series":"1.2.826.0.1.3680043.8.498.15870568859304326233369071321676057060","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903","series":"1.2.826.0.1.3680043.8.498.32243142107334417600044875429817861685","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13353249891620291438488727876474802879","series":"1.2.826.0.1.3680043.8.498.83549501015517358914700255877680953948","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13353966209257569249620562863006006441","series":"1.2.826.0.1.3680043.8.498.62709726707722651409431357501698502996","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13361877959630120404875545203408927433","series":"1.2.826.0.1.3680043.8.498.12656931828806985172369463495268251309","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13367691581890020433564928882602648123","series":"1.2.826.0.1.3680043.8.498.56786418380876425230359067798848666630","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13368881691140330168876825436602977388","series":"1.2.826.0.1.3680043.8.498.63453436187836216080201643652583732865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13377426362348493372112241208607685717","series":"1.2.826.0.1.3680043.8.498.93144839371916793725387652936151626288","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13383972749798909973472489456126521194","series":"1.2.826.0.1.3680043.8.498.43778766112235831323048269818689644000","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13391735462825209934041441767886593395","series":"1.2.826.0.1.3680043.8.498.89777955969898815239332321340508076335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13400273795954308944359576841942016981","series":"1.2.826.0.1.3680043.8.498.30217771612425904488110689235932749667","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13430522388784978790587514142693914965","series":"1.2.826.0.1.3680043.8.498.56915500623135787595672207666939921384","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13459473184684263663596121399737033933","series":"1.2.826.0.1.3680043.8.498.11443653654825332507307569780142819364","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13460268079835549295338279322112693785","series":"1.2.826.0.1.3680043.8.498.48900108950423597673145391859397003145","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13463672277039459152171414002273376185","series":"1.2.826.0.1.3680043.8.498.23419400109637696894217680534357903892","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13597463452741791628023738080583139636","series":"1.2.826.0.1.3680043.8.498.13214310059748104101048583636433684851","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13660439785395658826946769630921591235","series":"1.2.826.0.1.3680043.8.498.44649243327445986304644499239047338468","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13737588520456064856233576393747389054","series":"1.2.826.0.1.3680043.8.498.86419664226394341912550063578150639520","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13843368750899904654034226115826130084","series":"1.2.826.0.1.3680043.8.498.12614850377142758604071901250350499010","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13958765761660666865188384031150827763","series":"1.2.826.0.1.3680043.8.498.32395110215759589163395645686326515183","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13975027824128345697756313869165838055","series":"1.2.826.0.1.3680043.8.498.85603523949360654211616471181459881138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14061227335828384505403558788548348446","series":"1.2.826.0.1.3680043.8.498.32524759258264642848538705294327271474","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14117402902169664329719927863268363596","series":"1.2.826.0.1.3680043.8.498.11449458565026710704146920510416786457","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14218113768651197803980865495163189825","series":"1.2.826.0.1.3680043.8.498.57067784282580908133866557667910129047","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14236519693915284584208989450687591759","series":"1.2.826.0.1.3680043.8.498.35664891212247614201362248881152659712","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14347036698014475959973071477777915575","series":"1.2.826.0.1.3680043.8.498.70903169475423891835872890851131598379","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14400292895187181989838122627394761532","series":"1.2.826.0.1.3680043.8.498.95875593254469518994236310298572716865","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14472261871417480346162193127520918127","series":"1.2.826.0.1.3680043.8.498.11914414021060839385347519330555677886","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14490539215064287912254286565666891557","series":"1.2.826.0.1.3680043.8.498.11013993237497520180720700271144367421","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14511862461248267537051973802039959211","series":"1.2.826.0.1.3680043.8.498.47679068004927846611425783507248039401","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14728068688166107022558957899468489735","series":"1.2.826.0.1.3680043.8.498.32682713555794910834620259419821400989","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14770617961376953596780345652892307570","series":"1.2.826.0.1.3680043.8.498.16022199894736210710440883453196631893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14845527334849156639306067473358712770","series":"1.2.826.0.1.3680043.8.498.46570727660183872328887860399076413067","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14958560978176121375035839707004667832","series":"1.2.826.0.1.3680043.8.498.86515148329505604806140173492723960705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14963472809950919852176661973255059069","series":"1.2.826.0.1.3680043.8.498.37042945038334407747602238241975774203","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15025957400098336605043184930980421418","series":"1.2.826.0.1.3680043.8.498.13780380596893879617020167387975526804","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15103255966832759925770334719703425755","series":"1.2.826.0.1.3680043.8.498.12690835360618621182773817217379744642","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15219410096632334470101257478715035071","series":"1.2.826.0.1.3680043.8.498.11362474015434111817343257731229960634","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15326951535563258414956521742264374802","series":"1.2.826.0.1.3680043.8.498.12549477607856059593299326918124800547","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15360150088598749206650025637914821738","series":"1.2.826.0.1.3680043.8.498.21169512811955856819485321636523342821","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15399233309269178192583032682669258885","series":"1.2.826.0.1.3680043.8.498.69110703196177779434140057045207572323","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15468179600399954478612037548577772269","series":"1.2.826.0.1.3680043.8.498.64944061470568845504786298171384742749","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15522600887743817464976568872439594397","series":"1.2.826.0.1.3680043.8.498.11420295664043406028866535805333943608","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15595595490910196952568459534537629865","series":"1.2.826.0.1.3680043.8.498.12174323179508892199056845738612354691","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15630146025588580188363070220779173742","series":"1.2.826.0.1.3680043.8.498.12638340815947156821011822215922579484","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15744573360087157139672078937679122840","series":"1.2.826.0.1.3680043.8.498.19968217427183064866595312515172820518","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15969838281154273030237391394632849764","series":"1.2.826.0.1.3680043.8.498.13109147671558933273565629325083279436","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192","series":"1.2.826.0.1.3680043.8.498.75266919254774703903096233357689465742","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.16158812105029662172517682642229718094","series":"1.2.826.0.1.3680043.8.498.32304046180778387620456466563628026282","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16200518336931321995379999168737787921","series":"1.2.826.0.1.3680043.8.498.11793931380759400150724377641242242264","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16258190393569006141285550986533848849","series":"1.2.826.0.1.3680043.8.498.39073525039490251047956716318966117906","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16338296537380784459604897543069625390","series":"1.2.826.0.1.3680043.8.498.13896473674574658278943027776305934254","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16397020531009156965821242117417547331","series":"1.2.826.0.1.3680043.8.498.10540984475026191413807822617799041591","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16527747624688737126118546487228060848","series":"1.2.826.0.1.3680043.8.498.86971311273581432332225502069882977684","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16538738179840493095338324009830673843","series":"1.2.826.0.1.3680043.8.498.59164772246968645754590209453369238878","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16581426215518472805200227992550139190","series":"1.2.826.0.1.3680043.8.498.10671381805957692968234212715895062125","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16666020201961191623151386398626511849","series":"1.2.826.0.1.3680043.8.498.98632081059271591828164635680299864288","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16709565360569239280032743619008187278","series":"1.2.826.0.1.3680043.8.498.95337096257908275675205516501718123203","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16718788321951483456195507236392554704","series":"1.2.826.0.1.3680043.8.498.28419133188262548966924081904831990215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16812524557513419866531695739014681414","series":"1.2.826.0.1.3680043.8.498.10417646439814072027709005102414828140","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16862596867845295946732698348903803700","series":"1.2.826.0.1.3680043.8.498.53794421319470344804452278768014318188","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16886658450293145802897654768506439494","series":"1.2.826.0.1.3680043.8.498.33793722160540353649132319091907544080","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17070139245390396445395414066427036835","series":"1.2.826.0.1.3680043.8.498.19341347411166066673818682596769687334","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119","series":"1.2.826.0.1.3680043.8.498.49681633921337925891548087966569178377","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17107696939614797679012793547913769707","series":"1.2.826.0.1.3680043.8.498.16127201577091531597314566930840651776","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17160610480070701553118850947260201608","series":"1.2.826.0.1.3680043.8.498.92699192625251480850446200000368341186","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17265093113586420061761042467476063876","series":"1.2.826.0.1.3680043.8.498.33832347991913416219793870322426966546","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17336223563862025807066700226971845206","series":"1.2.826.0.1.3680043.8.498.55510287565913733887592857920014293456","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17420673628956367425739018933968443620","series":"1.2.826.0.1.3680043.8.498.10515248563472553827070311412588159787","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17547781127981007962008163248942714330","series":"1.2.826.0.1.3680043.8.498.11113573493327293373049696896520259691","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17618734998753079190687719289815512319","series":"1.2.826.0.1.3680043.8.498.58758972976214847618286048735415843285","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17691941055331862184525411493110144433","series":"1.2.826.0.1.3680043.8.498.52559014864932461071840583487131394898","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17768582402712001917772688462080201650","series":"1.2.826.0.1.3680043.8.498.45749044998377845507398973108555028784","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17836815413824722543229791044312174885","series":"1.2.826.0.1.3680043.8.498.11387702157879695178515238460511552056","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17856297841306582866782295284505406916","series":"1.2.826.0.1.3680043.8.498.11651380858812929294821435100470496923","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17909590711893200514107254988453660857","series":"1.2.826.0.1.3680043.8.498.82604192246447561319302873925465289540","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17985756708438762670829959560049370730","series":"1.2.826.0.1.3680043.8.498.72279334147075529866530189217253202750","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18090147030578547920423840630147682826","series":"1.2.826.0.1.3680043.8.498.79454006727337376915948969157375062011","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18114567737638963627123562895002602285","series":"1.2.826.0.1.3680043.8.498.23467885495122077521785372618975256610","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18176586296602067352845573876750324761","series":"1.2.826.0.1.3680043.8.498.11614254232170219813739471373531403350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18245500498843512203508664874652003124","series":"1.2.826.0.1.3680043.8.498.96651944418049140684492424848939241072","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18350185433331570523543207086298834067","series":"1.2.826.0.1.3680043.8.498.98348688142931913662353075247486579628","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378","series":"1.2.826.0.1.3680043.8.498.26020146442472659078480945124844158460","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18518393141873346710197645305647124682","series":"1.2.826.0.1.3680043.8.498.42382015534809337394137984322043148431","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18592535981451766048952107991674227570","series":"1.2.826.0.1.3680043.8.498.69606524653319303861014449647495607428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18653838887148864162027930218548124748","series":"1.2.826.0.1.3680043.8.498.25505968278831792073430194787273825323","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18680059761479598675335345641208680006","series":"1.2.826.0.1.3680043.8.498.10476133642443822291066907110434893272","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.18721466442792788108431096912844719622","series":"1.2.826.0.1.3680043.8.498.13372537514985125904935852472738897933","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18792568294433896490648198192877693850","series":"1.2.826.0.1.3680043.8.498.98102326384022384988135161740715554081","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18848481788456572234982139368351538716","series":"1.2.826.0.1.3680043.8.498.52959431480388852780687128886669137377","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18899625751876258520147203904985857313","series":"1.2.826.0.1.3680043.8.498.85362301305559162166027020467893662856","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18993960105596277135383396226525760874","series":"1.2.826.0.1.3680043.8.498.13224489138164491086503163950149421219","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19104350033985745058396432133828748480","series":"1.2.826.0.1.3680043.8.498.19829385089418190516726031526133020018","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19183011851352129901149443549646899548","series":"1.2.826.0.1.3680043.8.498.67887189610945625226915099508031445241","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19197037365998103318720742154440921358","series":"1.2.826.0.1.3680043.8.498.54619743366900864821089219777681777241","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19342628541833173544083802327412975225","series":"1.2.826.0.1.3680043.8.498.11553861702611850860890237400794140036","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19419878169415950073207624537163068568","series":"1.2.826.0.1.3680043.8.498.99288163693602784830342006423470088249","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19489875090732464985703624680756212359","series":"1.2.826.0.1.3680043.8.498.18252599830100162219561577105406538673","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19516545000256238042886469152307943182","series":"1.2.826.0.1.3680043.8.498.30510322146426203101201864422791710235","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19601428113073026637547887698342670296","series":"1.2.826.0.1.3680043.8.498.94296561593232394477053075772753979874","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19706775737718777290660344285046025310","series":"1.2.826.0.1.3680043.8.498.97002864023829673915009609427160287839","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19813267030974842239752491935957206955","series":"1.2.826.0.1.3680043.8.498.79694482717998342806956329779151130477","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19830663206351645385520268848711462246","series":"1.2.826.0.1.3680043.8.498.10093187984474868500349331230606019194","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19871496221665809577186118817075295760","series":"1.2.826.0.1.3680043.8.498.91995224230478586911197067423156920097","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19895710326297085463251295629948896948","series":"1.2.826.0.1.3680043.8.498.12374570687407247855221117424345872696","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19977273190231806393528180968361858122","series":"1.2.826.0.1.3680043.8.498.50841582421178995690011595958018396496","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20063306448698185569553477478719533390","series":"1.2.826.0.1.3680043.8.498.67688831152973306672465372239485233979","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20156361590688072955579948030116579099","series":"1.2.826.0.1.3680043.8.498.11326115210237828278674363965976312056","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20196165276643741937915346646793735100","series":"1.2.826.0.1.3680043.8.498.82264323217238564959827546032077957466","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20232953944759707220316171133595554284","series":"1.2.826.0.1.3680043.8.498.11363309414439675180176323191754729462","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20301796413163444642416918108994411350","series":"1.2.826.0.1.3680043.8.498.94148050239879735287016843081842617875","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20411068889491114730824265411248922690","series":"1.2.826.0.1.3680043.8.498.53374829398831722505513358794002082554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20488337421252584047930242186072775007","series":"1.2.826.0.1.3680043.8.498.38732057308647421985491915016016128470","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.20521417572710861366819886462837836465","series":"1.2.826.0.1.3680043.8.498.38332139832025121381291298672987640480","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.20587514285427671071934631922521946816","series":"1.2.826.0.1.3680043.8.498.82184541955473744443239611592037623135","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20640858366001462094044810443654811785","series":"1.2.826.0.1.3680043.8.498.31650080465618438430523212806679583621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20781255814255169306555078952156148784","series":"1.2.826.0.1.3680043.8.498.21371259768518048185071594843172302750","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.20850785960533583051456342090773230767","series":"1.2.826.0.1.3680043.8.498.13414813205009785412796705082197151799","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20933151653103548000061448393771614270","series":"1.2.826.0.1.3680043.8.498.11112070321128533021430860816288563789","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20953536352252551462901259297974956430","series":"1.2.826.0.1.3680043.8.498.83171250629713292101967120126611100629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21024377993947321291280779867493407199","series":"1.2.826.0.1.3680043.8.498.89838621652280574264638799996429371172","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21116046804798247223618072736212944615","series":"1.2.826.0.1.3680043.8.498.53368308984301699972971631608070411300","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21239630702461934662855021164119114203","series":"1.2.826.0.1.3680043.8.498.10675440579383180515456206609457714791","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21312254534068992054745674608909722454","series":"1.2.826.0.1.3680043.8.498.11814564742376856923567906401193955690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21403015320378798778538823075316902483","series":"1.2.826.0.1.3680043.8.498.12264255313612692393191787547712952938","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21449012857520712901884874638467532958","series":"1.2.826.0.1.3680043.8.498.78454970986687302386434760922475204291","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21556073772857867149576001337847113661","series":"1.2.826.0.1.3680043.8.498.12290615000300759695033261817422366439","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21584859831565112159503971937497384030","series":"1.2.826.0.1.3680043.8.498.10611814951360956088900540202383004575","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21634792491972658759276025394685307053","series":"1.2.826.0.1.3680043.8.498.86252377903266836424419458094816119188","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21726746202429506433532477857404854137","series":"1.2.826.0.1.3680043.8.498.50016497694788387918127169515047434638","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21862609534273524003153225756948913437","series":"1.2.826.0.1.3680043.8.498.11884836226472210399035529235445864726","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21904421326144238681996713485932410596","series":"1.2.826.0.1.3680043.8.498.13104042897006907776097952315530083102","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21954482915702797371609582020715955762","series":"1.2.826.0.1.3680043.8.498.23614350857393654552350098025989867927","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22007253197523827371600763959266047295","series":"1.2.826.0.1.3680043.8.498.54398150421721668346501735388684104661","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22027914323057444746912555313058666950","series":"1.2.826.0.1.3680043.8.498.37701523017165849723682681168729420937","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22088734384869383587217556574382319654","series":"1.2.826.0.1.3680043.8.498.57302123765465276236665607684861658747","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22119720417177816110156744770171496965","series":"1.2.826.0.1.3680043.8.498.64829786486536630948464935450584335543","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22137222939487357840166329770642627189","series":"1.2.826.0.1.3680043.8.498.49302343959027622739144379710212737862","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22275537949387641831010030150398063643","series":"1.2.826.0.1.3680043.8.498.73916414268695913902178175662667038544","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22357754074308826479280023393343731000","series":"1.2.826.0.1.3680043.8.498.62731812544988454420374535124433881817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22398731807962276404889164827238996005","series":"1.2.826.0.1.3680043.8.498.36499890710517785688285285712738238471","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22456525904575663955604077506103354060","series":"1.2.826.0.1.3680043.8.498.81723686668852084354990889993706156375","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22508124641661887430614105328316053644","series":"1.2.826.0.1.3680043.8.498.55495339953017349829393840783140625711","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22594249996322566888700984133307337128","series":"1.2.826.0.1.3680043.8.498.79719234863479159186696953984193889484","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22617494262793136174349268379191631799","series":"1.2.826.0.1.3680043.8.498.62433322486559794415402752903228840241","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22678957404909201599340396039237525550","series":"1.2.826.0.1.3680043.8.498.62900573817509184736974378888945140723","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22803020644232321685575897661362804423","series":"1.2.826.0.1.3680043.8.498.34502479403640662135460931269393539081","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22938598188283919466040132624303635817","series":"1.2.826.0.1.3680043.8.498.12240635137754156301037433932618630960","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22990786875064680564672188701015740170","series":"1.2.826.0.1.3680043.8.498.83742047537653682574301309911043355202","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23063675880057833741455102729816364091","series":"1.2.826.0.1.3680043.8.498.97742092296325539714471232293217558966","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23073694267101759922738121732233014980","series":"1.2.826.0.1.3680043.8.498.12781349281590949585384212688766791229","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23107903961514228607587963030481481744","series":"1.2.826.0.1.3680043.8.498.77672531461672479794926983892080370500","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23182005205627494484490011351673355568","series":"1.2.826.0.1.3680043.8.498.94858703494067451594142450362968933451","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23409201792139701921731647470152180932","series":"1.2.826.0.1.3680043.8.498.84912665976672177232557111341390799402","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23518857399095666263158269685495375084","series":"1.2.826.0.1.3680043.8.498.12665519216900994023481577756458599552","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23677916081296740815989149501889343663","series":"1.2.826.0.1.3680043.8.498.10884505545404627900694775598577945784","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23713320796530316370039202981138376061","series":"1.2.826.0.1.3680043.8.498.68372221597166303712990944660788325991","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23733147518259581649562651915997449088","series":"1.2.826.0.1.3680043.8.498.42188426405570945267869255262009336931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23833933822461809237193998300076847210","series":"1.2.826.0.1.3680043.8.498.76392502676362061372559739475165497986","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23939313916974849526459717576344063277","series":"1.2.826.0.1.3680043.8.498.17779640430046740334312730451682461624","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24012076387083114283116476459425015413","series":"1.2.826.0.1.3680043.8.498.10052879259579738300991388496456603397","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24101792822866991033671006059283560916","series":"1.2.826.0.1.3680043.8.498.37698888610335873883886105860096200978","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24219103446788192052503003052696523200","series":"1.2.826.0.1.3680043.8.498.20467360391222853705882631529564288786","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24267153760943721146187182369810590631","series":"1.2.826.0.1.3680043.8.498.20681459897877022440731006873212098042","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24302889622350917085868032747474929504","series":"1.2.826.0.1.3680043.8.498.13376820864586010906175862754163783608","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24369332907731805266450307536793220148","series":"1.2.826.0.1.3680043.8.498.11907334142398113941512710881411426028","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.24502332819381293694020055836255028614","series":"1.2.826.0.1.3680043.8.498.27829671829347639712783364607999706142","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24555528844026121007453886375651403314","series":"1.2.826.0.1.3680043.8.498.87869236251023237433175498895287268149","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24635589616467409099788838058892886889","series":"1.2.826.0.1.3680043.8.498.12172405046052761412603871232663392655","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24679905857546029995247839380830345841","series":"1.2.826.0.1.3680043.8.498.12181467542559493681973969554785015233","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24758830794084473905762034032291806038","series":"1.2.826.0.1.3680043.8.498.15682388138867287606013641869935336686","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24826041178550854190310664677274404684","series":"1.2.826.0.1.3680043.8.498.49065522942671354058167017519077767045","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24875052319550288230112434151792125187","series":"1.2.826.0.1.3680043.8.498.12947516651140576689949508020008947009","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25006652485159423287561656594171884310","series":"1.2.826.0.1.3680043.8.498.10325477107615557050655272451438345040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25103708168569906591706406794385334861","series":"1.2.826.0.1.3680043.8.498.52072487145766227898902390934934788043","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25145588904873072866960435474159832047","series":"1.2.826.0.1.3680043.8.498.39141507692504256701420468382119735773","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25188196199869378521949969245728443004","series":"1.2.826.0.1.3680043.8.498.12599785702932895642490945656611397807","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25252553477605568387898480443616118724","series":"1.2.826.0.1.3680043.8.498.90251684030991874638989332507517969030","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25266231062774974303723359806535863810","series":"1.2.826.0.1.3680043.8.498.12863631741719721083855811519874625449","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25359770839263995354733835365510047109","series":"1.2.826.0.1.3680043.8.498.11305807499339011780757582719118860130","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25440342930186170531037102906687829898","series":"1.2.826.0.1.3680043.8.498.13293606723922519238574251669348223512","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25487225187305802150215261512092051199","series":"1.2.826.0.1.3680043.8.498.21806478666433648866400815813726815082","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25549974159486441938843396591725289882","series":"1.2.826.0.1.3680043.8.498.63234699200695444824333214718517187912","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25604547074218147115077644853162295379","series":"1.2.826.0.1.3680043.8.498.87102702322517474315075792778661274517","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25635522141852534580088120697977308505","series":"1.2.826.0.1.3680043.8.498.62616938116595741434477858622651691354","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25663557443141366332431300959089922567","series":"1.2.826.0.1.3680043.8.498.76661753436956468477673209275900576180","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25679930222284447338484621953345520487","series":"1.2.826.0.1.3680043.8.498.27107818985566007727126733151744019399","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25697539332996211770844878649053316629","series":"1.2.826.0.1.3680043.8.498.82368703707891418350906120412087770429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25766387275525303860953321689718246405","series":"1.2.826.0.1.3680043.8.498.12327801996462058571367534371868963186","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25792216289349936450377008453709520260","series":"1.2.826.0.1.3680043.8.498.11565777756724407718390426619783377840","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25856396695552621816673203639618612020","series":"1.2.826.0.1.3680043.8.498.51693813682292308736611044319384266177","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.25904473139226718487560978103077541725","series":"1.2.826.0.1.3680043.8.498.12068607763887646715186810798378965240","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25953117039631879376428439063599611036","series":"1.2.826.0.1.3680043.8.498.64193787102513803240805093983703685317","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26004572902429209855865822977387283547","series":"1.2.826.0.1.3680043.8.498.10983196350673845856931784577337707881","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26023120405608635089544039694812003604","series":"1.2.826.0.1.3680043.8.498.79779651997225151907548873078335295861","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26150637525321126007832687915610995568","series":"1.2.826.0.1.3680043.8.498.68224159724434465541943980539642743463","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26257425287121280175418060407044075705","series":"1.2.826.0.1.3680043.8.498.12048493378264083008228385733631475830","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26333849754345698117396119042432557445","series":"1.2.826.0.1.3680043.8.498.68981841200965705492769602041685362899","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26373657290121957625263322898744712730","series":"1.2.826.0.1.3680043.8.498.92703988147781206971094781896415900808","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26474456265903975319674669470473661691","series":"1.2.826.0.1.3680043.8.498.10249584210438679866503470505287257405","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26512333939158074470511885252122191522","series":"1.2.826.0.1.3680043.8.498.59516771087289399009345207562276311096","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26619973716086136503589916868066348957","series":"1.2.826.0.1.3680043.8.498.49209061752725455857089713087240622651","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26698126402727061400851465377574886112","series":"1.2.826.0.1.3680043.8.498.10950221556960455523216653570654366777","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26792665054823243997838322477587733719","series":"1.2.826.0.1.3680043.8.498.49282811821535219968212549380043566468","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26848412380762311150802301982233072694","series":"1.2.826.0.1.3680043.8.498.66636030372521710296523051069869572000","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26880562799522930458745136318439622601","series":"1.2.826.0.1.3680043.8.498.12020564254050860308544150473835820895","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26989584479346775218799121753035575730","series":"1.2.826.0.1.3680043.8.498.66722960037301568910948937916439042819","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27023892731021470476500135946317920771","series":"1.2.826.0.1.3680043.8.498.11710864553788101841777199717036974654","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27059975738703546675398620281849285863","series":"1.2.826.0.1.3680043.8.498.60858825744700372941852899699445091789","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27161418589407291713644384596541786246","series":"1.2.826.0.1.3680043.8.498.42011930152970972092180536632543772383","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27191574009501966309382307682020135126","series":"1.2.826.0.1.3680043.8.498.47930091758997612669879534135310583421","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27197413293500684424735285039923168926","series":"1.2.826.0.1.3680043.8.498.35496205181006359389698046522561940506","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27245171350765293537022546751529180545","series":"1.2.826.0.1.3680043.8.498.36239668680213559387898119761598984166","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27319908980978184820438244981035567211","series":"1.2.826.0.1.3680043.8.498.12026683701110779868063049196966321765","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27339999385861467858422021400023533486","series":"1.2.826.0.1.3680043.8.498.64786788936862617526537579955012347247","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27422742748182689233651649793019570511","series":"1.2.826.0.1.3680043.8.498.21685972052112960545303784989832159767","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27456622705361831116491505615162380604","series":"1.2.826.0.1.3680043.8.498.12298676920246788266998876587310951308","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27593710915588175306316595263625481601","series":"1.2.826.0.1.3680043.8.498.45875608360947838055752829716962939699","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27644321991318957631114727895883331622","series":"1.2.826.0.1.3680043.8.498.52271027499773005902476328956986428597","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27656830051511853680197276412365020780","series":"1.2.826.0.1.3680043.8.498.12388427988605855724495048002231666010","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27681651277422558344520227187021583924","series":"1.2.826.0.1.3680043.8.498.34091113027771103856841199771963152946","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27723573213501259053792522608339812759","series":"1.2.826.0.1.3680043.8.498.10042261984457869507473297990209777112","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27793455627426963285485957211105588905","series":"1.2.826.0.1.3680043.8.498.67883296575838734555696737211685446638","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27883371281368994280894952749865160316","series":"1.2.826.0.1.3680043.8.498.47150649534984289795015335595839631846","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27925145307825795804342879375835424449","series":"1.2.826.0.1.3680043.8.498.82664644354064128566985732330373953044","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27946605663086243142356841956045514227","series":"1.2.826.0.1.3680043.8.498.12473518633714511441790738005244133734","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28011057282528460306257420499560926354","series":"1.2.826.0.1.3680043.8.498.15767148471223425241708395850301477365","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28040645186226259590182363324146810031","series":"1.2.826.0.1.3680043.8.498.10980327269888079988940056297512744580","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28134711422142706763867845558971485241","series":"1.2.826.0.1.3680043.8.498.18346554803767967832223702164737042027","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28218476802410349760103243990109474475","series":"1.2.826.0.1.3680043.8.498.56774897999071276725762783696212866708","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795","series":"1.2.826.0.1.3680043.8.498.55309159342678412834689609531200990471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28295265939946677460838974253299313798","series":"1.2.826.0.1.3680043.8.498.17461266856164125750776023064851742369","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.28333127304382930639614800697959296306","series":"1.2.826.0.1.3680043.8.498.38471933637474677643830289671962713573","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28425280718252632810092859612928047098","series":"1.2.826.0.1.3680043.8.498.58538535192706044789422942655001140265","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28488216314048990109149780234417422948","series":"1.2.826.0.1.3680043.8.498.80843077291347862224714423628942124958","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28579372595938810820078670002733005121","series":"1.2.826.0.1.3680043.8.498.94518535779486092332368257454669613669","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28626044336302232656987963623687293705","series":"1.2.826.0.1.3680043.8.498.10884727238355330006715724520802877510","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28634248756653012357434232845788268719","series":"1.2.826.0.1.3680043.8.498.50945933472617748581850471594795663954","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28655385212214986812679045835025619123","series":"1.2.826.0.1.3680043.8.498.36592440761318478770807278170498196642","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28686498221359515429184385385077600687","series":"1.2.826.0.1.3680043.8.498.10818885543492340557731213398905143143","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28770594655815108010532953376271504596","series":"1.2.826.0.1.3680043.8.498.53149522939619416895983160858736483804","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28873180831637763124652074775631752918","series":"1.2.826.0.1.3680043.8.498.86465742693557961238990424261998143793","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608","series":"1.2.826.0.1.3680043.8.498.11438210485488424020029704142703549114","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28951331578236330522646806432827370713","series":"1.2.826.0.1.3680043.8.498.62356991125479909541939134834275379498","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29039872266866623711060910233238440882","series":"1.2.826.0.1.3680043.8.498.10421687213507572060100968971627151837","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29227028592609694700827329867343028944","series":"1.2.826.0.1.3680043.8.498.20326170412114854529943692613755436228","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29294899952044311440141172596030689586","series":"1.2.826.0.1.3680043.8.498.89295744999034175746207493035329834074","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29354469534263517460113311852821643665","series":"1.2.826.0.1.3680043.8.498.20034255449502523343612328701848056807","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29371692234061436334815897970798625595","series":"1.2.826.0.1.3680043.8.498.91172542512879997184707830584288385693","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29418130858182890216867847991592582370","series":"1.2.826.0.1.3680043.8.498.10464310482373088262687446069490531194","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29473278891551292295080887778714088878","series":"1.2.826.0.1.3680043.8.498.10239497517580367586424412702659903662","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29735852958813434703969490187047665950","series":"1.2.826.0.1.3680043.8.498.10437984850413431191601804589102331133","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29741673719687535294395668150749894623","series":"1.2.826.0.1.3680043.8.498.19078353791719111780213416656294079838","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29794301133358685635777478470865282350","series":"1.2.826.0.1.3680043.8.498.10525973090127525199245484219382823716","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29866827083314645739444078187995496049","series":"1.2.826.0.1.3680043.8.498.49182001084882911047213797117242618411","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29976611818313906970687136815346573471","series":"1.2.826.0.1.3680043.8.498.65519576855866307207830134667124542524","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30066410959464367576375502522994033142","series":"1.2.826.0.1.3680043.8.498.46767896249477842741073392851465641332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30150613585152527569197215147135001922","series":"1.2.826.0.1.3680043.8.498.81465540891257784507998398935500600532","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30174064884443577177838026250637902388","series":"1.2.826.0.1.3680043.8.498.58445936259190390291885179531198837880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30233578499383083757206211752001832047","series":"1.2.826.0.1.3680043.8.498.12489737268904475536766245759728843144","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30248682294786626375369374249874501600","series":"1.2.826.0.1.3680043.8.498.92139942785919610552771792048237854520","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30264599677934095429161619981918944098","series":"1.2.826.0.1.3680043.8.498.41077234374886157948001871866821680674","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30381005430261460472880465705496199540","series":"1.2.826.0.1.3680043.8.498.68736040768804538774515494497425601550","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30423643741419627789701272644909134762","series":"1.2.826.0.1.3680043.8.498.35463376383565139019496109663226506341","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30490857770169699336468319880130884989","series":"1.2.826.0.1.3680043.8.498.11483539117590170354706106131069786357","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30537361712821464927469618193893850465","series":"1.2.826.0.1.3680043.8.498.10336787054576409759696671850137238418","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.30613528335797277620843542280889823612","series":"1.2.826.0.1.3680043.8.498.60698472327505392919689467385009926171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30673411238679998084995204090753297949","series":"1.2.826.0.1.3680043.8.498.10075850923678625558751478713371734745","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30746167840159843733412534591988948270","series":"1.2.826.0.1.3680043.8.498.72986544783094840909494432855642348980","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30763696628968291934469221453461209579","series":"1.2.826.0.1.3680043.8.498.10552235174703550393451849162388769721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30836611607735372299752226690388090391","series":"1.2.826.0.1.3680043.8.498.94743730538490529759039418564214757276","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30842406081180392627544336691360658070","series":"1.2.826.0.1.3680043.8.498.12587352551554696226622183864304417800","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30947630416867960170317316514422933066","series":"1.2.826.0.1.3680043.8.498.11738168882864593583050468510342507452","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30998941785989496591079894962766360976","series":"1.2.826.0.1.3680043.8.498.12894328905885541290514240965979994834","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31082592133742929889611857821843859701","series":"1.2.826.0.1.3680043.8.498.10289668687098608329970122267155116489","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31137162827930164367134811210741362416","series":"1.2.826.0.1.3680043.8.498.70132964232752334489155609436430729059","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31175171249406459042393597291141275669","series":"1.2.826.0.1.3680043.8.498.10144475926277948957560313410381417146","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31193772347652083986607183406944630716","series":"1.2.826.0.1.3680043.8.498.26150829645641947248841990605285832198","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31257542772176806182696506252925365119","series":"1.2.826.0.1.3680043.8.498.41891950106951295405636846446325207479","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31282366761674650660824706600044115805","series":"1.2.826.0.1.3680043.8.498.64526504370821051155729034814896814286","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31373064397225665483187291274194812486","series":"1.2.826.0.1.3680043.8.498.26630813579674647868972219535863352034","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31461240229123701082444627629841504099","series":"1.2.826.0.1.3680043.8.498.64146847745325599700142998289794744620","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31565705236407722278921437231164204914","series":"1.2.826.0.1.3680043.8.498.15771249767027999811152636111928625746","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31745644522310786426860265012373331979","series":"1.2.826.0.1.3680043.8.498.10147801545165415288778414875263422942","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31792234576444017394182974951507838927","series":"1.2.826.0.1.3680043.8.498.53636821062436630092720753976480414899","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31842732064173292828619477783097732181","series":"1.2.826.0.1.3680043.8.498.58618007156070812831754042070810928935","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.31898017986762601111404430885432769100","series":"1.2.826.0.1.3680043.8.498.13122958683644875989138898084858144307","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31949224274318638796196151105617844024","series":"1.2.826.0.1.3680043.8.498.65571872746013080822906271141595308133","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32109377199577933060124357239881959602","series":"1.2.826.0.1.3680043.8.498.11262329881658076025507573365689140580","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32172672595181417103935593872811182816","series":"1.2.826.0.1.3680043.8.498.10061753038850571612100729907998252038","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32241219263926672894433308808653719495","series":"1.2.826.0.1.3680043.8.498.23461323100591602817084410256142963832","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32299120582647333604770626110392750631","series":"1.2.826.0.1.3680043.8.498.96397702796886305814608193260923346766","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32329374180424300100441619205766536205","series":"1.2.826.0.1.3680043.8.498.12646225400755142334976056931069215814","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32417506901165787885027994386927321654","series":"1.2.826.0.1.3680043.8.498.11467473148378400081600010197605394037","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32501049887140121289143479773995481982","series":"1.2.826.0.1.3680043.8.498.12394605585478495650354839789338499145","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32513010438338970418994141548516096399","series":"1.2.826.0.1.3680043.8.498.31382637975243715938162059899579264566","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32593583441236180433075435541777683664","series":"1.2.826.0.1.3680043.8.498.58078243907799500829157590960476307943","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32624291809937483520460431326945616461","series":"1.2.826.0.1.3680043.8.498.20507202626171779659393214804895684549","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32723296226443325838892542529446214677","series":"1.2.826.0.1.3680043.8.498.61742871301254273801142469805464624227","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32784816158214124982843838883515640569","series":"1.2.826.0.1.3680043.8.498.40474193395221627556170427852793293905","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32862727544234948830522036721585095753","series":"1.2.826.0.1.3680043.8.498.43419033510565901141327663342179415752","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32996271790302034744448837505729856569","series":"1.2.826.0.1.3680043.8.498.49935206720648444230667989097255604864","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33085434860684110795966817803985794300","series":"1.2.826.0.1.3680043.8.498.11113069029145600227507412228423573354","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33215387283623093392227819856970670071","series":"1.2.826.0.1.3680043.8.498.22055759302623008432187044533870433667","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33284228284209148349668722931307588402","series":"1.2.826.0.1.3680043.8.498.35036795555005617914460898754114033340","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33324670630108603340869986205298694836","series":"1.2.826.0.1.3680043.8.498.13420170857279251416750619700177157973","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33393833442961030521286257650328223086","series":"1.2.826.0.1.3680043.8.498.10168925767348345750420021743597838536","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33427080769740419588952225424281396684","series":"1.2.826.0.1.3680043.8.498.62418404297630186322854627882646293606","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33584205642521728950676369503690551550","series":"1.2.826.0.1.3680043.8.498.91981559001664915135269769864048359071","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33620268340279901383196638228030589893","series":"1.2.826.0.1.3680043.8.498.13162125548596943515250247237101075217","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33709759248055631658322056320849494088","series":"1.2.826.0.1.3680043.8.498.73821954825825145367605030537825999011","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33834227140201599591317512187502720859","series":"1.2.826.0.1.3680043.8.498.39754014732866667213169077656627170017","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33885566709305241744786004177260915650","series":"1.2.826.0.1.3680043.8.498.11049701856745289026788998648837305994","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33957165685646407808982253731894148028","series":"1.2.826.0.1.3680043.8.498.91649193709483308694582795470195984020","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33997019840409023169542161093833920630","series":"1.2.826.0.1.3680043.8.498.11769637216355690545766976680153701931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34058295828142876023659425586740832405","series":"1.2.826.0.1.3680043.8.498.95932246056592312246812615733523966841","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34096149829586714035779314398766243514","series":"1.2.826.0.1.3680043.8.498.24339645712666858685179375522364797693","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34206043272866138622475050809785944941","series":"1.2.826.0.1.3680043.8.498.38609012253602413973348375461425265735","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34294522885703132478119231619149118745","series":"1.2.826.0.1.3680043.8.498.84560431237532933063235707514942596295","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34379812274299658721362253287768041613","series":"1.2.826.0.1.3680043.8.498.81685730186520461556502150660203967147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34441662343660776323013762699640240389","series":"1.2.826.0.1.3680043.8.498.88018136755123871404163218809977436230","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34501976934829436985900782482912019868","series":"1.2.826.0.1.3680043.8.498.35888944050785832148136547924535404721","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34572444983416782410188788741787280444","series":"1.2.826.0.1.3680043.8.498.40949071238003618458399787504799716130","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34636178772986348716257313987840123906","series":"1.2.826.0.1.3680043.8.498.28301611137198126379728613555819909352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34690162612328981644511154212357885455","series":"1.2.826.0.1.3680043.8.498.60804217678609138842476612961840682957","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34740783941206992242367609204835771599","series":"1.2.826.0.1.3680043.8.498.12820157211823903304859099255684594529","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34791780913322886484171421115861604853","series":"1.2.826.0.1.3680043.8.498.10723475473617500711100552856308904588","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34858782760614383566891001610985452165","series":"1.2.826.0.1.3680043.8.498.67830276063031494303229655707699492522","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34895742092137516664772577251243433579","series":"1.2.826.0.1.3680043.8.498.38640198570832429021233879159766934706","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35017973257585430544005763350601015208","series":"1.2.826.0.1.3680043.8.498.84548990433543537802840356042864831585","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35064504053970754752690367203246822139","series":"1.2.826.0.1.3680043.8.498.10069827528287999786917671939517939695","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35121262802959257590413582180924083425","series":"1.2.826.0.1.3680043.8.498.79209940972336397773509749834772154290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35210451107308788915675910467662426761","series":"1.2.826.0.1.3680043.8.498.12951707088915265923314717373520033854","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35273590880523815297008790288567354883","series":"1.2.826.0.1.3680043.8.498.10640121505821889986949870720285057310","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35338198410583984337908932327787063577","series":"1.2.826.0.1.3680043.8.498.56074104995044367849843435839577866875","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35549628666222168741574331855838621990","series":"1.2.826.0.1.3680043.8.498.10584458457220430784675828784874878828","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35633121799529901420381085923067525212","series":"1.2.826.0.1.3680043.8.498.98093373291582264909258138380728401642","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35664741425496814074030858561751287211","series":"1.2.826.0.1.3680043.8.498.20102003258506404796308076440177281499","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35769002792415947424087098083674166777","series":"1.2.826.0.1.3680043.8.498.56630439647767553676435432715000811942","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35790589718590759254855277551289842944","series":"1.2.826.0.1.3680043.8.498.75653765965925583035143333884822022671","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35826402289277331555736683320175912112","series":"1.2.826.0.1.3680043.8.498.13521936012287431956737393197653358181","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35853404993418331599021266761280794201","series":"1.2.826.0.1.3680043.8.498.46829231309380398622785608568523336097","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35892665011493087075493127252825546314","series":"1.2.826.0.1.3680043.8.498.99138049163872461567609682467144873068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35923183633813281697521537744486885300","series":"1.2.826.0.1.3680043.8.498.74118984302643045339573060383223078288","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35932368810339127867306228699757809449","series":"1.2.826.0.1.3680043.8.498.11631679242860424886110968019028173742","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35965770865730278787229040370963322662","series":"1.2.826.0.1.3680043.8.498.98048770808231186614885892710937917828","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35991881811942962264382690224767291006","series":"1.2.826.0.1.3680043.8.498.18999594658804197255126812494445050530","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36083794505435625841225237415909011547","series":"1.2.826.0.1.3680043.8.498.13138314812906443676229128676758240438","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36158956431673597472934120620368656246","series":"1.2.826.0.1.3680043.8.498.23438379719669658547532028313480201515","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36185754164795053628650116141620545139","series":"1.2.826.0.1.3680043.8.498.62866412471357540900715837583278255158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36235892729383338522021416304853301198","series":"1.2.826.0.1.3680043.8.498.22503311840445524015280485087172708191","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36314809560682514762544085885005264167","series":"1.2.826.0.1.3680043.8.498.43967344730312781820642659559877370792","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36431753977367847587395456605556703040","series":"1.2.826.0.1.3680043.8.498.10029175578742911134222249240104542598","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36521162836904095186654423648490092035","series":"1.2.826.0.1.3680043.8.498.52793820069406889486245070045293275915","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36599566036331923007586730435849419272","series":"1.2.826.0.1.3680043.8.498.10441700635066160711802023481175146913","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36648377536792309029975432419163533533","series":"1.2.826.0.1.3680043.8.498.46603460332195920659758441063118318551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36743562653862381001280112332273495985","series":"1.2.826.0.1.3680043.8.498.11827079243527376945301998473353391359","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36812284468308169396147366740331573100","series":"1.2.826.0.1.3680043.8.498.36814988399108615227918826257817731618","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36891467514776207809384804066253854193","series":"1.2.826.0.1.3680043.8.498.54137064041943429052914059314696098339","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36924091633099385100942228071545017565","series":"1.2.826.0.1.3680043.8.498.77263300413918280349751113580401850551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37005775889857426034942587344620888398","series":"1.2.826.0.1.3680043.8.498.12445112892792657149722242869228165531","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.37030593568962187244853961673610373876","series":"1.2.826.0.1.3680043.8.498.12110966186829565301855671616706489036","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37057548998853865829402084168010676727","series":"1.2.826.0.1.3680043.8.498.67649742548688741720552290704762242590","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37147839676713817130306478240352601535","series":"1.2.826.0.1.3680043.8.498.50647735796453585269462405188559610283","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37270291131952413969044156276264153316","series":"1.2.826.0.1.3680043.8.498.32032914707603206724533937695336393670","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.37347132459817985668038603215490355489","series":"1.2.826.0.1.3680043.8.498.22361502004015548450259567655986600465","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683","series":"1.2.826.0.1.3680043.8.498.62241028832981711826910629867693998502","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37427993203943111265113924891058049338","series":"1.2.826.0.1.3680043.8.498.86553169044556705797084140530436430628","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37447511763482014027579634343525200849","series":"1.2.826.0.1.3680043.8.498.11761399984320250000956691706278239497","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37485075271886447709158414670394650122","series":"1.2.826.0.1.3680043.8.498.12914911734365709739241274578620866877","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37519350121906786941281770770693979271","series":"1.2.826.0.1.3680043.8.498.62890796086885405469385912175554764744","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37572710131396388050286115883363501023","series":"1.2.826.0.1.3680043.8.498.28925465731473425673301837129782358696","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37591186611460576939009950935820113903","series":"1.2.826.0.1.3680043.8.498.12758801145097983461045801008115363747","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37644754953609976838298110418046213654","series":"1.2.826.0.1.3680043.8.498.73087825654886321525444281377357182262","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37729566534337521443213426693965453255","series":"1.2.826.0.1.3680043.8.498.10160486548260682784847305613166845471","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37833587429731221455928642963031995680","series":"1.2.826.0.1.3680043.8.498.24027939485299249865408996288050986555","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37846940236952007502147453013489359606","series":"1.2.826.0.1.3680043.8.498.11292337056688932769833796228810491998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37885316830093262977685802809269466252","series":"1.2.826.0.1.3680043.8.498.61050011371067631748396931511171214074","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37936061471366103885185528435327650981","series":"1.2.826.0.1.3680043.8.498.34667104890487349493793037424456431064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38010459551726691601986492657296948013","series":"1.2.826.0.1.3680043.8.498.13075232786895046857981113440068012018","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38056024157303576597111203925091929489","series":"1.2.826.0.1.3680043.8.498.10013256123553319041990966901203362989","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38085359363401170553191582038767900323","series":"1.2.826.0.1.3680043.8.498.23636929524529988014865017298984479136","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38173249623713271734596884672770674290","series":"1.2.826.0.1.3680043.8.498.10092438766027455184332619004644028377","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38238963443939714344730201261512911003","series":"1.2.826.0.1.3680043.8.498.82179043784827241574940839476848207741","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38303182087206935163155565441234078389","series":"1.2.826.0.1.3680043.8.498.23950577667555670223156924002342818205","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38306262614552497897681886368029860358","series":"1.2.826.0.1.3680043.8.498.78255436042446110609569563505480358063","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38361762550587464310603057450403132552","series":"1.2.826.0.1.3680043.8.498.87638378603685201352950101669000650334","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38373204312077453422262347769111605003","series":"1.2.826.0.1.3680043.8.498.10461090507523938547581889269479032612","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38427680930662301114332563903957471851","series":"1.2.826.0.1.3680043.8.498.10686467261406494801445631247948799710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38518069950656326194016175319535174299","series":"1.2.826.0.1.3680043.8.498.43534776163625870397785066211403085485","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38554121831397903041461691318968151267","series":"1.2.826.0.1.3680043.8.498.26968986406475204696989989640457692746","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38591915802992557518846862828618375715","series":"1.2.826.0.1.3680043.8.498.12643178794637069413169242106731200168","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38617606118499305781729905815407382451","series":"1.2.826.0.1.3680043.8.498.30573441844161757723278420036899706051","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38655145317957160440692625317823254736","series":"1.2.826.0.1.3680043.8.498.86026565195151832959498872990178805673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38717329577625255474350472405748992219","series":"1.2.826.0.1.3680043.8.498.79095860745428957370735004152029078426","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38795933423959135237935740989825843893","series":"1.2.826.0.1.3680043.8.498.11649852811390609742358593999850287204","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38844149478205202385909887748058004820","series":"1.2.826.0.1.3680043.8.498.61476702004046808390568896023354995868","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38893522232848525819919092148478879455","series":"1.2.826.0.1.3680043.8.498.77515014135954270348133815810781898089","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38912715505731269998129663552068223373","series":"1.2.826.0.1.3680043.8.498.29266039195719669038929181470023287134","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38928323954764175259962816410233336718","series":"1.2.826.0.1.3680043.8.498.38401507330085691510769885694159694427","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38974023308452365783768081643215200948","series":"1.2.826.0.1.3680043.8.498.19835186477875212394515435123174922487","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39020025030013167602991688044932114354","series":"1.2.826.0.1.3680043.8.498.39030510642941718339609532621048133231","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39064406065534097226924475518005522427","series":"1.2.826.0.1.3680043.8.498.11312757813418473346163359511013609768","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39135770176150551756142329948474103964","series":"1.2.826.0.1.3680043.8.498.71654125932167697763821160782392426433","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39197976722851762826265579540678924584","series":"1.2.826.0.1.3680043.8.498.12708319369411917946483359593718769538","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39275838957651450909114440721012584183","series":"1.2.826.0.1.3680043.8.498.72854825281938868801257638804102340035","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39375796600811714870169173423682043709","series":"1.2.826.0.1.3680043.8.498.12146784370017146606725310817680165652","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39468803878805753474151381590569125402","series":"1.2.826.0.1.3680043.8.498.47916514425762126852288936064018974231","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39507184510584030713933330702725991027","series":"1.2.826.0.1.3680043.8.498.30380359117266449622554321139644752311","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39624624539323722688093816284210287611","series":"1.2.826.0.1.3680043.8.498.12242058484533296580820974319390000631","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.39642276506855996181597769215091644995","series":"1.2.826.0.1.3680043.8.498.12727917933149553212258642779741081033","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917","series":"1.2.826.0.1.3680043.8.498.12942535021932890436700462849325179868","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39895705984068714346274689335576595443","series":"1.2.826.0.1.3680043.8.498.61934499337450598850419885279461337897","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39954567501513957418422043089587980092","series":"1.2.826.0.1.3680043.8.498.53460355451092606558915340877957421937","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39969071491383654382076256213867635377","series":"1.2.826.0.1.3680043.8.498.16040785505284573843354799192281178467","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40086005762552976451418579375198597540","series":"1.2.826.0.1.3680043.8.498.51878847790671110502458968245691150428","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40154708689850542672289658249059660788","series":"1.2.826.0.1.3680043.8.498.18262270406427638912565391885299027831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40239104167848425290754606013326180693","series":"1.2.826.0.1.3680043.8.498.46295907476678870883770804352341996043","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40257126480600927758308319831518876543","series":"1.2.826.0.1.3680043.8.498.51253250577204381477620199956916735284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40317527233615709136328116805254969837","series":"1.2.826.0.1.3680043.8.498.82659460020314123690041695745942892466","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40386781168330493993852288092654487818","series":"1.2.826.0.1.3680043.8.498.11899349136218168580495609503742047851","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40436886205373732563782477802719203679","series":"1.2.826.0.1.3680043.8.498.84786823210501751449532019160534682143","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40487147942341753806704765355921322161","series":"1.2.826.0.1.3680043.8.498.51489537472842631790253508528507761266","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40547250250566982489293057895661861722","series":"1.2.826.0.1.3680043.8.498.12320302693215633762757989717078479754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40612173652156509478338094582396889655","series":"1.2.826.0.1.3680043.8.498.56156567113543254630523085411260921192","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40627043074325458726412346974457616938","series":"1.2.826.0.1.3680043.8.498.73206408079620293463870900171441398201","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40703974473944295875248011843170371394","series":"1.2.826.0.1.3680043.8.498.83024274760719950901804711339231118284","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40765829900885943055669918091033014654","series":"1.2.826.0.1.3680043.8.498.10456970909369984124846709680078688457","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40828786094337186697222449293183309391","series":"1.2.826.0.1.3680043.8.498.94691887802675184992105615213942547929","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40898220266193841898869905714214454387","series":"1.2.826.0.1.3680043.8.498.11313546462622325726674711253567961027","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40967435550546093348221485942169879116","series":"1.2.826.0.1.3680043.8.498.77758425902021536767545254456154887289","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41036022211588451079713314564568887994","series":"1.2.826.0.1.3680043.8.498.10978627097490650215348774307046203098","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41108421820714108307078614688022098388","series":"1.2.826.0.1.3680043.8.498.76345182554739432174805359527870938870","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41129090437522031508960438281451672212","series":"1.2.826.0.1.3680043.8.498.12355257592382152554066479770544376507","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41254472410580125134596211010943842213","series":"1.2.826.0.1.3680043.8.498.99218499182289668780116208813958859168","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41363979247025384202932112601795525615","series":"1.2.826.0.1.3680043.8.498.36587215154091857388315044136404381055","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41382780862853765983216810855531499640","series":"1.2.826.0.1.3680043.8.498.30211349941252885931994679169369508894","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41440206520769979754790425253631809212","series":"1.2.826.0.1.3680043.8.498.62872618336234477395423936480211549581","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41587640208769641326416798167574152049","series":"1.2.826.0.1.3680043.8.498.18908560411803081249764230324799019269","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41605033773891953796694219242501252567","series":"1.2.826.0.1.3680043.8.498.12586045452763217896608833329820056760","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41640481150213029810772014344448781554","series":"1.2.826.0.1.3680043.8.498.68670841046989205258742142658803136869","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41662013720606493874198348776219875496","series":"1.2.826.0.1.3680043.8.498.17033866339782537366307796585758861749","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41696746845418651508748386392327996143","series":"1.2.826.0.1.3680043.8.498.11891517304135789991830741865544490972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41751049067043481681879071436091069601","series":"1.2.826.0.1.3680043.8.498.64956737892980438048025014181410186791","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41776233288569201013095816839442161537","series":"1.2.826.0.1.3680043.8.498.28672053003136168817926260424090371542","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41837648239243141231469586963862388683","series":"1.2.826.0.1.3680043.8.498.67309568703184788857141501756251202462","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41856137734822068230231773222820851271","series":"1.2.826.0.1.3680043.8.498.16654778754722541691576756268387122061","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42018410640565840140895012243840256810","series":"1.2.826.0.1.3680043.8.498.28336842808300424829258779983281255861","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42023788139360998656558751921918860808","series":"1.2.826.0.1.3680043.8.498.83527284941424003440417899363330689092","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42057349781448469704191384429694247725","series":"1.2.826.0.1.3680043.8.498.65304877352852657349564423169901859775","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42162585695624963017992042571207363427","series":"1.2.826.0.1.3680043.8.498.73124421961414236350219346103056802905","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42225165085995728538373532053227662192","series":"1.2.826.0.1.3680043.8.498.25363228211179891123801204988721393819","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42389063782063360163798575455522539633","series":"1.2.826.0.1.3680043.8.498.35708818646916196471660659373798699430","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42476443240785038044326483794941265936","series":"1.2.826.0.1.3680043.8.498.11611248701937661261441363009509594452","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42499092649567991851673654940784921876","series":"1.2.826.0.1.3680043.8.498.11595901705542016100749782539475430949","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42587087495729026068405824960397052997","series":"1.2.826.0.1.3680043.8.498.10588424586834579319399339275727468108","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42609322631348207494943622999125822017","series":"1.2.826.0.1.3680043.8.498.53582504484511609843827492729535116082","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42768603719459834345001093623865167866","series":"1.2.826.0.1.3680043.8.498.13031397803725770552468759037326792387","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42778524243264061817587398263842942394","series":"1.2.826.0.1.3680043.8.498.11891773528717921891487382385858601290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42829490507052740062127365747846030214","series":"1.2.826.0.1.3680043.8.498.11752746995614411058737668001690335024","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42896117106271721157113072562739356381","series":"1.2.826.0.1.3680043.8.498.12780515741149612329498400487154903375","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42942844751304929338846069579289911684","series":"1.2.826.0.1.3680043.8.498.95385667333716520958114484831244736696","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42990542863216044717944418712887800295","series":"1.2.826.0.1.3680043.8.498.75307636053326994529585915509818064705","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43046713539641995841029654207663845887","series":"1.2.826.0.1.3680043.8.498.46462143848599944195400469956234564374","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43127139928509033152155587499980767364","series":"1.2.826.0.1.3680043.8.498.10313130387660051047309348069367416839","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43256282639827216310036033378453939750","series":"1.2.826.0.1.3680043.8.498.15426982644755843911632677943707033557","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43325226037168674077736336919956196684","series":"1.2.826.0.1.3680043.8.498.25116955717101495883697051757061764095","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43355923025859051425546451466843844346","series":"1.2.826.0.1.3680043.8.498.16136275492082186359733031187980940422","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43432499267536761263202158662502957684","series":"1.2.826.0.1.3680043.8.498.16632119142636797850379378674435739967","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43445351322695981909807931874796758490","series":"1.2.826.0.1.3680043.8.498.66392453846289292120065963287047774243","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43587191909319947256183198986671106743","series":"1.2.826.0.1.3680043.8.498.70339529634787008447204466610926422273","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43709699920591091261137846976272625163","series":"1.2.826.0.1.3680043.8.498.12774546863268345307185412657411197869","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43832771919542075229536000564653592465","series":"1.2.826.0.1.3680043.8.498.44802313421942487636836119617889361918","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44002376543340544410495269060824194475","series":"1.2.826.0.1.3680043.8.498.44599899014301569690948560066350990698","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44149403030093858908988179411336839587","series":"1.2.826.0.1.3680043.8.498.66238923140346031434581882473390475160","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44261256084247541720465769177956875383","series":"1.2.826.0.1.3680043.8.498.71052968279148530028967493742768506587","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44333089398623004498721836325723633455","series":"1.2.826.0.1.3680043.8.498.12250949083562675911710575117399617803","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44365814642869065241166313231424697474","series":"1.2.826.0.1.3680043.8.498.51282057832668275381339822813572635407","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44424196147366603683090692394096032727","series":"1.2.826.0.1.3680043.8.498.53198365843090451265560068409274553630","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44476475746458439091354236934188679569","series":"1.2.826.0.1.3680043.8.498.11752465756834608864149078084366756279","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44629238055006732101555855930071163428","series":"1.2.826.0.1.3680043.8.498.19013932812636573172849704791694401117","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44713330360073884105636085566230396530","series":"1.2.826.0.1.3680043.8.498.66503296065832965495633527196252750685","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44758766790156297849747618145458535803","series":"1.2.826.0.1.3680043.8.498.98035315536902683660795012756137706789","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44875593707781066742286566388388035791","series":"1.2.826.0.1.3680043.8.498.10642122502633339343657470076124623359","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44879493336213912007673875382158420811","series":"1.2.826.0.1.3680043.8.498.10357810434675101822561414411808175567","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44941174190375343987327566550204051010","series":"1.2.826.0.1.3680043.8.498.41735302178073956894224366600888679192","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45059917085750212913588683242522799083","series":"1.2.826.0.1.3680043.8.498.49904916560427023163558370586129841927","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45082393649917309927267478072569185694","series":"1.2.826.0.1.3680043.8.498.65285442592369383354797615628086880826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45207812477654339877341538582090218293","series":"1.2.826.0.1.3680043.8.498.71690735438547011524485287190612529367","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45273393220765922863913956738614783632","series":"1.2.826.0.1.3680043.8.498.41681971461305169339232132998123186995","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45316453179223822297459860764883030492","series":"1.2.826.0.1.3680043.8.498.10062808718118342708343930875375542876","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45332185607602495786497298658937575959","series":"1.2.826.0.1.3680043.8.498.58566441006661022706229629515586332730","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45380312679924593883892686293576260616","series":"1.2.826.0.1.3680043.8.498.38753175240476272827856977625471924581","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45406212529450909512649737062964080190","series":"1.2.826.0.1.3680043.8.498.74485896509928837768523391330719738546","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45463321406810606039579747001339703552","series":"1.2.826.0.1.3680043.8.498.72733763913983210401646608634782404415","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45519502487291848581539950159353124045","series":"1.2.826.0.1.3680043.8.498.93773322168996530059493849839852887403","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45630051398709774279703407401383545642","series":"1.2.826.0.1.3680043.8.498.44412901608619173277718686478159675229","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45755198489211010290512228906807170657","series":"1.2.826.0.1.3680043.8.498.48414112119273690247135239367695222942","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45811964949115542204306831813385709206","series":"1.2.826.0.1.3680043.8.498.39863975188523831910221948737770253903","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45868806544202339567395643235910857549","series":"1.2.826.0.1.3680043.8.498.11742661850590017134898521714768808046","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45968957856865486437256472516713628614","series":"1.2.826.0.1.3680043.8.498.32757549991898480192262510893769798869","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46036209147177723970966724006242055397","series":"1.2.826.0.1.3680043.8.498.23580693959339804493194723278857149408","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46123761840464615814604226041496278995","series":"1.2.826.0.1.3680043.8.498.11621004856350607287870852257527274118","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46222691720198967161306520058359099856","series":"1.2.826.0.1.3680043.8.498.49814091991066216153477592673774831229","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46268820642949526111807948525363811264","series":"1.2.826.0.1.3680043.8.498.26529168179306750466971775399458413678","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46297226877524853395345229615300658523","series":"1.2.826.0.1.3680043.8.498.87409203695348073717675291359202078671","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46335849264221717416382765091672040855","series":"1.2.826.0.1.3680043.8.498.68440228418075246130318657043572637841","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46427879286435198611282055405541286766","series":"1.2.826.0.1.3680043.8.498.43196298385659073184927516819770116237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46506161284114284997220751358509505380","series":"1.2.826.0.1.3680043.8.498.10569151415092391928215641378777602520","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46700895961751940291334058396794206479","series":"1.2.826.0.1.3680043.8.498.94778906892265843768218055788860870259","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46745172905542117506863575411125924241","series":"1.2.826.0.1.3680043.8.498.10388068854509388645591025124962474792","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46797419946872257347118338518757027995","series":"1.2.826.0.1.3680043.8.498.27429498454600341944716898264180396812","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46842734152337184482152392783260161845","series":"1.2.826.0.1.3680043.8.498.16633580018688823174198555739111111271","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46909729013880708074310736136355969956","series":"1.2.826.0.1.3680043.8.498.82697507257166888373719856499918137822","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431","series":"1.2.826.0.1.3680043.8.498.53095564691930218629541635037170588568","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47061252152535181371772020997295000029","series":"1.2.826.0.1.3680043.8.498.58879983762363919089341907441036407496","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47094981922544354899183959858078311972","series":"1.2.826.0.1.3680043.8.498.62393593351290419061950869897302101587","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47239511009810375185594384849101241004","series":"1.2.826.0.1.3680043.8.498.37491266082270959080190795151489938207","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47268599732083036147289040424392838620","series":"1.2.826.0.1.3680043.8.498.25038210327598632268719205163146888204","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47339093185444044578247608022398494955","series":"1.2.826.0.1.3680043.8.498.75065516001850619284068847083366992468","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47465512474718256570933149642267926825","series":"1.2.826.0.1.3680043.8.498.11711989166919537641556058501264168103","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47531034194300620953276025401018051246","series":"1.2.826.0.1.3680043.8.498.45738913434039723352372637318187480654","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47666607805434428477774854287399691400","series":"1.2.826.0.1.3680043.8.498.33728178871169872204773753530308824834","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47707670074001059720671838493302874710","series":"1.2.826.0.1.3680043.8.498.12910413013171964410637092383453724559","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47855330524708374523226349383799712473","series":"1.2.826.0.1.3680043.8.498.43316563173767586468967856053982802437","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47870074144891988520855535250618228390","series":"1.2.826.0.1.3680043.8.498.76463723776195899281784860547011633705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47949251060660360953300340380324578835","series":"1.2.826.0.1.3680043.8.498.72776483266259440721638119251391842613","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48068031587855513429324313927339963541","series":"1.2.826.0.1.3680043.8.498.18426004358039155482770204957622047940","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48308485987218296338173512060246238691","series":"1.2.826.0.1.3680043.8.498.50969077806815765662077192434347893569","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48367523758673249284754003765836877239","series":"1.2.826.0.1.3680043.8.498.60467624187021640501014621708928342360","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48435973333068748044485658920315789799","series":"1.2.826.0.1.3680043.8.498.71020581027720589724316396812795224436","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48520238688122533624681796433258211236","series":"1.2.826.0.1.3680043.8.498.86691119338199392710609209192927954948","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48564780460694033783288540237012068889","series":"1.2.826.0.1.3680043.8.498.72181219637733027877337868094690928527","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48697800450463192944610588533290320545","series":"1.2.826.0.1.3680043.8.498.17471574295776086022990063661313915009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48780598610511011488578963416186905560","series":"1.2.826.0.1.3680043.8.498.73829016441924985593541788465682511471","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48818931767017230678539144135307491171","series":"1.2.826.0.1.3680043.8.498.36744020597283645288197728112470981487","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48872083192774100826667097552356058756","series":"1.2.826.0.1.3680043.8.498.16887932217001996186485567420881566564","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48933020429865163393792899740045803587","series":"1.2.826.0.1.3680043.8.498.12768664226383161706301551515474571075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49025249926859976858200443939208207124","series":"1.2.826.0.1.3680043.8.498.34659433684565983190207755433532211221","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49048739684287632681242054218807611262","series":"1.2.826.0.1.3680043.8.498.45797037706269478415415830230138676456","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49163335688297423808344168197551387860","series":"1.2.826.0.1.3680043.8.498.60926834289900657950749794475063278255","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49219586198038249030222134813685536295","series":"1.2.826.0.1.3680043.8.498.92887283673218101721034581101890884417","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49347024975371539869871751159907613638","series":"1.2.826.0.1.3680043.8.498.12899629518331113973095699992179391395","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49394404966674633152360206835416205330","series":"1.2.826.0.1.3680043.8.498.71741193407811557522932512822075934261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49481811111224238761880576486796603139","series":"1.2.826.0.1.3680043.8.498.32672605252067578296137987228081726416","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49500752658775616590911569693817273205","series":"1.2.826.0.1.3680043.8.498.68264146186701406301196096475934816338","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49583348226932515919984811293816232513","series":"1.2.826.0.1.3680043.8.498.16947880872103749465711570446111999459","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49660659440268002826070581802625582554","series":"1.2.826.0.1.3680043.8.498.10108739743808970566086723402422161936","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49818874332345742217581267858748819921","series":"1.2.826.0.1.3680043.8.498.11202400453001933516691791775974566303","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49850672111501136862025523789202033809","series":"1.2.826.0.1.3680043.8.498.47067099186349764932537900992441032740","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49866252311077397175807461727885923657","series":"1.2.826.0.1.3680043.8.498.95647887673248837374073341036050591673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49887758195164927372912584780941789021","series":"1.2.826.0.1.3680043.8.498.27450521707311540870965989121729489494","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49942556484410764813681696157580687854","series":"1.2.826.0.1.3680043.8.498.78657280650051042689079064320386647824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116","series":"1.2.826.0.1.3680043.8.498.95154401635694198563048001755689791287","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50089015780087008332657793589955395952","series":"1.2.826.0.1.3680043.8.498.68890713806740299232749171555089841045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50144715912379960437506930683683218333","series":"1.2.826.0.1.3680043.8.498.11351789854955834244598603000682307097","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50215409013618353346619153290049205718","series":"1.2.826.0.1.3680043.8.498.67167035522496233120080062429211200869","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50259348432783983788186173025555513832","series":"1.2.826.0.1.3680043.8.498.47334312073694683211811888162494286308","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50287512028986928372350279452197816661","series":"1.2.826.0.1.3680043.8.498.25747284277860755769940216329402492543","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50392110746914984873009735588879049321","series":"1.2.826.0.1.3680043.8.498.12765312076272447119504702400409439685","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50458745240631577290185672298591447264","series":"1.2.826.0.1.3680043.8.498.87888689973955745287235001101882263172","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50501020658943435430736717390350999395","series":"1.2.826.0.1.3680043.8.498.12493846575598668993866211830018361765","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50571692547974449396635351108344029536","series":"1.2.826.0.1.3680043.8.498.12497431191064628576251786354665859470","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50667444467788034101157821576803778774","series":"1.2.826.0.1.3680043.8.498.14582914458760378678395554680058521218","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50687743970686088872471288288960579306","series":"1.2.826.0.1.3680043.8.498.80648083528595831962796574168584632675","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50707407856055788475528375847872674241","series":"1.2.826.0.1.3680043.8.498.19979849893833081607223956611200795230","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50780438941220579448947609376330661130","series":"1.2.826.0.1.3680043.8.498.87186935634170489725900810205939860486","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50813123035221058777203670043892828582","series":"1.2.826.0.1.3680043.8.498.11123721473140515656072730818953353924","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50844227724977857739867646104945090414","series":"1.2.826.0.1.3680043.8.498.48407256682700512347722891357561343264","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50880891504121357724276091096406248845","series":"1.2.826.0.1.3680043.8.498.10293906837485274981408725924095307244","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50969691234820675847736765735133138324","series":"1.2.826.0.1.3680043.8.498.11888351946920644428870909071070000300","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51054592064718900209123281525185013961","series":"1.2.826.0.1.3680043.8.498.91746355653300855132991691435147504623","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51126807380879212261057967703844943102","series":"1.2.826.0.1.3680043.8.498.81675489457265125536918176595283492622","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51215847567976999838748705030071176367","series":"1.2.826.0.1.3680043.8.498.74126991582256179961291017712460391529","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.51351386796317851725461803790273870995","series":"1.2.826.0.1.3680043.8.498.43934573577208116853503209422563770247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51427159850769972151411336025891779423","series":"1.2.826.0.1.3680043.8.498.12799238984600249923024356680785586591","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51509214978951980397275258374070843967","series":"1.2.826.0.1.3680043.8.498.26135111930379933803277795944980268862","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51581188134109925758379838961317006116","series":"1.2.826.0.1.3680043.8.498.12555229871656573003790611958114203662","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51752963441899958127321113249182970131","series":"1.2.826.0.1.3680043.8.498.57751706291252882769551402128191852543","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51770131897828080663537020812721042704","series":"1.2.826.0.1.3680043.8.498.22400006676742661916750964670639870726","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51813125531970590883098432751115674643","series":"1.2.826.0.1.3680043.8.498.81249951574433853360620356184385916742","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51831304809871808790657397599860793132","series":"1.2.826.0.1.3680043.8.498.91100049390738363411552457848597913161","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51867133375333949773527521703402111601","series":"1.2.826.0.1.3680043.8.498.56465312822519677201643390929578521954","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51917043703891091519740084517167861965","series":"1.2.826.0.1.3680043.8.498.94650826921694668105023278855496826481","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51981019698056120061499970572334688555","series":"1.2.826.0.1.3680043.8.498.71138294959276248688894887993943561119","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52010826440192227660457107121818389761","series":"1.2.826.0.1.3680043.8.498.90386194879954722911746489831209925284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52059522418267239652374290810994711032","series":"1.2.826.0.1.3680043.8.498.12970269837582163643258267604746024026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52110163572817528199125320120779141456","series":"1.2.826.0.1.3680043.8.498.12334696238939393544585948564385881204","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52138967568492774495485454357106269286","series":"1.2.826.0.1.3680043.8.498.12541836355613384584309940648985459160","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52330656126848153803758928167944040055","series":"1.2.826.0.1.3680043.8.498.35584640256908501639422140317646262380","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52390908285903728334239836699232920065","series":"1.2.826.0.1.3680043.8.498.25684844482798737066123514533591868120","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52437948508356178175558637950043479411","series":"1.2.826.0.1.3680043.8.498.12820820724044894123655629267354214608","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52475562989926669452280424331791927661","series":"1.2.826.0.1.3680043.8.498.12429710719883570001559814093071665507","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52520288400853832269613213204540385687","series":"1.2.826.0.1.3680043.8.498.60087671561280943877275426426379291810","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52636981391578580436180193211113340410","series":"1.2.826.0.1.3680043.8.498.19338764122237791977310487514582982878","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52745530063404662086111315503240395535","series":"1.2.826.0.1.3680043.8.498.99167880137217114161995231517127545439","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52846023545870016653608198538437574931","series":"1.2.826.0.1.3680043.8.498.37485573813927669393227740920399862428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52937412433120578046842194545406797236","series":"1.2.826.0.1.3680043.8.498.71719857189654539591437413667260261867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53025489296610571044246428509154950855","series":"1.2.826.0.1.3680043.8.498.90707909662265171042146354215402405481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53080172184945962593713560671746700445","series":"1.2.826.0.1.3680043.8.498.92025063985966399913579423159312091434","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53131430415327479511358970615140982847","series":"1.2.826.0.1.3680043.8.498.10561770634134232304531030320798936672","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53156118232564605912649705207438612782","series":"1.2.826.0.1.3680043.8.498.18670314822539224118209678660102458310","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53174810138171412044410050984734412146","series":"1.2.826.0.1.3680043.8.498.86933529573392632516575863771248775592","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53271710112621372496324855809391580130","series":"1.2.826.0.1.3680043.8.498.11486821237325036782873544058039857458","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53337684515627962545396495541158882262","series":"1.2.826.0.1.3680043.8.498.95849797983906783380050620313365560540","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53361558447470435467999344560545063264","series":"1.2.826.0.1.3680043.8.498.94170848127157305849798172768767164377","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53393741562201846141435914434030220216","series":"1.2.826.0.1.3680043.8.498.11347857453911464043261797811326343183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53528011450161116437223332448942321596","series":"1.2.826.0.1.3680043.8.498.68962735408683973749510804465497634269","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53587920256280027683523152568832253423","series":"1.2.826.0.1.3680043.8.498.95562646799467953380977812799946595053","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53737349894831365159124471910871800583","series":"1.2.826.0.1.3680043.8.498.11287796590773782166190535366941025198","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53797236075631231042400235117445767744","series":"1.2.826.0.1.3680043.8.498.18894685625694412305303183224689632925","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53893586200869613140666806653041177558","series":"1.2.826.0.1.3680043.8.498.12934823928605245289311401993277572526","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53968943793571323467397128102531786394","series":"1.2.826.0.1.3680043.8.498.59394319907617757897299216856280666802","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092","series":"1.2.826.0.1.3680043.8.498.12060020803768477792550783184943558951","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54100786607660749952814251186461344633","series":"1.2.826.0.1.3680043.8.498.21881229220109302655581108858549731199","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54132814503202891434380002378663763655","series":"1.2.826.0.1.3680043.8.498.64214117001205001738028680331180712795","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54164004416963296439322252627774426375","series":"1.2.826.0.1.3680043.8.498.10391827751217329197423351364975724681","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54280954106925918796002359856408530043","series":"1.2.826.0.1.3680043.8.498.88042680129188899708380852283616921267","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54316764984549863227549326352573342232","series":"1.2.826.0.1.3680043.8.498.23621500367528989585949211865845851510","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54374735423703474541190464716308417985","series":"1.2.826.0.1.3680043.8.498.60698982273882425815091569917317119242","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54448516923758211534546401490623824012","series":"1.2.826.0.1.3680043.8.498.36537998105997744380849216718324725583","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54482848766921329667469114743547203929","series":"1.2.826.0.1.3680043.8.498.16899123316756782260037079782036205955","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54589880111295324708950851057241927191","series":"1.2.826.0.1.3680043.8.498.69186923531712441197646790403011514736","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54628782412849994943720963251113534589","series":"1.2.826.0.1.3680043.8.498.62507701878511014477591146451592573100","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54689086194769265957947122616025330642","series":"1.2.826.0.1.3680043.8.498.45237431716415062469567127307248696719","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.54699196872730350880681018674893216245","series":"1.2.826.0.1.3680043.8.498.85893952380944272627508776194716336520","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54761306437713414745748829128854749469","series":"1.2.826.0.1.3680043.8.498.89486915970888472635658689880556980067","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54833384793167149128371137537527194757","series":"1.2.826.0.1.3680043.8.498.11614547918629514567472569576401278917","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55008913229055942530957507434366806883","series":"1.2.826.0.1.3680043.8.498.84981480307520126498685759066811425091","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55022631713888448062201243978634363406","series":"1.2.826.0.1.3680043.8.498.95117958825463861185419087390259428134","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55067956175569778132589128971795053431","series":"1.2.826.0.1.3680043.8.498.34713506740349916578131376425477604384","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55183153701332792661581741475876132309","series":"1.2.826.0.1.3680043.8.498.78723237032349693052240786943491428731","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55332521421041708438748125666498042763","series":"1.2.826.0.1.3680043.8.498.66908737781661153468290884170542915918","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55344353927048912237233441917779875587","series":"1.2.826.0.1.3680043.8.498.61145034196151112324670459796775330598","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55415479446253262335685494509470808516","series":"1.2.826.0.1.3680043.8.498.10157179452721584748854322594803309753","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55583505147281684100202399148099429721","series":"1.2.826.0.1.3680043.8.498.49439582244962016660591612867758346527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55687452977471862155042703116192295932","series":"1.2.826.0.1.3680043.8.498.32311841257638940567366824745431844955","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55713753582074871470943388423906540792","series":"1.2.826.0.1.3680043.8.498.36208870925398175755374325344056478303","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55771181380353653474896884649540028856","series":"1.2.826.0.1.3680043.8.498.11036914294036234900721729777868767368","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55850385284038070407841702672827533728","series":"1.2.826.0.1.3680043.8.498.10353661917785250454481135938575978860","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55887747667305814945719986974538814207","series":"1.2.826.0.1.3680043.8.498.18916623970016492017969051860483575739","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55971152627433130850649668964887611410","series":"1.2.826.0.1.3680043.8.498.45036097233431283621699760439504460395","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55991293001848799160041967098772087659","series":"1.2.826.0.1.3680043.8.498.59173525360046264689871973216142418183","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56045687004558569005240775213510192805","series":"1.2.826.0.1.3680043.8.498.85581931119233519691858741252546559323","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56091473614612257999053529249190958828","series":"1.2.826.0.1.3680043.8.498.52352363695693459708694769285603901730","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56156923369480999484275191793049573490","series":"1.2.826.0.1.3680043.8.498.11822692005308537763136667265368769962","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56188634971310070481285525830793231849","series":"1.2.826.0.1.3680043.8.498.55795007966187408596366271748156382386","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56269695574661037525483516531194514456","series":"1.2.826.0.1.3680043.8.498.54163703347661988113333013449268088080","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56428314239264997930645427543149285770","series":"1.2.826.0.1.3680043.8.498.58523164262230208481438834047051620382","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56506415277488978301157490514745400410","series":"1.2.826.0.1.3680043.8.498.12236723344041545244537085242510157530","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265","series":"1.2.826.0.1.3680043.8.498.31095500830059682825216378963500489309","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56575942633137496743464648679775042979","series":"1.2.826.0.1.3680043.8.498.26503815104360800716291148508955030635","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56715311294415711253166312546942356882","series":"1.2.826.0.1.3680043.8.498.12595297814704836177816469725082360617","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56786556618312090924315296993756843070","series":"1.2.826.0.1.3680043.8.498.98351040496142690760890445750044716010","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56798070549054124854685511360777559033","series":"1.2.826.0.1.3680043.8.498.18640649658410847034643299971779381813","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56839374698050163738039760889156066445","series":"1.2.826.0.1.3680043.8.498.86737986652358173105378075127122665109","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56893426177755960410197244250599230068","series":"1.2.826.0.1.3680043.8.498.75883720979449513800221469211864638625","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56915079210789596803972894081756325026","series":"1.2.826.0.1.3680043.8.498.56857346040662192577042205959308056114","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56989113744241846544086409786820246754","series":"1.2.826.0.1.3680043.8.498.88650956295890075283706128289464824909","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56993108275828399240508695615212151102","series":"1.2.826.0.1.3680043.8.498.85328690965058576344629565367214309875","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57074072196714325300896734167161517369","series":"1.2.826.0.1.3680043.8.498.98692803953896266971857299089583196714","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57091203959244495164081406481825915856","series":"1.2.826.0.1.3680043.8.498.13052626453348151912997500871968288313","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57239450717434070615919278253974598533","series":"1.2.826.0.1.3680043.8.498.12425103791033660864267096357337973588","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57348281565196196401055821640494904475","series":"1.2.826.0.1.3680043.8.498.41883004558025425452945674528901544446","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57537915810250993237120953960404180449","series":"1.2.826.0.1.3680043.8.498.71211516167274549438586531880802020367","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57703608122587911055931213603182679454","series":"1.2.826.0.1.3680043.8.498.51690997280871488460790894542010029859","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57789603189240866041812385320399435864","series":"1.2.826.0.1.3680043.8.498.12377240034498714585087964953473295203","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57811199787891897795884073594875380873","series":"1.2.826.0.1.3680043.8.498.34186285667789481235808506133142829491","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57845032486422442404222046144765204905","series":"1.2.826.0.1.3680043.8.498.96424006715396436482146516024021304210","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57882532838957577543516026386452055977","series":"1.2.826.0.1.3680043.8.498.12702332494023011047318547841404363836","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57951268100539350409054562078476694936","series":"1.2.826.0.1.3680043.8.498.10809972020949643838794011273707905332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58143227319316521737029358930629086887","series":"1.2.826.0.1.3680043.8.498.33555161572111768714862433711281848237","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58250779958866595505585700517927225230","series":"1.2.826.0.1.3680043.8.498.11215038533992311378784326533352116232","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58280349821647977336008109443058641195","series":"1.2.826.0.1.3680043.8.498.80703110968491384858937592668351714389","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58346459085553680879151588730895136332","series":"1.2.826.0.1.3680043.8.498.64845052358433086149120315521738084627","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58404766256608483217059706069436050741","series":"1.2.826.0.1.3680043.8.498.76632328607370511017971106164062213622","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58475208045272914292874190001248518687","series":"1.2.826.0.1.3680043.8.498.45352378446514244026997960769353257788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58522891982092784622515590012748600644","series":"1.2.826.0.1.3680043.8.498.46796793117511577521460354716075555172","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58585441284202143792468863402617212167","series":"1.2.826.0.1.3680043.8.498.80162545614266206616457646586560794457","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58634494571833713296574414948455471101","series":"1.2.826.0.1.3680043.8.498.53640391237684225690037234682347967669","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58710813820037448352310278234947865485","series":"1.2.826.0.1.3680043.8.498.66207001273810787006225527322644793754","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58739535411760425418941874065260802744","series":"1.2.826.0.1.3680043.8.498.31991915662762721785336029828965329259","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58768083117076186082335984587845964130","series":"1.2.826.0.1.3680043.8.498.58023842327367812165189373807002348287","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58811528407321581206563412079643094538","series":"1.2.826.0.1.3680043.8.498.48166497037904694756424533464852633753","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58857477075095368361468658035369975572","series":"1.2.826.0.1.3680043.8.498.13151516325441759705143035396754331929","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58965370475364247222892582428665460530","series":"1.2.826.0.1.3680043.8.498.85325771287281028194524945087418229955","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59047534248122071167153110802170658177","series":"1.2.826.0.1.3680043.8.498.55202818606662673412875524904706813750","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59073122184069676919857712290159511305","series":"1.2.826.0.1.3680043.8.498.47572435301106289363639482267442828505","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59140342594072117638979402278277248612","series":"1.2.826.0.1.3680043.8.498.73562464802593333194928914131448199278","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59262240088132401543519466539885561578","series":"1.2.826.0.1.3680043.8.498.28761949526775768972950173260825833209","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59420153636832938628088507776889305823","series":"1.2.826.0.1.3680043.8.498.29910923725402038567519721566926623297","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59475691595504590121311599301170361869","series":"1.2.826.0.1.3680043.8.498.54093326606080780583382605527497113439","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710","series":"1.2.826.0.1.3680043.8.498.29846587191183567670535184626018287451","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59533384841491000088839469462294817273","series":"1.2.826.0.1.3680043.8.498.13263119737797114507578490477710465998","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59680220055968160813801840287391339651","series":"1.2.826.0.1.3680043.8.498.30549818471693252015018389500968020120","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59758232061190668535098343218115599228","series":"1.2.826.0.1.3680043.8.498.10422902385610242288188587675578950158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59835677513721120155185666132462195647","series":"1.2.826.0.1.3680043.8.498.12316748216580027057393689974048998570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59932425615255727459402684084295081504","series":"1.2.826.0.1.3680043.8.498.10924952968619236791646034721283392502","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59939977812549570931951749553467050226","series":"1.2.826.0.1.3680043.8.498.30334007939205996872995331872265506959","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59954207362160100069708115719895503197","series":"1.2.826.0.1.3680043.8.498.42517257065627405460236576402438882201","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60064080350033586611530748685370040101","series":"1.2.826.0.1.3680043.8.498.11792698483374486361887296963224032133","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60163612617486071760358965933890738162","series":"1.2.826.0.1.3680043.8.498.24134873400842451151222825004944947853","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60227430483444032531479844365716485643","series":"1.2.826.0.1.3680043.8.498.10732649912441277434464807857531029203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60339775486026531260673620094004244044","series":"1.2.826.0.1.3680043.8.498.58134515856801668842464731266394282848","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60430475172382878092821699001847797153","series":"1.2.826.0.1.3680043.8.498.29440453968874859713443006232516608670","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60439571175176992075370053747045234796","series":"1.2.826.0.1.3680043.8.498.11897515670252811573328413587331952753","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60465293103574559546982440125572490190","series":"1.2.826.0.1.3680043.8.498.71351919822820690212911649849931053155","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60492772672040498730937577391401159109","series":"1.2.826.0.1.3680043.8.498.68387737765777661562450425469236414624","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60582762894748063612925320003677861384","series":"1.2.826.0.1.3680043.8.498.12083740916880129509111590574831857251","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60614975796208617794180432816390721331","series":"1.2.826.0.1.3680043.8.498.89571444230437089902602667810354520241","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60723378470101669902158229125336455520","series":"1.2.826.0.1.3680043.8.498.21490025757609097982226867648512236011","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60840220723824894619895264673401129352","series":"1.2.826.0.1.3680043.8.498.88199497859146747052763172835898912365","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60965286164497194740320124904525623873","series":"1.2.826.0.1.3680043.8.498.98119785831505492276753965670090504976","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61043690805522768840878574610198420558","series":"1.2.826.0.1.3680043.8.498.28385003961900233934403873570223415197","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61140619849833544411463750382539885141","series":"1.2.826.0.1.3680043.8.498.12692471264687564175358312216352843340","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61220609061901632500124675031032586841","series":"1.2.826.0.1.3680043.8.498.81743691796939402698548787954302649341","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61238657014751597736608285205361041675","series":"1.2.826.0.1.3680043.8.498.10272784643552088131134449567513327193","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61333047594780344399535845209036589563","series":"1.2.826.0.1.3680043.8.498.54532495347707624697901644750307330406","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61456481310419978302697858056095086811","series":"1.2.826.0.1.3680043.8.498.96277991810811875714029752940828246101","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61518377417979517706237284901970943470","series":"1.2.826.0.1.3680043.8.498.19590369119635653577476867329502129777","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61554384977916135091630433680780524386","series":"1.2.826.0.1.3680043.8.498.10538365807185608843966137893889572151","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61581513083114643407028382481113097015","series":"1.2.826.0.1.3680043.8.498.10323156271931360840362270736561928157","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61797111379749329977228618245440305623","series":"1.2.826.0.1.3680043.8.498.44310080388864592919937266821037221141","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61904856234837992279531381374350177464","series":"1.2.826.0.1.3680043.8.498.78952990491791479704944118766485690781","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61956863718716027369068406125965902569","series":"1.2.826.0.1.3680043.8.498.12478071169947807265248987898567947665","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62030472630193758412620945408364583008","series":"1.2.826.0.1.3680043.8.498.39844588156533161930724408117122172818","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62154969360158054310282857093665911261","series":"1.2.826.0.1.3680043.8.498.22521548029357564983597320646913635134","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62205799963299902160268565319924399775","series":"1.2.826.0.1.3680043.8.498.45438885713280828772539117880859940164","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62251470468640645490157592046417355534","series":"1.2.826.0.1.3680043.8.498.85324620498609032479734691453343504260","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62383135836496954297324866452887364097","series":"1.2.826.0.1.3680043.8.498.44279493575481435618718992952121972942","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62400625562782184774973349983498962743","series":"1.2.826.0.1.3680043.8.498.10298928572345986891042200191968980320","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62437671010172828772205139640126159272","series":"1.2.826.0.1.3680043.8.498.99113447112555657091217556945071003738","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236","series":"1.2.826.0.1.3680043.8.498.98515298626079372374618896434085510643","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62752755711205966800746994120425231855","series":"1.2.826.0.1.3680043.8.498.99114979471650780401242718422625257347","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62776347582771469289738234607517546504","series":"1.2.826.0.1.3680043.8.498.73904340009173620670892958759642491215","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62918046703989259691005447883746340916","series":"1.2.826.0.1.3680043.8.498.28340360106926195078820502752225041717","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62928513795250694194589804784951257414","series":"1.2.826.0.1.3680043.8.498.10867337875706037829943715448959314780","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63002559681883387414876870926128159682","series":"1.2.826.0.1.3680043.8.498.59173737578587154605123233607285632653","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63069333313670617776227577192591993067","series":"1.2.826.0.1.3680043.8.498.92309026066778271143146446624411230320","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63105151375349194807502162946823966410","series":"1.2.826.0.1.3680043.8.498.95615034407823328007885276366553224240","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63151733210715530368323422018984705316","series":"1.2.826.0.1.3680043.8.498.16430620340935153791906404647159397908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63204735160524958513470115761010950181","series":"1.2.826.0.1.3680043.8.498.31989381559748856550327096317361968455","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63223220827342079068491460847713457238","series":"1.2.826.0.1.3680043.8.498.58579003983047361359207186452522693956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63279205896049798083339413931561428540","series":"1.2.826.0.1.3680043.8.498.41587355261256572059615948750152007118","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63320397279666280420359587173167195895","series":"1.2.826.0.1.3680043.8.498.80797310708758987431170887644241481661","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63424633611847022112809173449690688015","series":"1.2.826.0.1.3680043.8.498.11389000340877103328711389080374302207","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63439188800533762543950349022148547855","series":"1.2.826.0.1.3680043.8.498.10837292782492393036209509391766620238","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63519003265636561965405570006747767668","series":"1.2.826.0.1.3680043.8.498.11707117303091576895140578317291765989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63539809431911516912857504094611479480","series":"1.2.826.0.1.3680043.8.498.10417155712936102937790174940470335576","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63556690547679257097888724096967682731","series":"1.2.826.0.1.3680043.8.498.96823207454494403729486280178860252072","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63565089119380386830816381560603756258","series":"1.2.826.0.1.3680043.8.498.76104440137229874107171767725356509603","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63683477631564232018930819747238111809","series":"1.2.826.0.1.3680043.8.498.86926471095027623071896778697596637813","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63849051301265505051181245915656194652","series":"1.2.826.0.1.3680043.8.498.75996315239585467238718448457643872696","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63894147741855851067411708837856142566","series":"1.2.826.0.1.3680043.8.498.48744821012265682092411622489093098125","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64022949342538714157641824612944253302","series":"1.2.826.0.1.3680043.8.498.80517486696363605853052240738674786074","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64034149345689391514924451724232196567","series":"1.2.826.0.1.3680043.8.498.56283714095107313607477606327242223936","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64119798231341423127851203797783576453","series":"1.2.826.0.1.3680043.8.498.24265693779716117598255595242222212306","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64186424577324542519674308830472181373","series":"1.2.826.0.1.3680043.8.498.10325229758829749556646655546011774415","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64286017646659573966584053755363026715","series":"1.2.826.0.1.3680043.8.498.28231156191865461626997485731653546647","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64381606512330581256470279413868645789","series":"1.2.826.0.1.3680043.8.498.30026869523470066525287675433013439947","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64408775259838960958354835366765941676","series":"1.2.826.0.1.3680043.8.498.86673515719515829348538119976859767377","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64443149121726771900576109439951701105","series":"1.2.826.0.1.3680043.8.498.86594158605439781979916694360639878088","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64527059976573114931020581328924993069","series":"1.2.826.0.1.3680043.8.498.26905267876188408963618052656149521538","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64606794953061143951972249525094490665","series":"1.2.826.0.1.3680043.8.498.55109208994646466239216183370530444549","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64663285742626599570990862360902141188","series":"1.2.826.0.1.3680043.8.498.33838145234550556235812560223255489745","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039","series":"1.2.826.0.1.3680043.8.498.11346458840980855602023845654340426652","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64897285598346101578442697515294908341","series":"1.2.826.0.1.3680043.8.498.11186943294297094768644960079939651613","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64950600385134699009390197520158979183","series":"1.2.826.0.1.3680043.8.498.37408878015869290286874401930341594826","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65077085081642080527129904737510276643","series":"1.2.826.0.1.3680043.8.498.12696627953245672082762230475561013788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65124952546314712352837396207287384405","series":"1.2.826.0.1.3680043.8.498.16633711217206257861071554099852141516","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65187850119346448625449915281726338455","series":"1.2.826.0.1.3680043.8.498.71797651622196152541808638747134664467","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65244223225926362862970496735749098549","series":"1.2.826.0.1.3680043.8.498.11130683754916324789492342141492974876","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65322445533737673246312151513913460816","series":"1.2.826.0.1.3680043.8.498.88963488228697613487806114325591585474","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65438121008780366054026567981577676554","series":"1.2.826.0.1.3680043.8.498.71373638034577398945738347900970971270","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65524487168129064460609645228098420719","series":"1.2.826.0.1.3680043.8.498.11839643139892072710001543003899451163","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65547889329285432771702699287910809921","series":"1.2.826.0.1.3680043.8.498.61445796608455414243971534298514699545","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65607807464576681752987746458209389847","series":"1.2.826.0.1.3680043.8.498.10425948313696164109132063899695894955","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327","series":"1.2.826.0.1.3680043.8.498.49386996151147572908458775202466876717","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65756786850574169314934708606155033765","series":"1.2.826.0.1.3680043.8.498.24980216166327521994696172777661804951","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65788206282522750507559682036750692505","series":"1.2.826.0.1.3680043.8.498.97714029532884443444989507027070568931","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65834774574423727070722226151408235786","series":"1.2.826.0.1.3680043.8.498.71322678139236107061443731139693976064","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65896962360688978581417748865609785541","series":"1.2.826.0.1.3680043.8.498.22967851505521136238034632665948162036","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65939700606136711089565313916789162980","series":"1.2.826.0.1.3680043.8.498.75700624966820701786273920367490983804","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66020213859389864645118554257523478921","series":"1.2.826.0.1.3680043.8.498.12733723715713373999710461270514380443","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66031142975529812579758652269360365398","series":"1.2.826.0.1.3680043.8.498.14255000984769027187399566320730094166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66083855405250624607521810369199218887","series":"1.2.826.0.1.3680043.8.498.86511398670339211317510932924892125084","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66210455934011506415244876770725282769","series":"1.2.826.0.1.3680043.8.498.79242959172563177167246889045140308566","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66287049728466409037095442705228040084","series":"1.2.826.0.1.3680043.8.498.40538931074467395277510485970681736849","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66308195780676793917989368423009849637","series":"1.2.826.0.1.3680043.8.498.50393810560941961221843466008936207872","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66327751482076404294387513492434577338","series":"1.2.826.0.1.3680043.8.498.11416690840082424080088261501496033799","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66401771232064170830621215861092532421","series":"1.2.826.0.1.3680043.8.498.24496945471271037714947220567229014285","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66489634638847834550694493585147280754","series":"1.2.826.0.1.3680043.8.498.11000326800806757682835283015118458314","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66640124635222298792902508374800337726","series":"1.2.826.0.1.3680043.8.498.65619428715823901612906602209426105384","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66694519343321893230666089565893349853","series":"1.2.826.0.1.3680043.8.498.13026239692059573908650878295033168527","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66841103065389483583108279900721522084","series":"1.2.826.0.1.3680043.8.498.27384506294408130972778777855061184749","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66960891947495601371173426277430301127","series":"1.2.826.0.1.3680043.8.498.44853551854475472278659266751276701739","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67001329390153622359718581388215312710","series":"1.2.826.0.1.3680043.8.498.82589890820861753509096062367891337337","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67157948144868269956661697924341234924","series":"1.2.826.0.1.3680043.8.498.10460485878483105547201094265537558844","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67189969532330535369638450306601659967","series":"1.2.826.0.1.3680043.8.498.62623546380905335026661258641836404987","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67238774585857531338533947812568490258","series":"1.2.826.0.1.3680043.8.498.29693935867150799452372464104395993888","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67374476726034608591738910363595695123","series":"1.2.826.0.1.3680043.8.498.24537268537426318062597273180795015781","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67404282038417089622852480023919627784","series":"1.2.826.0.1.3680043.8.498.44345621861560586730032217133864647591","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67468397709097947104298147948850473142","series":"1.2.826.0.1.3680043.8.498.56260449347093779579431716569671432929","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67498362978754615554767658151787188674","series":"1.2.826.0.1.3680043.8.498.10434299174621715523658137213176582947","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67601368724828563564325678719126190797","series":"1.2.826.0.1.3680043.8.498.12055620206699557869493968990013252909","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67620512607520186685421323991166816451","series":"1.2.826.0.1.3680043.8.498.37532972168562668875685995557610180338","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67705068860320494485379949506807098172","series":"1.2.826.0.1.3680043.8.498.80733050632648781547787109959340433874","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67739091465243228820766982459725849814","series":"1.2.826.0.1.3680043.8.498.10623657413008423494228198631193755922","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67775301856843451215029610725114809378","series":"1.2.826.0.1.3680043.8.498.56613369743021181509550100916688950448","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67817378785531423675449308236680249899","series":"1.2.826.0.1.3680043.8.498.86517081266143490308445254136676850069","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67901792488494108577057563458320460313","series":"1.2.826.0.1.3680043.8.498.78902769632395248083151849629153571792","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67931702364673809781455566584550075686","series":"1.2.826.0.1.3680043.8.498.10868886992321124207774046148371083938","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67958551704692248657133951054036903879","series":"1.2.826.0.1.3680043.8.498.86056078538848220922935648699359794769","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68042422993686716523227090971667037309","series":"1.2.826.0.1.3680043.8.498.11206885780396794680955852238237005903","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68098431505945078581285587720599762771","series":"1.2.826.0.1.3680043.8.498.27709113386696003649553252828819287944","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68115220080063728900711167735074276147","series":"1.2.826.0.1.3680043.8.498.10792264221541475017018535455022492835","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68158735450883179427440578190038002606","series":"1.2.826.0.1.3680043.8.498.48883170320480161291491402852410098989","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68168197665046786209369894696801397179","series":"1.2.826.0.1.3680043.8.498.58446813329545488154455749742123242567","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68236154531625015457518732674625182872","series":"1.2.826.0.1.3680043.8.498.68004838935345888752092437571608916321","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68263925314265871578642679583159577645","series":"1.2.826.0.1.3680043.8.498.11050082121602659922250816672659054670","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68328371080087298795966933114044350359","series":"1.2.826.0.1.3680043.8.498.52193817051294539804219237296004667503","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68434116759606759586218589635360502038","series":"1.2.826.0.1.3680043.8.498.16676263877661139413657746558499059461","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68442883836038453562548037358230921748","series":"1.2.826.0.1.3680043.8.498.48524968192762532676589624998933048291","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68524920268469128133836299591793117756","series":"1.2.826.0.1.3680043.8.498.41445067782884315978988976064433053731","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68576465611266620513305434082656807028","series":"1.2.826.0.1.3680043.8.498.42693557226211202080433017002937984475","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68593524008570334427931338225473487643","series":"1.2.826.0.1.3680043.8.498.12973302737091268453301710188951510591","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68707023438918237611039069622720041636","series":"1.2.826.0.1.3680043.8.498.98801376132091558200001866161602956128","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68744926735490285461495285547312308522","series":"1.2.826.0.1.3680043.8.498.53679230767644887109198415373081978770","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68799355386870083063665227055340582612","series":"1.2.826.0.1.3680043.8.498.59619746317115151745138309598598200848","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157","series":"1.2.826.0.1.3680043.8.498.35893658552374639529539901045602783184","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68969263295435881669720950631709959471","series":"1.2.826.0.1.3680043.8.498.10013812389274190747275238680244804953","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69002590469755721401448924002989552840","series":"1.2.826.0.1.3680043.8.498.50980167784734344345092820841039750913","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69077351372476086930472412978307038819","series":"1.2.826.0.1.3680043.8.498.74767954159147731439215137785186844702","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69156800026549034619316278954442147703","series":"1.2.826.0.1.3680043.8.498.95400235959746707754731236848735349820","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69260280362382086729793018979234418276","series":"1.2.826.0.1.3680043.8.498.84353246355257058161186088492165415747","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69311065409015951283858403030798356475","series":"1.2.826.0.1.3680043.8.498.40627012780931465060014714376567314025","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69411539430142524612470544041263824359","series":"1.2.826.0.1.3680043.8.498.80495686179634380336525829322837432802","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69429384137362531712760113705595820981","series":"1.2.826.0.1.3680043.8.498.58608663803055102707154472967119881271","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69514709065182531331987458626496800737","series":"1.2.826.0.1.3680043.8.498.11666903709362045940410034924642096171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69632034454628807663252084651278084839","series":"1.2.826.0.1.3680043.8.498.10466880668297225259169983524928305243","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.69654898267139720912940475772303404772","series":"1.2.826.0.1.3680043.8.498.11722337243519359188271529022245204447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69717507929047897681405659943474469018","series":"1.2.826.0.1.3680043.8.498.11410384038779882544308736148860057808","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69755849784902271124335765927125722483","series":"1.2.826.0.1.3680043.8.498.93679428734422453756283541625385307333","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69846596572083911408018130524077120494","series":"1.2.826.0.1.3680043.8.498.10517324299134749383898739890805847981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69954533310505596128507234525019777805","series":"1.2.826.0.1.3680043.8.498.61335802637311646048334145569893406948","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70129413051088864999976563404818211137","series":"1.2.826.0.1.3680043.8.498.74432028606394795043708649349298783611","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70212141059019236877414201247791694101","series":"1.2.826.0.1.3680043.8.498.25832639803733962355582245006542717884","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70310162525433830894324495692973986063","series":"1.2.826.0.1.3680043.8.498.19541678458071343188922990940888261007","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70375834273924102402706635540898552043","series":"1.2.826.0.1.3680043.8.498.41260471891460154318348209625729939695","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70435722790459313233345196619727661231","series":"1.2.826.0.1.3680043.8.498.87762211145201674647136286958697185692","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70471859687546886677145171619638581757","series":"1.2.826.0.1.3680043.8.498.81924461817707081378595207921358755818","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70507536279875475164400849670959748314","series":"1.2.826.0.1.3680043.8.498.80178940651868606143975629151968168803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70609649448269244608860767754947143395","series":"1.2.826.0.1.3680043.8.498.11861686649330003827484619730109918867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70705483212961922828901596159131697717","series":"1.2.826.0.1.3680043.8.498.54514979785578969205710000857296898234","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70829434408808897601630953382239533220","series":"1.2.826.0.1.3680043.8.498.46522087221326180385606196870572067616","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70873330457307886117949106432465978541","series":"1.2.826.0.1.3680043.8.498.64203575357997538753648790486469543314","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70880546533848103776823598726224805522","series":"1.2.826.0.1.3680043.8.498.11750087490728659536749528128081518570","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70918221697435256352088727172904502068","series":"1.2.826.0.1.3680043.8.498.32864629002975889481070946122500601855","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315","series":"1.2.826.0.1.3680043.8.498.87525213138081154927558103790159162056","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70966951020892708303866968206211604316","series":"1.2.826.0.1.3680043.8.498.44399965226355610710072431949789051293","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71001339006150287453095055802545712130","series":"1.2.826.0.1.3680043.8.498.79335777667953970852000489159295154316","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71084400653875583977681616665093993652","series":"1.2.826.0.1.3680043.8.498.42433581236262282777156296814166353260","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71262022207903748073532915013888861999","series":"1.2.826.0.1.3680043.8.498.88938566375481329315032053880821339251","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71355457409240695394093233905149541215","series":"1.2.826.0.1.3680043.8.498.56259320743913591654286419763033002512","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71405492944866967669069984757332849184","series":"1.2.826.0.1.3680043.8.498.42959568419872895819129270619585492857","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71622107620007706778976946784787365505","series":"1.2.826.0.1.3680043.8.498.19121917954307830798561661440731446444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71671013221669628598634284971059228267","series":"1.2.826.0.1.3680043.8.498.10231619549231310954947452369676342837","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71688651674041466985305836912343513997","series":"1.2.826.0.1.3680043.8.498.61918875183507811206453903186328898829","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71707317731065645278250956442336515504","series":"1.2.826.0.1.3680043.8.498.46416143782455904406027101419217423469","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71744523263023933066263506201344989825","series":"1.2.826.0.1.3680043.8.498.22092008617067701124815413667030562623","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71766161038805688022754893134126528749","series":"1.2.826.0.1.3680043.8.498.20282039881399270076591589040212035559","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71814919885777053270239861047114794237","series":"1.2.826.0.1.3680043.8.498.51226495998815301149581759071663672048","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71836684300308056297405632410209248276","series":"1.2.826.0.1.3680043.8.498.10162282523284664796736828096063362714","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71899370657875960028369229847153071577","series":"1.2.826.0.1.3680043.8.498.60672898004145481782818295018782799694","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.72008869367139958325145888733651022033","series":"1.2.826.0.1.3680043.8.498.55765075016184938666464661026441248935","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72018602547183896629903902899903468287","series":"1.2.826.0.1.3680043.8.498.10077875530578967644311474218291364422","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72095214441332094906401331319197122529","series":"1.2.826.0.1.3680043.8.498.41418868332391801931115966568959871258","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72182847112910012489603688436912469641","series":"1.2.826.0.1.3680043.8.498.17420438223912535649246031505223769847","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72191353854496806844040272237700346774","series":"1.2.826.0.1.3680043.8.498.10445332488427565850266729803028989909","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72252601583562621620422058467969183753","series":"1.2.826.0.1.3680043.8.498.22716449171009870419151451648054968350","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72270915206075693056344475438055463113","series":"1.2.826.0.1.3680043.8.498.72460583210015210982753088915945698687","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72342425990274562150756501476989691572","series":"1.2.826.0.1.3680043.8.498.72097936822787395766390722594710943022","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72375103774239787721371778221019402834","series":"1.2.826.0.1.3680043.8.498.71882334282552086612495648388298839613","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72494404099796908482534991590660770670","series":"1.2.826.0.1.3680043.8.498.62664806376963318514421683628038716959","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72495677804641720406765749568114476096","series":"1.2.826.0.1.3680043.8.498.51011285777361124977962837205209069765","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72576069358772864331810785663326683534","series":"1.2.826.0.1.3680043.8.498.43404222004583649872189887458908079229","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72680485820462987585370556994612687540","series":"1.2.826.0.1.3680043.8.498.30918978866937144708714209056324552427","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72740499664060575780267947543695605585","series":"1.2.826.0.1.3680043.8.498.16041400130546567411185979528690125122","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72781608662300877077691853670862414309","series":"1.2.826.0.1.3680043.8.498.11384414575384375879609510799633231326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72814437733065652700292896915085936230","series":"1.2.826.0.1.3680043.8.498.11637368446188125918635663455831883045","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72880034930990967339546757487847803284","series":"1.2.826.0.1.3680043.8.498.61756497944551572164374627976025597688","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72931900540220826595003388066347326492","series":"1.2.826.0.1.3680043.8.498.46444221873048847952170938721582633928","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72943191222961132416678221885831944435","series":"1.2.826.0.1.3680043.8.498.12506834395354221152837263794562626502","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73033521237848260228708170280586043408","series":"1.2.826.0.1.3680043.8.498.21788215216464239602976645062917854698","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73070182506490199787361298167985971577","series":"1.2.826.0.1.3680043.8.498.11343531028236723298620561336997812508","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73101997284755549124715401557235981642","series":"1.2.826.0.1.3680043.8.498.53112715039168631741815363269161592601","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73194579460722181989151410198309642266","series":"1.2.826.0.1.3680043.8.498.40292698674306795245904570163356952825","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73215265387578908341609746559916587894","series":"1.2.826.0.1.3680043.8.498.12930771808366376859368797302895163549","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73241519025825862373621891294281253158","series":"1.2.826.0.1.3680043.8.498.65281647169337345829886040299075228025","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73257122462858353041066181004123253930","series":"1.2.826.0.1.3680043.8.498.43241109327792250556578170767401670398","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73314900301724532218347713512381622074","series":"1.2.826.0.1.3680043.8.498.11306049495693640711759832696249968275","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73394992092536626285935384324191560182","series":"1.2.826.0.1.3680043.8.498.58282048410120494300880183022565289017","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73477249559479870799427355988138990349","series":"1.2.826.0.1.3680043.8.498.12716864306687729082231900706655632989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73511703934549662392765328175356230615","series":"1.2.826.0.1.3680043.8.498.10363154028406841291288730295643707714","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73539750992809874106013052513162732951","series":"1.2.826.0.1.3680043.8.498.19784209857416770209647360985415896125","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73602922823389605007702521335136347605","series":"1.2.826.0.1.3680043.8.498.10318318299752202838318883910696931382","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73682608760611536443754406204042210115","series":"1.2.826.0.1.3680043.8.498.56632664393267341263458738880370575262","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73792361507751648386401969571717557389","series":"1.2.826.0.1.3680043.8.498.96852252666327697362398491124162622839","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73869775636181013203432845292462040922","series":"1.2.826.0.1.3680043.8.498.20355093255619127488408277607375871287","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73983081622010008601849687650985260980","series":"1.2.826.0.1.3680043.8.498.12006434320147316220803012996097487703","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74052473719853343027548507306186291134","series":"1.2.826.0.1.3680043.8.498.36837439447383005706042620964614416813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74127742661271965093205833471645893520","series":"1.2.826.0.1.3680043.8.498.11313635129529960718628854752267105630","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.74160604730133050784449689952695940558","series":"1.2.826.0.1.3680043.8.498.21797233540155117256806210839450339196","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74214829456846525506232367308573550186","series":"1.2.826.0.1.3680043.8.498.24105806716452889683112377310089977177","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74222284584657051737421776919599206132","series":"1.2.826.0.1.3680043.8.498.96244790624462197907089717829240493928","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74243564762473331581882860440560337725","series":"1.2.826.0.1.3680043.8.498.48757824188273604451935136183495175643","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74274855596893595390880617896069524465","series":"1.2.826.0.1.3680043.8.498.42462092604496699490618859675905229930","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74362364640190722036251935797260325860","series":"1.2.826.0.1.3680043.8.498.13163532855337819780947621690190070530","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74394784271003789564066563841806954326","series":"1.2.826.0.1.3680043.8.498.10867066386483317588513123388539663337","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74471396469132211069109289423495141062","series":"1.2.826.0.1.3680043.8.498.23626126480455004693206161046754127199","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74622031396885197580453304563088863451","series":"1.2.826.0.1.3680043.8.498.13232554917321753647587536940761980158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74714018385278253151175853998945654373","series":"1.2.826.0.1.3680043.8.498.11417191687444914857475929323025151443","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74749071555443582954158991248361308204","series":"1.2.826.0.1.3680043.8.498.61798156919750472859155561670853884332","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74872419726042767543822107291861307866","series":"1.2.826.0.1.3680043.8.498.11090948462695212408511470312456560993","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.74939269621045823332837094516056211606","series":"1.2.826.0.1.3680043.8.498.11403474485638666401153838877664038839","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75067227246352035607675978826177487284","series":"1.2.826.0.1.3680043.8.498.99659830072007764855857117866480504526","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75168480371573015758963049782687952716","series":"1.2.826.0.1.3680043.8.498.24129634118952240141537822867006467036","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75268429892526831752484062674699274309","series":"1.2.826.0.1.3680043.8.498.11723421861943418223833433148402765779","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75282577107802752393765861202996650638","series":"1.2.826.0.1.3680043.8.498.67184969331663719635934231775272928214","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75384600248630012126841890313149674691","series":"1.2.826.0.1.3680043.8.498.12553942338912997929719814954568539303","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75420186662442688955616630735303321846","series":"1.2.826.0.1.3680043.8.498.78198485875867543155368118449671719299","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75518253718111135068488417046278011303","series":"1.2.826.0.1.3680043.8.498.55179554405458535075390929522574563389","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75547515925579172661099437338637082554","series":"1.2.826.0.1.3680043.8.498.39318294163634401902124472552686296217","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75605038901688937188489854019478082759","series":"1.2.826.0.1.3680043.8.498.95148968774435651663271018788600080505","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75639423607035582240174986752280730347","series":"1.2.826.0.1.3680043.8.498.57022160575648381582689792095286920060","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75688484058613633923943882730405145535","series":"1.2.826.0.1.3680043.8.498.10855836314769251601288863065686541431","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75746520601616488682662195284941583868","series":"1.2.826.0.1.3680043.8.498.97636374720398740682216217092420256977","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75760698509105388883811155432691667447","series":"1.2.826.0.1.3680043.8.498.11423935702908355889923640046426012172","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75893915698353338311947268382839152877","series":"1.2.826.0.1.3680043.8.498.79179279376473426102644676042564684284","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75926011938952404257302292495731660728","series":"1.2.826.0.1.3680043.8.498.50682273162955015895248984346730155656","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76012896432318288623173593280895772876","series":"1.2.826.0.1.3680043.8.498.60432544561772876420423934882325822002","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76093766204928582819565636658977147523","series":"1.2.826.0.1.3680043.8.498.12505977794764555526616802248467026299","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76122127303562588545128053661609574837","series":"1.2.826.0.1.3680043.8.498.10605593775131373057646692982185367536","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76167153566894354739332364376483514559","series":"1.2.826.0.1.3680043.8.498.40762391324766802223300236939533931215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76242245340157474968574926079920787199","series":"1.2.826.0.1.3680043.8.498.55964906318432800163196960808684767549","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76350662767360412185091180512006419379","series":"1.2.826.0.1.3680043.8.498.41862699238028654683897101765885790126","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76514504104121634569792846980451896030","series":"1.2.826.0.1.3680043.8.498.75918722840183673641552980818100790777","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76543241221133987658552440969465372120","series":"1.2.826.0.1.3680043.8.498.11205802914533485705694502802275967349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76651080491383110752977286124044311636","series":"1.2.826.0.1.3680043.8.498.83784700912299442521264658857968151914","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76680092665218514711300086707222109323","series":"1.2.826.0.1.3680043.8.498.68825239703133336208215056873500277222","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76772148369531098768751348676329945789","series":"1.2.826.0.1.3680043.8.498.48246043814334831203390969916822824887","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76903562636417038907113894538863784737","series":"1.2.826.0.1.3680043.8.498.78676736508029890633382953085133674889","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76914908131203373740345988718674850939","series":"1.2.826.0.1.3680043.8.498.73919372051804568358688467137260265719","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76993922254007307143291254174266755701","series":"1.2.826.0.1.3680043.8.498.11929681063596707829827616859736521793","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77058587552329832558354544874120592826","series":"1.2.826.0.1.3680043.8.498.70998882310081018441641348110773193071","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77061877382110597010150770981465581536","series":"1.2.826.0.1.3680043.8.498.10729200148670497651650678537027548252","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77155817997584903787597001328868124855","series":"1.2.826.0.1.3680043.8.498.41967779045880377858903429545241222744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77215476384973989055525592556183934827","series":"1.2.826.0.1.3680043.8.498.12915668453932718458909853139126235395","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77261014467268552758826551926128217025","series":"1.2.826.0.1.3680043.8.498.17780651052016264389751668739938073497","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77283213969268676660453937180363826917","series":"1.2.826.0.1.3680043.8.498.23520294294517657672346506982835983465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77299727102400166279263320763707637485","series":"1.2.826.0.1.3680043.8.498.46756253068899498189771736872377321882","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77319661699409997424800377883567027839","series":"1.2.826.0.1.3680043.8.498.11857305198558868450449700077444812865","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77370119801633353780677378448761665685","series":"1.2.826.0.1.3680043.8.498.12387137925947687829668763899872254897","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77416382267531567401698555072875486512","series":"1.2.826.0.1.3680043.8.498.19659829560458632892692002574763104285","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77433065090757862487482575794398458402","series":"1.2.826.0.1.3680043.8.498.24739093036519711258999230622190511814","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77498121906981512168120474112422492971","series":"1.2.826.0.1.3680043.8.498.51037444671403759945930253354451786131","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77524507552350448792970399203131998592","series":"1.2.826.0.1.3680043.8.498.91598369233820269148211831189489557607","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77546769004367585830455474984769277713","series":"1.2.826.0.1.3680043.8.498.87769556901818619114455834291680210507","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77593859518024366170735307975173961777","series":"1.2.826.0.1.3680043.8.498.97952593527375080021897007436429850933","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77626578713099735143745396709359607445","series":"1.2.826.0.1.3680043.8.498.25389456937397263689131413956785890271","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77735690782343511872544703464260892488","series":"1.2.826.0.1.3680043.8.498.13094176949675319393365694971047837150","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77753753872601674668756777373761660873","series":"1.2.826.0.1.3680043.8.498.13046007868947017774498833093314673113","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77800740811302686332282410929816056283","series":"1.2.826.0.1.3680043.8.498.10906271477167147785815634727765675076","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77823946901144772346251781172043480398","series":"1.2.826.0.1.3680043.8.498.10813122722764339267399893719529029985","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77893704208186903913532950495036889026","series":"1.2.826.0.1.3680043.8.498.10872005957091797807039260879648155982","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77933128806366660573360274958258917742","series":"1.2.826.0.1.3680043.8.498.41408856415906081137090766232761248215","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78012314847102407375138809615977711653","series":"1.2.826.0.1.3680043.8.498.12529078943207678403394743045675613602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78083057157844664959843364584162587026","series":"1.2.826.0.1.3680043.8.498.12036841367029866713301380221939682736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78188863861812651079751164964756388411","series":"1.2.826.0.1.3680043.8.498.38521000807172142562899279048895267615","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78236036694398253902871510456985685293","series":"1.2.826.0.1.3680043.8.498.92438382555971945162881922271399336998","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78286395605671977872377352305890466464","series":"1.2.826.0.1.3680043.8.498.11715113704665255717601053385099619086","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78420763928408680447797586163977368190","series":"1.2.826.0.1.3680043.8.498.78635256828035465473707066893886407223","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78475145549489837635803968565380247517","series":"1.2.826.0.1.3680043.8.498.91848469928166529380049430919830755902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78521396404736679127326025240279747196","series":"1.2.826.0.1.3680043.8.498.30517928590254193801105040325905902794","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78646473961447523033955716877524794707","series":"1.2.826.0.1.3680043.8.498.54782218780244394801357491830436330240","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78674598286586525734293563394409166190","series":"1.2.826.0.1.3680043.8.498.28474147504091932490739246033112867683","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78757011872917563830995996191850541453","series":"1.2.826.0.1.3680043.8.498.47130775219503863126532343882629782365","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78808320605816811006382053060951793555","series":"1.2.826.0.1.3680043.8.498.62971932963147480815420404967923379682","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78853931928960361455025737444267199231","series":"1.2.826.0.1.3680043.8.498.26711982445287508097725662891362625857","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79021499290691125513421477204330532755","series":"1.2.826.0.1.3680043.8.498.17501240733105756252678544296563592659","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79080521163097116147379189510104170611","series":"1.2.826.0.1.3680043.8.498.10570171691692312183932864737027217279","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79117047831806895544174734229992359986","series":"1.2.826.0.1.3680043.8.498.63471163738554771627089019310304254812","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79165531588966776246377098630025444815","series":"1.2.826.0.1.3680043.8.498.76417901644611690107485690371100296898","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79245771997885598399203314109623928718","series":"1.2.826.0.1.3680043.8.498.18931878606114799610464498975485894680","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79258497507223824227830886693749460348","series":"1.2.826.0.1.3680043.8.498.10975434384789387910330084007281680784","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.79274948549030940175536603063980168192","series":"1.2.826.0.1.3680043.8.498.28745069675729699610844116161700090085","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79292862882980756453890778996691993351","series":"1.2.826.0.1.3680043.8.498.53932479537683080674857728300167263629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79505118958475739396320647697583979079","series":"1.2.826.0.1.3680043.8.498.70009451822299279121609993749720777540","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79560170727391465844958390474997798803","series":"1.2.826.0.1.3680043.8.498.12497737756100671753810311168218374020","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79582457162604674356422857305405255763","series":"1.2.826.0.1.3680043.8.498.12899714781546661734875961552788394015","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79648517065511897282938523065537768426","series":"1.2.826.0.1.3680043.8.498.67281170210301180403136101151698124727","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79725520325008084984331823107016254775","series":"1.2.826.0.1.3680043.8.498.72533851986917493022341745130069530036","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79798779812052605703861699094268016306","series":"1.2.826.0.1.3680043.8.498.21945820627480795359359992533758361813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79991838737793096019496013540529642900","series":"1.2.826.0.1.3680043.8.498.16364010211329477998138791562420994579","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80100785345864355815698147663964005920","series":"1.2.826.0.1.3680043.8.498.67353218273364926842653450342673585275","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80180578697901833764761405542790185180","series":"1.2.826.0.1.3680043.8.498.37768617788105825246045655619609359615","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80207056225476761432712249807198836187","series":"1.2.826.0.1.3680043.8.498.42545927862387406950511580573656170451","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80284846390295834578273590480193235798","series":"1.2.826.0.1.3680043.8.498.12631077019665148227471988356484338206","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80292580287849976694993628109721422478","series":"1.2.826.0.1.3680043.8.498.30730551313623314158110057665146594650","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80319546205646617603546887163539306146","series":"1.2.826.0.1.3680043.8.498.53477424301291864424919993540851998946","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80391378873630377472192017862302619947","series":"1.2.826.0.1.3680043.8.498.34777444489118102822010542789446443026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80492324225032367918842505447056622561","series":"1.2.826.0.1.3680043.8.498.48254345861324259938349700495382095217","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80556515221794602430701573588498144286","series":"1.2.826.0.1.3680043.8.498.75455580669087394534986825955949580812","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80612964209397056653334784776523875647","series":"1.2.826.0.1.3680043.8.498.10260293555353258969411669239890287296","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80629565747112614845746475820152813307","series":"1.2.826.0.1.3680043.8.498.10477354259025413804386522880281985968","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80651977424179089141346174053751183506","series":"1.2.826.0.1.3680043.8.498.10507374692519826743927906611913945889","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80671597537348112977780954910198682272","series":"1.2.826.0.1.3680043.8.498.11592973573082501000010288573305859218","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80735285867775404153100236162962472952","series":"1.2.826.0.1.3680043.8.498.21845561934260309725404434539954197687","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80758336123908522542787716219939751704","series":"1.2.826.0.1.3680043.8.498.53153424392896705025810191881242696043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80876235930771882271600826613593202364","series":"1.2.826.0.1.3680043.8.498.11776191548256235062862814896924424982","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80886525412417948114478368374120704834","series":"1.2.826.0.1.3680043.8.498.10954084090299627221819379393795336577","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81018911907787774214591206085583545057","series":"1.2.826.0.1.3680043.8.498.87171864277478031414954465104434941922","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81031057654953109295788159826841315360","series":"1.2.826.0.1.3680043.8.498.73276210989115985431949558876171649881","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81109405367395078564887780335570099673","series":"1.2.826.0.1.3680043.8.498.89047293126184995974013395049279569431","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81248820934296461385324385812581970340","series":"1.2.826.0.1.3680043.8.498.35494623811913472807119379726515197978","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81327951282145199298572980264557740321","series":"1.2.826.0.1.3680043.8.498.11784120971296859878745621717472322699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81351574724227319632426427855692606694","series":"1.2.826.0.1.3680043.8.498.79576925041673806615746943298046126486","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81462603621780919719466264356289554669","series":"1.2.826.0.1.3680043.8.498.32621951442871996108012552329548692895","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81475058647553697318837239968139077272","series":"1.2.826.0.1.3680043.8.498.51002151683701013422543184135556692511","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81540134120507822899514611863725372563","series":"1.2.826.0.1.3680043.8.498.89683028691505142211515774424278934364","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81629704995225424568187579791704967233","series":"1.2.826.0.1.3680043.8.498.10899361608375770671667431056077608329","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81651132686162360431299931833372141617","series":"1.2.826.0.1.3680043.8.498.11402234998850102751676684059555869671","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81692996137620934043276253858381812472","series":"1.2.826.0.1.3680043.8.498.10901891109906104104279616676584971617","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81740348974208459068937862799489495782","series":"1.2.826.0.1.3680043.8.498.87464141169488619816660840568318070755","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81793454171031578492222004629265937656","series":"1.2.826.0.1.3680043.8.498.11006124140980445438475978122882952305","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81819209302731920404237374320026807282","series":"1.2.826.0.1.3680043.8.498.10589626585149266576664171412199134594","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81832287065207835637503039077685315236","series":"1.2.826.0.1.3680043.8.498.76208043330451046486096394411061901498","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81957447654112577010261210572986132211","series":"1.2.826.0.1.3680043.8.498.52275121567585998348178137354212838498","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81974732979506931694079719601760393277","series":"1.2.826.0.1.3680043.8.498.87157468243754563779456206710783263897","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81984200870681776955493738987078746894","series":"1.2.826.0.1.3680043.8.498.11720870217690580774976340577760783783","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82074883987134049971242727164506046476","series":"1.2.826.0.1.3680043.8.498.11584097603445185954412683981861589196","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82144602300666174450721045102799681426","series":"1.2.826.0.1.3680043.8.498.28594609788416422805871779550012978151","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254","series":"1.2.826.0.1.3680043.8.498.10347272512623805065972057458376849254","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82355456395534835058857499033494900716","series":"1.2.826.0.1.3680043.8.498.48805337500180351478503821845200297349","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82440268452598207215458064865523988222","series":"1.2.826.0.1.3680043.8.498.52255852827183589749116947120527933202","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82498616123349854719801627024073383874","series":"1.2.826.0.1.3680043.8.498.13147190584295199097758938898988524157","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82548952153746644951736669119607083230","series":"1.2.826.0.1.3680043.8.498.22340028195144259030245705271583922026","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82582558624896662901163827145277490232","series":"1.2.826.0.1.3680043.8.498.88701579963800154657591307763685773902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82655005222804459605894403147145992714","series":"1.2.826.0.1.3680043.8.498.83142196748118741695328772598724606278","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82757616216788919181323176147396770758","series":"1.2.826.0.1.3680043.8.498.69507732957957702099093384398376169535","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82902769135373284012762237744774967400","series":"1.2.826.0.1.3680043.8.498.48810001379184855241925789194397958200","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82967086493280898295669930903240677729","series":"1.2.826.0.1.3680043.8.498.40768004389812241063040009905672738092","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83030371526667747898709613600825477966","series":"1.2.826.0.1.3680043.8.498.63845435130143039684440600707356824337","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83055401642066749019091368358783700783","series":"1.2.826.0.1.3680043.8.498.43870036412248171668755630552333643691","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83121378128521985713113311344685112825","series":"1.2.826.0.1.3680043.8.498.27488291108358638578310143225831236194","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83220896521302064187926748035734200732","series":"1.2.826.0.1.3680043.8.498.80228517401239896656692709089929562360","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83253778356746892201320443660102495790","series":"1.2.826.0.1.3680043.8.498.31248943513300060534858798390654365498","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83376388091077619281568190257019735638","series":"1.2.826.0.1.3680043.8.498.91027913832433079712689892470836838284","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.83465081013175989426506371406463660936","series":"1.2.826.0.1.3680043.8.498.10052874809669177005295371530225804164","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83509478059288902959149676250911654986","series":"1.2.826.0.1.3680043.8.498.59651919514803119235562910774512519035","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83691358175810149975084604975065122220","series":"1.2.826.0.1.3680043.8.498.13179275355797771005866565254781979603","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83747418172021065605877274971789603771","series":"1.2.826.0.1.3680043.8.498.16392686695686624447568905179918267839","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84007436672323587547274248257117331532","series":"1.2.826.0.1.3680043.8.498.25602888319738711375062638442857340747","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84059077072076289248966490696872414783","series":"1.2.826.0.1.3680043.8.498.40595517851757064173523476660622111660","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84062980592611347665739105270731842220","series":"1.2.826.0.1.3680043.8.498.11830320587424142624576809879113227498","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84123495463663584148839831457665741488","series":"1.2.826.0.1.3680043.8.498.72487736396522727107976923322952546583","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84238814030979264404768702945290704724","series":"1.2.826.0.1.3680043.8.498.38838269267205260452475158216100742891","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84419297518873367123595080276985526259","series":"1.2.826.0.1.3680043.8.498.15807752588513454483392212953615508461","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84486787871353595019134028899317143475","series":"1.2.826.0.1.3680043.8.498.25328424457663930854895628721906442864","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84630493975910180537831353884518764364","series":"1.2.826.0.1.3680043.8.498.27316377527596457764598333824763966220","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84661341697154104963062177064495044195","series":"1.2.826.0.1.3680043.8.498.10384735195657086082856589318374515404","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84748479740284893375192048321707057482","series":"1.2.826.0.1.3680043.8.498.80825774132333777977638840890997555617","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84855341377363129926967387137216814518","series":"1.2.826.0.1.3680043.8.498.10572081800211102345216867714469970828","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84982310626395569925821771375535259394","series":"1.2.826.0.1.3680043.8.498.39448946227808888525214771213368212224","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85031127696558971105465423134104223548","series":"1.2.826.0.1.3680043.8.498.12470124389453565447975340347610488218","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85095389649021071892478435298763928925","series":"1.2.826.0.1.3680043.8.498.53711151650956827684990404765233378905","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85194002080793317087210934629814349622","series":"1.2.826.0.1.3680043.8.498.27801446478378665317642273298705370161","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85294435004325211584698794441083866921","series":"1.2.826.0.1.3680043.8.498.66851781898540959962369172532854214180","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85341019006931436802294963713199308935","series":"1.2.826.0.1.3680043.8.498.88519094153784700000130162176887854026","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85391052076760686301971262244561025345","series":"1.2.826.0.1.3680043.8.498.10107973575530622745700838381824812295","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85431902914673414035492907270636012892","series":"1.2.826.0.1.3680043.8.498.38857778420053814048756013285526303163","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85464150757448712711832350617010924327","series":"1.2.826.0.1.3680043.8.498.11617338159312411055222791565081347123","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85545102350084131396266710667808696561","series":"1.2.826.0.1.3680043.8.498.85740092627195027821639305552578694265","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85602291032341007550376696695283794665","series":"1.2.826.0.1.3680043.8.498.44415677127836609062005479566221367805","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85646781189339988389190891868001374533","series":"1.2.826.0.1.3680043.8.498.11570035849725280318667755964077048776","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85687351346612079711706269264454944647","series":"1.2.826.0.1.3680043.8.498.15258324350768642266778227050200885137","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85719235937472088594405415004352845221","series":"1.2.826.0.1.3680043.8.498.12783314360486554744093628127442827918","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85748882309364598407865143890978767674","series":"1.2.826.0.1.3680043.8.498.17070177008431954460526243983780566690","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85849908712977285006159496192673293501","series":"1.2.826.0.1.3680043.8.498.55324302482354540073541342307689229614","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85882078780717905717989373451578844863","series":"1.2.826.0.1.3680043.8.498.87301291756113725481742847517742679066","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85890890545815190189345870912381925290","series":"1.2.826.0.1.3680043.8.498.12608700357682717425056635907494944495","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85970222049966384156120176704985355996","series":"1.2.826.0.1.3680043.8.498.12431348927991501899906324157370645641","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85992037603186775437749979929724514304","series":"1.2.826.0.1.3680043.8.498.76576866025573854688935095601202770125","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86020966816754359840982061252012700091","series":"1.2.826.0.1.3680043.8.498.65498082934391112963382551120066712456","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86067849233330188988240710585112183308","series":"1.2.826.0.1.3680043.8.498.76867570618654542813292196932966328301","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86142930956005716169649168306759482915","series":"1.2.826.0.1.3680043.8.498.33223013912381033045464219151822443351","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86251314013737658351649446042801810374","series":"1.2.826.0.1.3680043.8.498.78854472677308059457894709662612318547","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86273853115722509980053190345047859057","series":"1.2.826.0.1.3680043.8.498.81178167153820994479027949421814125457","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86315835011470465506843590014408831373","series":"1.2.826.0.1.3680043.8.498.98326838114575998852502488280912166492","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86341139705884690862568425859303870601","series":"1.2.826.0.1.3680043.8.498.28493273713629395972168778802603726584","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86405969333062807387894608315897106350","series":"1.2.826.0.1.3680043.8.498.17527290973280363045583523023345587173","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86422343639522250286002195622435564347","series":"1.2.826.0.1.3680043.8.498.10603126295079189062086922817966806775","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86516555757334047240474191101770683543","series":"1.2.826.0.1.3680043.8.498.12094442659626629927770217542051561843","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86566584776908143877603164704838586097","series":"1.2.826.0.1.3680043.8.498.10982205291197328653454025722706676031","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.86651345701812229172985086531423369421","series":"1.2.826.0.1.3680043.8.498.71570488916625193299834294280071289256","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86733565200815744326667230375642936291","series":"1.2.826.0.1.3680043.8.498.37859526303004743735941014555552595345","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86881633316011036335786865282908198831","series":"1.2.826.0.1.3680043.8.498.85447165276989551888061059601734140524","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87009029107490862793898457518597022930","series":"1.2.826.0.1.3680043.8.498.10788203609786257182687482205638627942","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87046380372347615219261898797043742937","series":"1.2.826.0.1.3680043.8.498.18859005427102093212341906031220942750","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87102698749288269439517333660896096520","series":"1.2.826.0.1.3680043.8.498.90170986362259477849998645766532381199","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87198191797585172912979594638355685739","series":"1.2.826.0.1.3680043.8.498.13092087597366305016387951094011982229","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87222653734797522858506220026427900177","series":"1.2.826.0.1.3680043.8.498.39064660084997867058401976923997106816","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87241650441242408799533998101943024283","series":"1.2.826.0.1.3680043.8.498.50550535683621716865662203550772608204","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87267658933515453963224923798100651549","series":"1.2.826.0.1.3680043.8.498.45643299086357747059087755022843725448","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87404730020015145302086817782843205171","series":"1.2.826.0.1.3680043.8.498.33864338492147305320115142624953347673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87465202523430400780596885230492167155","series":"1.2.826.0.1.3680043.8.498.35769109513946275295714207632894275229","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87516159856486535118712846576943170757","series":"1.2.826.0.1.3680043.8.498.56728681281470037950500691913928954372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87563449504927531224663601218159863518","series":"1.2.826.0.1.3680043.8.498.12111970650071054377470785842939560886","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87650297321648368205324227429768256110","series":"1.2.826.0.1.3680043.8.498.41800979442160513913985927157065260502","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87742737485142445536023876862256284839","series":"1.2.826.0.1.3680043.8.498.28145185219813399073125815391834058871","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87769640167791104899516835133527936885","series":"1.2.826.0.1.3680043.8.498.70197761834037180372914985703423551968","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87799106106199445436621174506197684035","series":"1.2.826.0.1.3680043.8.498.86196392058942692678111396182845916219","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87900330305833722536121052961334699743","series":"1.2.826.0.1.3680043.8.498.20507047716316129835741958568816267290","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88022456207467042544403752144674777523","series":"1.2.826.0.1.3680043.8.498.10339128212959657604888704813541605732","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88074618626471004030644935418777151905","series":"1.2.826.0.1.3680043.8.498.70981420114174995009543628107370205150","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88114918358665488269761388157480523350","series":"1.2.826.0.1.3680043.8.498.11972397382034285679336714305062089333","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88274753211076645976496220746740420894","series":"1.2.826.0.1.3680043.8.498.30436816151756216240595792409699059985","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88439732365857773633852840383740478214","series":"1.2.826.0.1.3680043.8.498.26678829351257395861043014601284097540","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88480827968231439729532202780922145132","series":"1.2.826.0.1.3680043.8.498.10853408131969035814236154165376076500","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88523737187772411911621088333701306485","series":"1.2.826.0.1.3680043.8.498.97331787897712277532523664883674920655","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88571985843649843469694556575325271560","series":"1.2.826.0.1.3680043.8.498.10182079484010866336352024084895732044","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88609906210780568637209291182572648571","series":"1.2.826.0.1.3680043.8.498.10064481329092717090324940006461756315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88655485247489827200671636011965281184","series":"1.2.826.0.1.3680043.8.498.10269137325047049810520648626739055349","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88682169866098509464628948503453852928","series":"1.2.826.0.1.3680043.8.498.54081078940214468239870188606365216723","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88726804476694411363564819498296119763","series":"1.2.826.0.1.3680043.8.498.34205254060997681777161643267772728437","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88753415430913291809543943690942744997","series":"1.2.826.0.1.3680043.8.498.12268546440883267953527728350857826273","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88845193477394064535233691626377077328","series":"1.2.826.0.1.3680043.8.498.31564813049933408756935844804668161042","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.88874647535130028956725641820977918615","series":"1.2.826.0.1.3680043.8.498.11898626759696270178374822463044437874","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88899519473612920868099376939780201067","series":"1.2.826.0.1.3680043.8.498.95474611783956714637201589490936171709","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89026102874017923932934673207699042800","series":"1.2.826.0.1.3680043.8.498.68665176865626719760064176045746853211","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89048907638162719840222508855779172366","series":"1.2.826.0.1.3680043.8.498.10443980500881724793181652454399951341","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89110271437785174288044757208973128908","series":"1.2.826.0.1.3680043.8.498.16265225217392865998588897948688158975","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89204062904097086723738217311755833877","series":"1.2.826.0.1.3680043.8.498.84066237019807855435328804902547902462","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89247482662611898098489676481780144186","series":"1.2.826.0.1.3680043.8.498.94540793167476802435291726290801402938","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89326347774248795083360112232993521530","series":"1.2.826.0.1.3680043.8.498.13315611681390413032078096257603270829","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89346975863478164334871204856160071946","series":"1.2.826.0.1.3680043.8.498.86082432621071966969710599477100917182","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89357823844365374776758395347764601030","series":"1.2.826.0.1.3680043.8.498.97341504612171875413174894909749801756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89445342796957655182687412965056270903","series":"1.2.826.0.1.3680043.8.498.10494120344739378101520124392737515354","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89569762088162031469299314053759758927","series":"1.2.826.0.1.3680043.8.498.49036660895744379288188509432278545687","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89590880980421560369495600204866789646","series":"1.2.826.0.1.3680043.8.498.84752994767250069984516472434200961559","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89729486637365590877469535082675069395","series":"1.2.826.0.1.3680043.8.498.11722935498100841256374399254450479991","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89842390998700046088751402253903762827","series":"1.2.826.0.1.3680043.8.498.11499561251768576397389055599694747508","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89935384074170571164295050203650829722","series":"1.2.826.0.1.3680043.8.498.72211137336357482045806496189390600972","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89953731420879076053772747816096629231","series":"1.2.826.0.1.3680043.8.498.11888452102647518118909138118854946706","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90003472003278237920648198243464374280","series":"1.2.826.0.1.3680043.8.498.37939901699599309335587174426676564917","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90057130466053377217652879986591534745","series":"1.2.826.0.1.3680043.8.498.29922234556909163941651148995030706296","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90170179734833731758923964833173536172","series":"1.2.826.0.1.3680043.8.498.50798747652881186498535545605055003627","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90206716031776606956011363950625953310","series":"1.2.826.0.1.3680043.8.498.29624330075065774997234433037024142721","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90227414485305096825508911050013279704","series":"1.2.826.0.1.3680043.8.498.25683286594782661132291696701214661713","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90268709354762000249148795667759483876","series":"1.2.826.0.1.3680043.8.498.14044834284265760766651279694861332544","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767","series":"1.2.826.0.1.3680043.8.498.69393995607370120323004157329094509077","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90313529150165069133283531436628039942","series":"1.2.826.0.1.3680043.8.498.83852462107005519193328893127839549402","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90340114291852810896219650388952790164","series":"1.2.826.0.1.3680043.8.498.77281832621160517188133985399849159385","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90446842983860370183814559926724769347","series":"1.2.826.0.1.3680043.8.498.40340468199720916391759441225777848435","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90490891782778285565462083536468192410","series":"1.2.826.0.1.3680043.8.498.50354376114204027466766236761210496885","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90514150380296346197442897381548735293","series":"1.2.826.0.1.3680043.8.498.16750335922939986337656041827627426227","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90573769367031841574114648773801425781","series":"1.2.826.0.1.3680043.8.498.36198607349820088780335161208302553430","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90668646188784648688693620159738574680","series":"1.2.826.0.1.3680043.8.498.11268091180452583667838268657410870513","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90758414832541790868268922794274257839","series":"1.2.826.0.1.3680043.8.498.11788611099019230623318093043606097224","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.90771070773751767226565553195436971014","series":"1.2.826.0.1.3680043.8.498.75770182493207907384924113279958877776","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90976227335921831531105845602655777296","series":"1.2.826.0.1.3680043.8.498.86096178782602012989153604293348930050","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91064647744940506223536204418057554573","series":"1.2.826.0.1.3680043.8.498.11313820274172413803691377391333426404","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91122873808284731048016330734204992698","series":"1.2.826.0.1.3680043.8.498.20971337922516946275011004781641191141","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91139126799820974665417903552066103454","series":"1.2.826.0.1.3680043.8.498.49857786611643026586681973728842590554","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91206973544724011503639050238118945528","series":"1.2.826.0.1.3680043.8.498.76868818928853329521204577595326435535","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91230059913252604666971822137553690811","series":"1.2.826.0.1.3680043.8.498.12713396012446548076336651169114724447","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91323765268136520804399328350809179765","series":"1.2.826.0.1.3680043.8.498.41514225220149391394897154395127935565","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91346221361264140052760282748336176394","series":"1.2.826.0.1.3680043.8.498.84069077419405185904543802217599084367","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91385694666101904780012335954322545042","series":"1.2.826.0.1.3680043.8.498.42798392461940196624840098942793840202","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91431012289716035498822977054491790785","series":"1.2.826.0.1.3680043.8.498.15519619291868759169903952028965597876","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91497176426480836856097456438014037999","series":"1.2.826.0.1.3680043.8.498.12664787797674809843666198835185372352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91606395836371290680039618164703651751","series":"1.2.826.0.1.3680043.8.498.46073135324872132536004366772116050252","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91686669332744972709173289374472094988","series":"1.2.826.0.1.3680043.8.498.76693001956932265115688747105861682253","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91744384920314309659802891351055961910","series":"1.2.826.0.1.3680043.8.498.11297980855569477596819883811639737340","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91860013313252950044036077977399321593","series":"1.2.826.0.1.3680043.8.498.53485623883472949347590309920837010868","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91910585599088493175245713843727208369","series":"1.2.826.0.1.3680043.8.498.95210080384496597269956849351476896702","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91940000162714863613139612302783091395","series":"1.2.826.0.1.3680043.8.498.53011417147975357848070408165456235193","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92030581934838097812004394123949263069","series":"1.2.826.0.1.3680043.8.498.60102043452851417775672024361157621736","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92071199603599654827155753204585082303","series":"1.2.826.0.1.3680043.8.498.11544221075752804081480882246956908417","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92222956609345084635188178384706987675","series":"1.2.826.0.1.3680043.8.498.75246217875053726442354757876978690403","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92285140961160393284537044926797259671","series":"1.2.826.0.1.3680043.8.498.11076642878926679164357669484812526965","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92386519544130078776672685639580497728","series":"1.2.826.0.1.3680043.8.498.52106917665287870384276379174851384859","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92410537691481397330646029810090178745","series":"1.2.826.0.1.3680043.8.498.90266939495101149845831162899809628136","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92455257597708674644536154732642099964","series":"1.2.826.0.1.3680043.8.498.64114372098825018780082581775947278581","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92597065276819676534178545455668171085","series":"1.2.826.0.1.3680043.8.498.84530932792481213677732918796127123105","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92626490033376759309712417622692944095","series":"1.2.826.0.1.3680043.8.498.87984434176812337369263642452986052772","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92655291659311409459093330375014445330","series":"1.2.826.0.1.3680043.8.498.41414876399337254721147697723902696918","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92722393862174380736027353110424417588","series":"1.2.826.0.1.3680043.8.498.83310419690466851740167603956050571509","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92839397144213404004277739877191283401","series":"1.2.826.0.1.3680043.8.498.11640514548652905370048889699927922381","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92925901047699000727725798824992891055","series":"1.2.826.0.1.3680043.8.498.93667721269082831672538018578722834175","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93054327536380380487760790221613186305","series":"1.2.826.0.1.3680043.8.498.55118943032892830913031509274159203194","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93104685852024334015651558514195715956","series":"1.2.826.0.1.3680043.8.498.44034215502343583859947266687745368740","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93177444420568709574758402836371200945","series":"1.2.826.0.1.3680043.8.498.22783580283484824925167187725047582409","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93222021909191248270818335069738080963","series":"1.2.826.0.1.3680043.8.498.17193798758899535526105706974161934621","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93301337627536436086612556500687157032","series":"1.2.826.0.1.3680043.8.498.16524179539088535345247940605283851143","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93344237572402384894445814841725878706","series":"1.2.826.0.1.3680043.8.498.27033225577979713520233371899744051398","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93478919245462741704531324128061533679","series":"1.2.826.0.1.3680043.8.498.54411427588047046094013669672473694519","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93623421864452751381778712192879121271","series":"1.2.826.0.1.3680043.8.498.11536814152260457974335726377307891067","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93718451128867963123946970296603639068","series":"1.2.826.0.1.3680043.8.498.41547434324415086745831671638221530484","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93728117237268000947508816107096388086","series":"1.2.826.0.1.3680043.8.498.15644146908239619120472597058200885924","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93774667493652492119129601342627338680","series":"1.2.826.0.1.3680043.8.498.93768937867332880545862590444657449090","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93921993164883084178408030385007465064","series":"1.2.826.0.1.3680043.8.498.46053977402302229106254332733879544969","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93985093305024344873337249433396141838","series":"1.2.826.0.1.3680043.8.498.80545508275343304166170345566827463199","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94030816415316308210133175314873817171","series":"1.2.826.0.1.3680043.8.498.17870456507376342945256315987774426668","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94244885518758327449236878653096161475","series":"1.2.826.0.1.3680043.8.498.12394031477575871776610934102880067676","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94255124739558547668304022649099251744","series":"1.2.826.0.1.3680043.8.498.59444261913652793996207921616839286716","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94328216642444410827846344286227627425","series":"1.2.826.0.1.3680043.8.498.91488676505110819897072444451119609496","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94381282809153270156987123429739805206","series":"1.2.826.0.1.3680043.8.498.10456943385686562184190189822667463367","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94447683257072169359896104855962933413","series":"1.2.826.0.1.3680043.8.498.83705893941585631653092350003651908052","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94553664348118184200007654265863262684","series":"1.2.826.0.1.3680043.8.498.30126047399896475948336383754035346248","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94601583997736688762302974573903982304","series":"1.2.826.0.1.3680043.8.498.74349508945058157268315334314680611457","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94625454484290996254750481989036812516","series":"1.2.826.0.1.3680043.8.498.11721550529197630725102074533687154813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94677297888135961827159679484207792618","series":"1.2.826.0.1.3680043.8.498.40517785234021193764786206681499758648","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94729634553835820944505159643192054175","series":"1.2.826.0.1.3680043.8.498.20477532864815202608731467960203855315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94761755148413531477206243804832613511","series":"1.2.826.0.1.3680043.8.498.12005547350053600702292464413650710788","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94822220672465914426995315295921716764","series":"1.2.826.0.1.3680043.8.498.74986989502658825253010692674185293362","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94944218313788305917509271079443871442","series":"1.2.826.0.1.3680043.8.498.57964571844439579222498805731244131136","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94954489705497211843190734684710878341","series":"1.2.826.0.1.3680043.8.498.10679603613099611876524439419734045533","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95030266788619257057351879256576338162","series":"1.2.826.0.1.3680043.8.498.80611727050911496363292930543923882489","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95080449640993916851075185393444497308","series":"1.2.826.0.1.3680043.8.498.61624370135980400072923118868070356709","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95142886737239438773421088967802136318","series":"1.2.826.0.1.3680043.8.498.57226428928318613457189049159200833175","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95196756237527666678816095708949563380","series":"1.2.826.0.1.3680043.8.498.10006337896290853723104114832962427832","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95262231945171099723559613554350591139","series":"1.2.826.0.1.3680043.8.498.50126205431307217773521333466389012023","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95324318295506605557933083155792587297","series":"1.2.826.0.1.3680043.8.498.24834951316493595040459969220994527902","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95376417067109020835342111882595126109","series":"1.2.826.0.1.3680043.8.498.22325127075892451138893648571740664791","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95464807346156547985763374990661923815","series":"1.2.826.0.1.3680043.8.498.38311613831150420386807431983600101529","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95500234888498554736637315199668317948","series":"1.2.826.0.1.3680043.8.498.12269639604915811234152583167593070613","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95560861705581165570145459410262387310","series":"1.2.826.0.1.3680043.8.498.22146157013055620905154976481243909660","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95586477246200285088746818821808017494","series":"1.2.826.0.1.3680043.8.498.12400856454728899729788270854036261680","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95621693348567815920675804584086926261","series":"1.2.826.0.1.3680043.8.498.61492794300967409722045431958135491590","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95694598580744163428934772566469311180","series":"1.2.826.0.1.3680043.8.498.37920723957797560041831500800415142828","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95720534093121754146413060811354489247","series":"1.2.826.0.1.3680043.8.498.68071046982081347701568637623094202780","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95728724960383464194568391302477328450","series":"1.2.826.0.1.3680043.8.498.12875235954099742080676128469286875183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95850844945129153291759207212815927078","series":"1.2.826.0.1.3680043.8.498.74421461777237059152437511690162119699","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95882731821729917461275694847968503951","series":"1.2.826.0.1.3680043.8.498.11214466458205596033640753377618619741","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95984406324600336713471618620285270466","series":"1.2.826.0.1.3680043.8.498.62567007493538614107781700466460031956","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96094001324999448553107805455529187871","series":"1.2.826.0.1.3680043.8.498.11043096711298654959032092227130000412","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96146854841734718524530740965296119916","series":"1.2.826.0.1.3680043.8.498.10873452264345909510827388652681331335","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96158169859491339496094295508194547199","series":"1.2.826.0.1.3680043.8.498.99765592856949801390793066633934100749","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96316229168827165128604998083731481211","series":"1.2.826.0.1.3680043.8.498.13366602631140263411823818970429606240","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96425049974772570897047688166336162674","series":"1.2.826.0.1.3680043.8.498.42494337424243151736819852958626134171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96510621665552630060821422211995212854","series":"1.2.826.0.1.3680043.8.498.25022898020882325653827363895915968284","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96561978771090020509310535265236331946","series":"1.2.826.0.1.3680043.8.498.98729912204648617433792469292593860506","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96587029379314618196381307882235433083","series":"1.2.826.0.1.3680043.8.498.71367192344046999038992878799456726412","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96621557839848412555023112236627663339","series":"1.2.826.0.1.3680043.8.498.10369739610683148108649475039653276324","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96818533353374094273022322108861564926","series":"1.2.826.0.1.3680043.8.498.90820010486838582886157645134117510379","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96943946564115188893954628924857155670","series":"1.2.826.0.1.3680043.8.498.53253022316300505115497190376436607061","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96947237995559790704304892745226362883","series":"1.2.826.0.1.3680043.8.498.10618489032192665817238230460224415232","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97003157807630958179276831877591411623","series":"1.2.826.0.1.3680043.8.498.11584511005857254882257309630898733616","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97042446513400958172996637321308032556","series":"1.2.826.0.1.3680043.8.498.36288401335593644898563054343333091999","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97087958215554026148585108910609248523","series":"1.2.826.0.1.3680043.8.498.16788010914842515729536979717185551987","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97156518925820521351080132062400795929","series":"1.2.826.0.1.3680043.8.498.67936293451691155406207527870697210566","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97248397528214121944398119979516268902","series":"1.2.826.0.1.3680043.8.498.11567152704466060822829948290981112538","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97259977321685658337087695233264517545","series":"1.2.826.0.1.3680043.8.498.95205313233076265380474381227985166465","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97274847570547462056218828853253311223","series":"1.2.826.0.1.3680043.8.498.64048340611924967907545823087028891197","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97348112008590316355392924312704081684","series":"1.2.826.0.1.3680043.8.498.12886698767826739293263527282880624445","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97464332941382646555656038793589898504","series":"1.2.826.0.1.3680043.8.498.62992662355600540252061771531683730823","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97488501634729541923597957717971376373","series":"1.2.826.0.1.3680043.8.498.12765928118329448134601845051031663298","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97500632594529766269369977508352975364","series":"1.2.826.0.1.3680043.8.498.13115249201544999166524459450098729467","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97564564380731258764291146761742841527","series":"1.2.826.0.1.3680043.8.498.13708465990840674813301893276991793419","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97600121574703193898619052529830275028","series":"1.2.826.0.1.3680043.8.498.38570888350478346700798869655183235809","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97694479664965569386338738579573728752","series":"1.2.826.0.1.3680043.8.498.11474053144150918921635177414632028372","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97705920250069816684861966612302390614","series":"1.2.826.0.1.3680043.8.498.88505980881210103366418806867355845861","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97759980486481979519068526243106370997","series":"1.2.826.0.1.3680043.8.498.10028090115069501801858560715672119975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97849030682292314880463675622456900933","series":"1.2.826.0.1.3680043.8.498.11254064448148853027672991420341256773","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97944261713902372188894456162512065499","series":"1.2.826.0.1.3680043.8.498.30838528101219329662934041519223113284","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98027050915885651900992366068887550187","series":"1.2.826.0.1.3680043.8.498.12064688553051448846779540919458156891","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98039072306618106705944452906395982381","series":"1.2.826.0.1.3680043.8.498.48433716860160536152933522698415135238","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98081486779628580946589119341754353354","series":"1.2.826.0.1.3680043.8.498.35240239757380081791604122065164270058","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98112373403331748422888362310572528002","series":"1.2.826.0.1.3680043.8.498.10794571810050001903876671613186227371","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98151095279541278201351784048265686573","series":"1.2.826.0.1.3680043.8.498.89799449968576275180355642894876118266","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98239758242192048511912905088391835938","series":"1.2.826.0.1.3680043.8.498.84446047077305537781230639303530514072","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98333658788330318335997032513277632526","series":"1.2.826.0.1.3680043.8.498.12148909510424887091786540394576803462","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98351891935507349137982199658103205287","series":"1.2.826.0.1.3680043.8.498.28144086315074911201148392526060519436","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98454206323401382140655387224146512225","series":"1.2.826.0.1.3680043.8.498.12866270522776771758731155523880965627","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98581537941208552065033680853629122764","series":"1.2.826.0.1.3680043.8.498.21185789106721388821265406707666884010","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98752901295449528283322564256444025625","series":"1.2.826.0.1.3680043.8.498.11190767291837145026606136783099634136","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98795677271204502833293873138219040186","series":"1.2.826.0.1.3680043.8.498.12922961107811897628935721830856124651","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98871650330162988912501872031191530068","series":"1.2.826.0.1.3680043.8.498.50740569070424657609474058350873974359","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98890851347757949328159691828768030366","series":"1.2.826.0.1.3680043.8.498.43730653114557935285560193112707390418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98958651320983196982606333548522846934","series":"1.2.826.0.1.3680043.8.498.23701403134050340053520057612299581333","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99088247184917460283293617180083591028","series":"1.2.826.0.1.3680043.8.498.99296278159743017154669111174057043954","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99155175553749543768821558247962844052","series":"1.2.826.0.1.3680043.8.498.62651775271966871226031493606762959826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99243902497374852166139435719549686908","series":"1.2.826.0.1.3680043.8.498.10278741895034003872308468788970235037","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99309981642816644502709148844743995902","series":"1.2.826.0.1.3680043.8.498.11840699125649499571625716420840407986","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99414163787650487720542840540379270468","series":"1.2.826.0.1.3680043.8.498.11279388378589115855914406391620036839","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99444110025206148999881196086066868280","series":"1.2.826.0.1.3680043.8.498.18467353455515669631004185587298838199","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99454657594216441037514296544803013407","series":"1.2.826.0.1.3680043.8.498.69381700453492471702839986638835382447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99526998617163591447503530185291775961","series":"1.2.826.0.1.3680043.8.498.11794103524878338458957836242404585577","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99578894341571394220300473118855007095","series":"1.2.826.0.1.3680043.8.498.37923702957884362423359242882474114475","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99708702196778120583644711299950439036","series":"1.2.826.0.1.3680043.8.498.12060332525024522944496759188091091266","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99726731706411379395165830480890405115","series":"1.2.826.0.1.3680043.8.498.65590457703911397416957759032098537669","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99813269006669714758887909212386431934","series":"1.2.826.0.1.3680043.8.498.13323672942748002012829825211849675113","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99864345386635409953374818045984558970","series":"1.2.826.0.1.3680043.8.498.81043194074304253961751040429357535585","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99880021280666757817587834481314366555","series":"1.2.826.0.1.3680043.8.498.64324647689002382904014367598143788918","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99939100657291954890286900729983287973","series":"1.2.826.0.1.3680043.8.498.75652438013374920566145809256507662433","plane":"Coronal","fluid":1,"slot":"COR_FS"}]')
print('series 320 rgb, UID figes depuis crop130_rgb25d :', len(jobs))
assert len(jobs) == 1778
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray) -> np.ndarray:
    sl = cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w) volume ORIGINAL trie

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    # memes 24 centres que crop130 (linspace dans la bande),
    # mais voisins ±1 dans le volume original, pas dans les 24
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    out = np.empty((N_SLICES, 3, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    last = n - 1
    n_clamp = 0
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        if km == k or kp == k:
            n_clamp += 1
        out[i, 0] = _to_u8(vol[km])
        out[i, 1] = _to_u8(vol[k])
        out[i, 2] = _to_u8(vol[kp])

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
        "n_clamp": int(n_clamp),
        "rgb": "dicom_neighbors",
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_rgb25d_320_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / f"manifest_rgb25d_320_part{SHARD_ID}.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "shape": [N_SLICES, 3, IMG_SIZE, IMG_SIZE],
    "rgb": "dicom_neighbors",
    "note": (
        "1 serie fluid-prioritaire par plan. Crop 130 mm (PixelSpacing) sinon "
        "centre 70 %. Bande 20-80 %, linspace 24 centres. Pour chaque centre k "
        "dans le volume ORIGINAL : R=k-1, G=k, B=k+1 (1 coupe DICOM d'ecart, "
        "clamp aux bords). Resize 320, uint8. PAS les voisins des 24 linspace."
    ),
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
            "n_clamp": None, "rgb": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        assert vol.shape == (N_SLICES, 3, IMG_SIZE, IMG_SIZE), vol.shape
        assert vol.dtype == np.uint8, vol.dtype
        rg = np.abs(vol[:, 0].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        bg = np.abs(vol[:, 2].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        print(
            name, vol.shape, vol.dtype,
            "min", int(vol.min()), "max", int(vol.max()),
            "mean", float(vol.mean()),
            "|R-G|", round(float(rg), 2), "|B-G|", round(float(bg), 2),
        )
print("attendu : shape (24, 3, 320, 320), |R-G| et |B-G| > 0 (voisins DICOM reels)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
